In [1]:
import pyvisa, iv_automation, lf6_automation, spectral_experiments
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import scipy as scipy # type: ignore
import pandas as pd
import winsound
rm = pyvisa.ResourceManager()
print(rm.list_resources())
plt.rcParams['figure.raise_window'] = False
import os
import time
# --- 1. Setup & Unique Filename ---
def unique_filename(path):
    if not os.path.exists(path): return path
    directory, filename = os.path.split(path)
    name, ext = os.path.splitext(filename)
    n = 1
    while True:
        new_path = os.path.join(directory, f"{name}_{n}{ext}")
        if not os.path.exists(new_path): return new_path
        n += 1

ModuleNotFoundError: No module named 'spectral_experiments'

In [2]:
POWER_EFF = 0.44  # power efficiency factor

In [3]:
# Save data to user folder
DEV_NAME = 'YZD300_202604_PL'
spectral_experiments.USER_FOLDER = rf"D:\instrument_control_v3_1\{DEV_NAME}\initial data"


In [4]:
# change address
keithley1 = iv_automation.KeithControl('GPIB0::02::INSTR', 'GPIB26', 'Vtg', rm)
keithley1.set_volt_step(curr_compliance=1E-8, delay=0.1, volt_compliance=20)
# change address
keithley2 = iv_automation.KeithControl('GPIB0::03::INSTR', 'GPIB23', 'Vbg', rm)
keithley2.set_volt_step(curr_compliance=1E-8, delay=0.1, volt_compliance=20)
instrument_list = [keithley1, keithley2]
iv = iv_automation.IVSetup(instrument_list)

#x_goto(self, x_name, target, delta, delay)
iv.x_goto('Vtg', 0, 0.2, 0.1)
iv.x_goto('Vbg', 0, 0.2, 0.1)
iv.report_status()


KEITHLEY INSTRUMENTS INC.,MODEL 2400,4502039,D02 Jan 20 2021 10:18:49/B01  /W/N
:SOUR:FUNC VOLT
:SENS:FUNC 'CURR'
:SENS:CURR:PROT 1.00e-06
SOUR:DEL 0.100
:SENS:FUNC:CONC ON
:FORM:ELEM VOLT ,CURR
:SOUR:VOLT:MODE FIXED
:SOUR:VOLT:RANG 20
TRIG:COUN 1
:OUTP ON
:SOUR:FUNC VOLT
:SENS:FUNC 'CURR'
:SENS:CURR:PROT 1.00e-08
SOUR:DEL 0.100
:SENS:FUNC:CONC ON
:FORM:ELEM VOLT ,CURR
:SOUR:VOLT:MODE FIXED
:SOUR:VOLT:RANG 20
TRIG:COUN 1
:OUTP ON
KEITHLEY INSTRUMENTS INC.,MODEL 2400,1045870,C32   Oct  4 2010 14:20:11/A02  /K/H
:SOUR:FUNC VOLT
:SENS:FUNC 'CURR'
:SENS:CURR:PROT 1.00e-06
SOUR:DEL 0.100
:SENS:FUNC:CONC ON
:FORM:ELEM VOLT ,CURR
:SOUR:VOLT:MODE FIXED
:SOUR:VOLT:RANG 20
TRIG:COUN 1
:OUTP ON
:SOUR:FUNC VOLT
:SENS:FUNC 'CURR'
:SENS:CURR:PROT 1.00e-08
SOUR:DEL 0.100
:SENS:FUNC:CONC ON
:FORM:ELEM VOLT ,CURR
:SOUR:VOLT:MODE FIXED
:SOUR:VOLT:RANG 20
TRIG:COUN 1
:OUTP ON
x_channel Vtg: value: 0.0
x_channel Vbg: value: 0.0
y_channel measured_Vtg: value: 0.0
y_channel Vtg_leakage: value: -2.119122e-11

In [129]:
lf6 = lf6_automation.LF6Setup()
lf6.print_saved_experiments()

My Saved Experiments:
	2100
	attodry1000 new
	attodry1000
	cxt
	EMCCD
	Experiment1
	Experiment2
	Experiment3
	Modulation
	PL_Lei
	ref
	test


In [5]:
#load powermeter_shuai
import pyvisa
import time

rm = pyvisa.ResourceManager()
inst = rm.open_resource("USB0::0x1313::0x8078::P0011342::INSTR")

print(inst.query("*IDN?"))

inst.write("SENS:CORR:WAV 660")
inst.write("SENS:POW:RANG:AUTO ON")

time.sleep(0.5)
power = inst.query("MEAS:POW?")
print("Power =", power)

Thorlabs,PM100D,P0011342,2.4.0

Power = 2.79269756E-07



In [6]:
power = float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF
print(f"{power:.3f} uW")

0.118 uW


In [8]:
#load powermeter
from datetime import datetime
from ctypes import cdll,c_long, c_ulong, c_uint32,byref,create_string_buffer,c_bool,c_char_p,c_int,c_int16,c_double, sizeof, c_voidp
from TLPMX import TLPMX
import time

from TLPMX import TLPM_DEFAULT_CHANNEL

# Find connected power meter devices.
tlPM = TLPMX()
deviceCount = c_uint32()
tlPM.findRsrc(byref(deviceCount))

print("Number of found devices: " + str(deviceCount.value))
print("")

resourceName = create_string_buffer(1024)

for i in range(0, deviceCount.value):
    tlPM.getRsrcName(c_int(i), resourceName)
    print("Resource name of device", i, ":", c_char_p(resourceName.raw).value)
print("")
tlPM.close()

# Connect to last device.
tlPM = TLPMX()
tlPM.open(resourceName, c_bool(True), c_bool(True))

message = create_string_buffer(1024)
tlPM.getCalibrationMsg(message,TLPM_DEFAULT_CHANNEL)
print("Connected to device", i)
print("Last calibration date: ",c_char_p(message.raw).value)
print("")

time.sleep(2)
wavelength = c_double(730)
tlPM.setWavelength(wavelength,TLPM_DEFAULT_CHANNEL)
# Enable auto-range mode.
# 0 -> auto-range disabled
# 1 -> auto-range enabled
tlPM.setPowerAutoRange(c_int16(1),TLPM_DEFAULT_CHANNEL)

# Set power unit to Watt.
# 0 -> Watt
# 1 -> dBm
tlPM.setPowerUnit(c_int16(0),TLPM_DEFAULT_CHANNEL)


Number of found devices: 1

Resource name of device 0 : b'USB0::0x1313::0x8078::::INSTR'



NameError: b'Insufficient location information or the device or resource is not present in the system.'

In [ ]:
# Take power measurement
tlPM.setWavelength(c_double(660),TLPM_DEFAULT_CHANNEL)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = meas_power.value*1e6
print(power)


NameError: name 'tlPM' is not defined

In [7]:
##load linear stage
import pylablib as pll
from pylablib.devices import Thorlabs


c:\Users\commo\AppData\Local\Programs\Python\Python313\Lib\site-packages\pylablib\core\utils\module.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [8]:
def move_stage_checked(stage, stage_pos, max_retries=3, tolerance=1.0,
                       settle_s=1.0, retry_wait_s=0.5, round_to=1, verbose=True) -> bool:
    stage_pos = round(stage_pos, round_to) if round_to is not None else stage_pos
    for attempt in range(max_retries):
        try:
            if verbose and attempt > 0:
                print(f"  -> Retry move to {stage_pos} (Attempt {attempt + 1})...")
            stage.move_to(stage_pos)
            time.sleep(settle_s)
            current_pos = stage.get_position()
            if abs(current_pos - stage_pos) <= tolerance:
                return True
            if verbose:
                diff = current_pos - stage_pos
                print(f"  -> Position Mismatch! Target: {stage_pos}, Actual: {current_pos:.2f} (Diff: {diff:.2f})")
        except Exception as e:
            if verbose:
                print(f"  -> Error on move attempt {attempt + 1}: {e}")
            time.sleep(retry_wait_s)
    return False

In [9]:
stage = Thorlabs.ElliptecMotor("COM6")

In [335]:
probe_stage = Thorlabs.ElliptecMotor("COM8")

In [68]:
stage.close()

In [ ]:
probe_stage.close()

In [10]:
move_stage_checked(stage, 1400)

True

In [11]:
import winsound
def beep():
    duration = 1000  # milliseconds
    freq = 440  # Hz
    winsound.Beep(freq, duration)

In [ ]:
# 500 probe off 2000 probe on
probe_stage.move_to(1500)
time.sleep(2)
wavelength = c_double(730)
tlPM.setWavelength(wavelength,TLPM_DEFAULT_CHANNEL)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = meas_power.value*1e6*POWER_EFF
print(power)

In [16]:
# 500 probe off 2000 probe on
# probe_stage.move_to(0)
time.sleep(2)
wavelength = c_double(730)
tlPM.setWavelength(wavelength,TLPM_DEFAULT_CHANNEL)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = meas_power.value*1e6
print(power)

316.853402


In [12]:
stage.move_to(3300)
power = float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF
print(f"{power:.3f} uW")

69.886 uW


In [14]:
from typing import Any, Literal, Optional

def spectra_to_matrix(
    spectra: Any,
    H: int = 256,
    W: int = 1024,
    layout: Literal["row-major", "column-major"] = "row-major",
    dtype: Optional[np.dtype] = None,
) -> np.ndarray:
    """
    Convert a flat lightfield spectra (length H*W) to a 2D matrix (H, W).
    - Accepts NumPy arrays, Python lists, or bytes-like buffers (then pass dtype).
    - layout="row-major": left-to-right, top-to-bottom (most devices).
      Use layout="column-major" if the stream is column-wise (transposed).
    """
    expected = H * W

    # Normalize to 1-D NumPy array without copying when possible
    if isinstance(spectra, (bytes, bytearray, memoryview)):
        if dtype is None:
            raise ValueError("When spectra is bytes-like, pass dtype (e.g., np.uint16).")
        a = np.frombuffer(spectra, dtype=dtype)
    else:
        a = np.asarray(spectra, dtype=dtype if dtype is not None else None)

    a = a.ravel()  # 1-D view if possible
    if a.size != expected:
        raise ValueError(f"Got {a.size} values; expected {expected} ({H}×{W}).")

    M = a.reshape(H, W)  # O(1) view if contiguous
    if layout == "column-major":
        M = M.T
    return M

## Andor camera

In [13]:
import pyvisa, iv_automation, lf6_automation, spectral_experiments
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import scipy as scipy # type: ignore
import pandas as pd
import winsound
rm = pyvisa.ResourceManager()
print(rm.list_resources())
# plt.rcParams['figure.raise_window'] = False
import os

('USB0::0x1313::0x8078::P0011342::INSTR', 'ASRL1::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL6::INSTR', 'ASRL7::INSTR', 'ASRL8::INSTR', 'ASRL9::INSTR', 'GPIB0::2::INSTR', 'GPIB0::3::INSTR')


#### 

In [14]:
### control andor camera 
# Si:
# InGaAs
import pylablib as pll
pll.par["devices/dlls/andor_sdk2"] = r"D:\Insturment control v3\andor dll"
pll.par["devices/dlls/andor_shamrock"] = r"D:\Insturment control v3\andor dll\shamrock dll"
from pylablib.devices import Andor
import ctypes
ctypes.WinDLL(r"D:\Insturment control v3\andor dll\shamrock dll\ShamrockCIF.dll")
# cam = Andor.AndorSDK2Camera()

<WinDLL 'D:\Insturment control v3\andor dll\shamrock dll\ShamrockCIF.dll', handle 7ffad3e30000 at 0x1929c709370>

In [15]:
n = Andor.get_cameras_number_SDK2()
print("SDK2 cameras:", n)

for idx in range(n):
    cam = Andor.AndorSDK2Camera(idx=idx)
    try:
        dev = cam.get_device_info()          # (controller_model, head_model, serial_number)
        det = cam.get_detector_size()        # (width, height)
        pix = cam.get_pixel_size()           # pixel size in meters
        mode = cam.get_read_mode()
        roi  = cam.get_roi()
        print(f"\nidx={idx}")
        print("  device_info:", dev)
        print("  detector_size:", det)
        print("  pixel_size (m):", pix)
        print("  read_mode:", mode)
        print("  roi:", roi)
    finally:
        cam.close()

SDK2 cameras: 1

idx=0
  device_info: TDeviceInfo(controller_model='USB', head_model='DU490_17', serial_number=17029)
  detector_size: (512, 1)
  pixel_size (m): (2.4999999999999998e-05, 0.0005)
  read_mode: image
  roi: (0, 512, 0, 1, 1, 1)


In [16]:
cam_ingaas = Andor.AndorSDK2Camera(idx=0)

In [17]:
# Set cool down temp
cam_ingaas.set_temperature(-75)
cam_ingaas.set_fan_mode('full')
cam_ingaas.setup_shutter('auto')

('auto', 0, 0, 0)

In [18]:
print("Shamrock list:", Andor.list_shamrock_spectrographs())  # should show serial(s) :contentReference[oaicite:8]{index=8}

Shamrock list: ['SR-2219', '€ùÛa', '€ùÛa', '€ùÛa', '€ùÛa', '€ùÛa', '€ùÛa']


In [19]:

spec = Andor.ShamrockSpectrograph(idx=0)


In [20]:
spec.set_slit_width("input_side", 1000e-6)  # 50 µm :contentReference[oaicite:12]{index=12}


0.001

In [21]:
spec.get_grating_info(1)

TGratingInfo(lines=150.0, blaze_wavelength='1250', home=4018, offset=340)

In [22]:
spec.set_grating(1)                 # 1-based grating index :contentReference[oaicite:10]{index=10}


1

In [23]:
spec.set_wavelength(1210e-9)         # meters (900 nm) :contentReference[oaicite:11]{index=11}


1.21e-06

In [24]:

# 4) Control spectrograph
# spec.set_shutter("opened")          # open/closed :contentReference[oaicite:13]{index=13}

# 5) Get wavelength axis for your camera pixels
spec.setup_pixels_from_camera(cam_ingaas)  # copies pixel size & count from camera :contentReference[oaicite:14]{index=14}
wl_m = spec.get_calibration()       # wavelength per pixel (meters) :contentReference[oaicite:15]{index=15}
wl_nm = wl_m * 1e9

In [28]:
cam_ingaas.close()

In [25]:
cam_ingaas.set_cooler(on=True)

True

In [27]:
cam_ingaas.get_temperature()

-53.2869987487793

In [28]:
def get_one_spectra(
    cam,
    exposure_time=1,
    num_acc=1,
    plot=False,
    fig=None,
    ax=None,
    line=None,
    *,
    spec=None,
    include_wl=True,
    timeout=10.0,
    timeout_margin=2.0,
    discard_first=False,
    x_axis=None,
    x_unit="nm",
    invert_wl_axis=False,   # <<< NEW: reverse wavelength axis only
):
    # cleanup
    for fn in ("stop_acquisition", "clear_acquisition"):
        if hasattr(cam, fn):
            try: getattr(cam, fn)()
            except Exception: pass

    cam.set_exposure(exposure_time)

    frame_timeout = max(float(timeout), float(exposure_time) + float(timeout_margin))

    nframes = int(num_acc) + (1 if discard_first else 0)
    if nframes <= 0:
        raise ValueError("num_acc must be >= 1")

    if nframes > 1 and hasattr(cam, "grab"):
        frames = cam.grab(nframes=nframes, frame_timeout=frame_timeout)
    elif hasattr(cam, "snap"):
        frames = [cam.snap(timeout=frame_timeout) for _ in range(nframes)]
    else:
        raise RuntimeError("Camera object doesn't expose grab() or snap().")

    if isinstance(frames, (list, tuple)):
        arr = np.stack([np.asarray(f).squeeze() for f in frames], axis=0)
    else:
        arr = np.asarray(frames)
        if arr.ndim == 1:
            arr = arr[None, :]

    arr = arr.astype(np.float32, copy=False)
    if discard_first and arr.shape[0] > 1:
        arr = arr[1:]

    y = arr.mean(axis=0)

    # build x
    if x_axis is not None:
        x = np.asarray(x_axis)
        x_label = "Wavelength (nm)"
    elif include_wl and (spec is not None):
        spec.setup_pixels_from_camera(cam)
        wl_m = np.asarray(spec.get_calibration())
        if x_unit.lower() == "nm":
            x = wl_m * 1e9
            x_label = "Wavelength (nm)"
        else:
            x = wl_m
            x_label = "Wavelength (m)"
    else:
        x = np.arange(y.size)
        x_label = "Pixel"

    # >>> your requested behavior: wl_nm low->high, but spectrum should be high->low
    # so reverse x only
    if invert_wl_axis:
        x = x[::-1]

    # plot
    if plot:
        if fig is None or ax is None or line is None:
            plt.ion()
            fig, ax = plt.subplots()
            (line,) = ax.plot(x, y)
            ax.set_title(f"exp = {exposure_time}")
            ax.set_xlabel(x_label)
            ax.set_ylabel("Intensity")
            ax.grid(True)
            plt.show()
        else:
            line.set_data(x, y)   # IMPORTANT: update both x and y
            ax.set_xlabel(x_label)
            ax.relim()
            ax.autoscale_view()
            fig.canvas.draw()
            fig.canvas.flush_events()

    # cleanup
    for fn in ("stop_acquisition", "clear_acquisition"):
        if hasattr(cam, fn):
            try: getattr(cam, fn)()
            except Exception: pass

    return x, y, fig, ax, line

In [29]:
def live_spectra_plot(
    cam,
    *,
    spec=None,
    exposure_time=1.0,
    refresh_period=0.05,
    timeout=5.0,
    timeout_margin=2.0,
    discard_first=False,
    x_axis=None,
    use_wavelength=True,
    x_unit="nm",
    force_recalc_wl=False,
    autoscale_y=True,
    y_margin=0.05,
    invert_wl_axis=True,
):
    import numpy as np
    import matplotlib.pyplot as plt
    import time

    def _try_call(obj, name):
        if obj is not None and hasattr(obj, name):
            try:
                getattr(obj, name)()
            except Exception:
                pass

    _try_call(cam, "stop_acquisition")
    _try_call(cam, "clear_acquisition")

    cam.set_exposure(exposure_time)
    frame_timeout = max(float(timeout), float(exposure_time) + float(timeout_margin))

    wl_cache = {"key": None, "x": None, "label": None}

    def _spec_key():
        if spec is None:
            return None
        key_parts = []
        for fn in ("get_wavelength", "get_wavelength_nm"):
            if hasattr(spec, fn):
                try:
                    key_parts.append((fn, float(getattr(spec, fn)())))
                except Exception:
                    pass
                break
        if hasattr(spec, "get_grating"):
            try:
                key_parts.append(("grating", int(spec.get_grating())))
            except Exception:
                pass
        for fn in ("get_binning", "get_image_parameters", "get_detector_size"):
            if hasattr(cam, fn):
                try:
                    key_parts.append((fn, str(getattr(cam, fn)())))
                except Exception:
                    pass
        return tuple(key_parts) if key_parts else ("spec_present",)

    def _make_x(n):
        if x_axis is not None:
            xa = np.asarray(x_axis)
            return xa, ("Wavelength (nm)" if xa.size == n else "X"), True

        if use_wavelength and (spec is not None):
            key = _spec_key()
            if force_recalc_wl or wl_cache["key"] != key or wl_cache["x"] is None or wl_cache["x"].size != n:
                spec.setup_pixels_from_camera(cam)
                wl_m = np.asarray(spec.get_calibration())
                if x_unit.lower() == "nm":
                    wl = wl_m * 1e9
                    lab = "Wavelength (nm)"
                else:
                    wl = wl_m
                    lab = "Wavelength (m)"
                wl_cache.update({"key": key, "x": wl, "label": lab})
            return np.asarray(wl_cache["x"]), wl_cache["label"], True

        return np.arange(n), "Pixel", False

    plt.ion()
    fig, ax = plt.subplots()
    line = None
    frame_i = 0
    last_draw = 0.0

    has_sequence = hasattr(cam, "setup_acquisition") and hasattr(cam, "start_acquisition")
    has_read_newest = hasattr(cam, "read_newest_image")
    has_wait = hasattr(cam, "wait_for_frame")
    use_backend_A = has_sequence and has_read_newest and has_wait

    print("use_backend_A =", use_backend_A)
    print("frame_timeout =", frame_timeout)

    if use_backend_A:
        cam.setup_acquisition(mode="sequence", nframes=10**9)
        cam.start_acquisition()

        if discard_first:
            cam.wait_for_frame(since="now", nframes=1, timeout=frame_timeout)
            _ = cam.read_newest_image()

        print("[live_spectra_plot] Backend A: sequence mode + read_newest_image")

        try:
            while True:
                print("waiting frame...")
                cam.wait_for_frame(since="now", nframes=1, timeout=frame_timeout)
                print("frame arrived")

                frame = cam.read_newest_image()
                if frame is None:
                    print("frame is None")
                    continue

                spec_y = np.asarray(frame).squeeze().astype(np.float32, copy=False)
                print("frame shape:", spec_y.shape)

                x, xlab, is_wl = _make_x(spec_y.size)
                if invert_wl_axis and is_wl:
                    x = x[::-1]

                if line is None:
                    (line,) = ax.plot(x, spec_y)
                    ax.set_xlabel(xlab)
                    ax.set_ylabel("Intensity")
                    ax.grid(True)
                else:
                    line.set_data(x, spec_y)
                    ax.set_xlabel(xlab)

                frame_i += 1

                ax.relim()
                ax.autoscale_view(scalex=True, scaley=False)

                if autoscale_y:
                    y0 = float(np.nanmin(spec_y))
                    y1 = float(np.nanmax(spec_y))
                    if np.isfinite(y0) and np.isfinite(y1) and y1 > y0:
                        pad = (y1 - y0) * y_margin
                        ax.set_ylim(y0 - pad, y1 + pad)

                now = time.time()
                if now - last_draw >= refresh_period:
                    ax.set_title(f"exp={exposure_time}  frame={frame_i}")
                    fig.canvas.draw()
                    fig.canvas.flush_events()
                    last_draw = now

        except KeyboardInterrupt:
            print("\n[live_spectra_plot] Stopped by user (Ctrl+C).")
        except Exception as e:
            print(f"\n[live_spectra_plot] ERROR: {type(e).__name__}: {e}")
            raise
        finally:
            _try_call(cam, "stop_acquisition")
            _try_call(cam, "clear_acquisition")

        return fig, ax, line

    if not hasattr(cam, "snap"):
        raise RuntimeError("Camera does not support Backend A, and snap() is also unavailable.")

    print("[live_spectra_plot] Backend B: repeated snap()")

    if discard_first:
        _ = cam.snap(timeout=frame_timeout)

    try:
        while True:
            frame = cam.snap(timeout=frame_timeout)
            spec_y = np.asarray(frame).squeeze().astype(np.float32, copy=False)

            x, xlab, is_wl = _make_x(spec_y.size)
            if invert_wl_axis and is_wl:
                x = x[::-1]

            if line is None:
                (line,) = ax.plot(x, spec_y)
                ax.set_xlabel(xlab)
                ax.set_ylabel("Intensity")
                ax.grid(True)
            else:
                line.set_data(x, spec_y)
                ax.set_xlabel(xlab)

            frame_i += 1
            ax.relim()
            ax.autoscale_view(scalex=True, scaley=False)

            if autoscale_y:
                y0 = float(np.nanmin(spec_y))
                y1 = float(np.nanmax(spec_y))
                if np.isfinite(y0) and np.isfinite(y1) and y1 > y0:
                    pad = (y1 - y0) * y_margin
                    ax.set_ylim(y0 - pad, y1 + pad)

            now = time.time()
            if now - last_draw >= refresh_period:
                ax.set_title(f"exp={exposure_time}  frame={frame_i}")
                fig.canvas.draw()
                fig.canvas.flush_events()
                last_draw = now

    except KeyboardInterrupt:
        print("\n[live_spectra_plot] Stopped by user (Ctrl+C).")
    except Exception as e:
        print(f"\n[live_spectra_plot] ERROR: {type(e).__name__}: {e}")
        raise
    finally:
        _try_call(cam, "stop_acquisition")
        _try_call(cam, "clear_acquisition")

    return fig, ax, line

In [30]:
def Andor_dual_gate_spectra_sweep(
    sample_name, exp_name,
    cam,
    iv,
    vbg_start, vbg_stop, vtg_start, vtg_stop,
    frames, repeat,
    centerwav,                 # nm
    exposure_time,
    *,
    Devname="Dev1",            # <- 新增：文件夹名
    save_root=".",             # <- 新增：保存根目录（默认当前目录）
    spec=None,
    plot=False,
    num_acc=1,
    timeout=10.0,
    discard_first=False,
):
    """
    Save CSV into:  {save_root}/{Devname}/  folder
    """
    # ---- build save path ----
    save_dir = os.path.join(save_root, str(Devname), "initial data")
    os.makedirs(save_dir, exist_ok=True)

    file_path = os.path.join(save_dir, f"{sample_name}~{exp_name}.csv")
    file_name = unique_filename(file_path)

    vbgs = np.linspace(vbg_start, vbg_stop, frames)
    vtgs = np.linspace(vtg_start, vtg_stop, frames)

    fig = ax = line = None

    # ---- wavelength axis from spectrograph ----
    wl_nm = None
    if spec is not None:
        spec.set_wavelength(centerwav * 1e-9)
        spec.setup_pixels_from_camera(cam)
        wl_nm = np.asarray(spec.get_calibration()) * 1e9

    if wl_nm is None:
        n_pix_guess = 512
        wl_nm = np.arange(n_pix_guess, dtype=float)
    wl_nm = wl_nm[::-1]
    # ---- write header once ----
    with open(file_name, "w") as f:
        cols = np.concatenate(
            (np.array(["Vbg", "Vtg", "Vtg+Vbg", "Vtg-Vbg"], dtype="U"), wl_nm.astype("U"))
        ).reshape(1, -1)
        np.savetxt(f, cols, fmt="%s", delimiter=",")

    for i in range(repeat):
        print(f"repeat {i+1}/{repeat}")
        with open(file_name, "a") as f:
            for vbg, vtg in zip(vbgs, vtgs):
                iv.x_goto("Vbg", float(vbg), 0.05, 0.1)
                iv.x_goto("Vtg", float(vtg), 0.05, 0.1)

                x, spectra, fig, ax, line = get_one_spectra(
                    cam,
                    exposure_time=exposure_time,
                    num_acc=num_acc,
                    plot=plot,
                    fig=fig, ax=ax, line=line,
                    spec=spec,
                    include_wl=False,
                    timeout=timeout,
                    discard_first=discard_first,
                )

                if spectra.size != wl_nm.size:
                    raise RuntimeError(
                        f"Spectra length changed ({spectra.size}) != wavelength axis length ({wl_nm.size}). "
                        "ROI/binning changed mid-run; start a new file."
                    )

                row = np.concatenate(
                    (np.array([vbg, vtg, vbg + vtg, vtg - vbg], dtype=np.float64), spectra)
                ).reshape(1, -1)
                np.savetxt(f, row, fmt="%.5e", delimiter=",")

        iv.x_goto("Vbg", 0, 0.1, 0.1)
        iv.x_goto("Vtg", 0, 0.1, 0.1)

    return file_name


In [31]:
def Andor_dual_gate_megasweep(
    sample_name,
    cam,
    iv,
    vbg_start, vbg_stop, vtg_start, vtg_stop,
    frames, repeat,
    centerwav,            # nm
    exposure_time,
    *,
    spec=None,            # Andor.ShamrockSpectrograph(idx=0)
    Devname="Dev1",
    save_root=".",
    plot=False,
    num_acc=1,
    timeout=10.0,
    timeout_margin=2.0,
    discard_first=False,
    invert_wl_axis=True,  # True => save header high->low (reverse wl only)
):
    # ---------------- save path ----------------
    save_dir = os.path.join(save_root, str(Devname), "initial data")
    os.makedirs(save_dir, exist_ok=True)

    file_path = os.path.join(save_dir, f"{sample_name}.csv")
    file_name = unique_filename(file_path)

    # ---------------- wavelength axis ----------------
    wl_nm = None
    if spec is not None:
        # set center wavelength (nm -> m)
        spec.set_wavelength(centerwav * 1e-9)
        spec.setup_pixels_from_camera(cam)
        wl_nm = np.asarray(spec.get_calibration()) * 1e9  # low->high

    if wl_nm is None:
        # fallback: pixel axis
        # (better than incorrect linear nm estimate)
        # will label as 0..N-1
        # NOTE: we will infer N from first spectrum later and rewrite header then if you want;
        # for now assume 512.
        wl_nm = np.arange(512, dtype=float)

    # your requirement: wl header should be high->low
    if invert_wl_axis:
        wl_save = wl_nm[::-1].copy()
    else:
        wl_save = wl_nm.copy()

    # ---------------- write header ONCE ----------------
    with open(file_name, "w") as f:
        header = np.concatenate(
            (
                np.array(["Vbg", "Vtg", "Vtg+Vbg", "Vtg-Vbg"], dtype="U"),
                wl_save.astype("U"),
            )
        ).reshape(1, -1)
        np.savetxt(f, header, fmt="%s", delimiter=",")

    # ---------------- sweep grids ----------------
    vbgs = np.linspace(vbg_start, vbg_stop, frames)
    vtgs_base = np.linspace(vtg_start, vtg_stop, frames)

    fig = ax = line = None

    # Optional: wavelength cache for get_one_spectra to avoid repeated spec.get_calibration
    wl_cache = {"key": None, "x": None, "label": None}

    for r in range(repeat):
        print(f"[Repeat {r+1}/{repeat}] Saving to: {file_name}")

        with open(file_name, "a") as f:
            for i, vbg in enumerate(vbgs):
                vtgs = vtgs_base[::-1] if (i % 2 == 1) else vtgs_base  # snake scan

                iv.x_goto("Vbg", float(vbg), 0.1, 0.1)

                for vtg in vtgs:
                    iv.x_goto("Vtg", float(vtg), 0.1, 0.1)
                    time.sleep(0.05)

                    # Acquire spectrum
                    x, spectrum, fig, ax, line = get_one_spectra(
                        cam,
                        exposure_time=exposure_time,
                        num_acc=num_acc,
                        plot=plot,
                        fig=fig, ax=ax, line=line,
                        spec=spec,
                        include_wl=False,          # we already computed wl for header
                        timeout=timeout,
                        timeout_margin=timeout_margin,
                        discard_first=discard_first,
                        wl_cache=wl_cache,         # safe reuse if you later turn include_wl True
                    )

                    # IMPORTANT: do NOT reverse spectrum
                    # We only reversed the wavelength header (wl_save), because your camera pixel order
                    # corresponds to high->low.

                    if spectrum.size != wl_save.size:
                        raise RuntimeError(
                            f"Spectra length ({spectrum.size}) != wavelength header length ({wl_save.size}). "
                            "ROI/binning changed or calibration length mismatch. Start a new file."
                        )

                    meta = np.array([vbg, vtg, vbg + vtg, vtg - vbg], dtype=np.float64)
                    row = np.concatenate((meta, spectrum)).reshape(1, -1)
                    np.savetxt(f, row, fmt="%.5e", delimiter=",")

        # Reset gates after each repeat
        iv.x_goto("Vbg", 0, 0.1, 0.1)
        iv.x_goto("Vtg", 0, 0.1, 0.1)

    return file_name

In [208]:
spec.set_wavelength(1210e-9)
spec_axis_nm, I, fig, ax, line = get_one_spectra(
    cam_ingaas,
    exposure_time=20,
    num_acc=1,
    plot=True,
    spec=spec,      
    include_wl=True,
    invert_wl_axis=True
)


In [90]:
### repeatly get spectra to find good signal
# spec.set_grating(1)
spec.set_wavelength(1200e-9)
%matplotlib qt
# 连续实时显示（自动用 wavelength 做 x 轴）
live_spectra_plot(cam_ingaas, spec=spec, exposure_time=1, timeout=10, timeout_margin=3,
                discard_first=True)


use_backend_A = True
frame_timeout = 10.0
[live_spectra_plot] Backend A: sequence mode + read_newest_image
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arrived
frame shape: (512,)
waiting frame...
frame arr

(<Figure size 640x480 with 1 Axes>,
 <Axes: title={'center': 'exp=1  frame=26'}, xlabel='Wavelength (nm)', ylabel='Intensity'>,
 <matplotlib.lines.Line2D at 0x192a8782210>)

In [37]:
probe_stage.move_to(2300)
power_coeff = 0.36
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = np.round(meas_power.value*1e6*power_coeff,2)
print(power)

NameError: name 'probe_stage' is not defined

In [207]:
iv.x_goto('Vtg', 0, 0.2, 0.05)
iv.x_goto('Vbg', 0, 0.2, 0.05)


variable: ['Vtg']
start: [14.]
end: [0]
steps: 71
variable: ['Vbg']
start: [14.]
end: [0]
steps: 71


In [205]:
# stage.move_to(0)
stage.move_to(1740)
POWER_EFF = 0.434
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 3)
print(f"{power:} uW")

# stage 3300 2990 2660 2270 1720 1310
# power 500  200  100   50   20   10

#with ND
# stage 2270 1720 1300 870 300  0
# power  5    2    1   0.5 0.2 0.1
# center

#  powers    = [500,  200,  100,   50,   20,   10,    1,   2,    5 ]      #uW
# pos_stages = [3300, 2995, 2665, 2280, 1740, 1340,   5,  395,  1890] 
# centerwavs = [1200, 1200, 1200, 1210, 1210, 1210, 1210, 1210, 1210]      # nm
# exp_times  = [5,     5,    10,   10,   20,   20,   60,   30,  30]      # s


19.945 uW


In [95]:
POWER_EFF = 0.434
stage.move_to(3300)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 3)
print(f"{power:} uW")
centerwav=1200
exp_time = 5
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}$~$p2$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-8, vbg_stop=15,
vtg_start=-8, vtg_stop=15,
frames=231,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

706.119 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vtg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vbg']
start: [-8.]
end: [-7.9]
steps: 2
variable: ['Vtg']
start: [-8.]
end: [-7.9]
steps: 2
variable: ['Vbg']
start: [-7.9]
end: [-7.8]
steps: 3
variable: ['Vtg']
start: [-7.9]
end: [-7.8]
steps: 3
variable: ['Vbg']
start: [-7.8]
end: [-7.7]
steps: 2
variable: ['Vtg']
start: [-7.8]
end: [-7.7]
steps: 2
variable: ['Vbg']
start: [-7.7]
end: [-7.6]
steps: 3
variable: ['Vtg']
start: [-7.7]
end: [-7.6]
steps: 3
variable: ['Vbg']
start: [-7.6]
end: [-7.5]
steps: 2
variable: ['Vtg']
start: [-7.6]
end: [-7.5]
steps: 2
variable: ['Vbg']
start: [-7.5]
end: [-7.4]
steps: 2
variable: ['Vtg']
start: [-7.5]
end: [-7.4]
steps: 2
variable: ['Vbg']
start: [-7.4]
end: [-7.3]
steps: 3
variable: ['Vtg']
start: [-7.4]
end: [-7.3]
steps: 3
variable: ['Vbg']
start: [-7.3]
end: [-7.2]
steps: 2
variable: ['Vtg']
start: [-7.3]
end: [-7.2]
steps: 2
variable: ['Vbg']
start: [-7

In [ ]:
#v1_10. power dependent 
POWER_EFF = 0.434 
num_acc = 1

# #powers    = [420,  350,   300,  200,  100,  50,  20,   10,    5,    2,    1]      #uW
# pos_stages = [3300, 3201, 3160, 3033, 2760, 2420, 1890, 1520, 1060, 560,  150] 
# centerwavs = [1200, 1200, 1200, 1200, 1200, 1210, 1210, 1220, 1220, 1230, 1230]      # nm
# exp_times  = [10,    10,   10,   15,   15,   15,   20,   20,   20,   45,   60]      # s

#powers    = [420,  200,  100,    2,    5,   20]      #uW
# pos_stages = [3300, 3033, 2760,  560, 1060, 1890] 
# centerwavs = [1200, 1200, 1200, 1230, 1230, 1220]      # nm
# exp_times  = [15,    20,    20,  60,   30,   30]      # s

# #powers    = [420,  200,  100,    1,    5,   20]      #uW
# pos_stages = [3300, 3033, 2760,  140, 1060, 1890] 
# centerwavs = [1200, 1200, 1200, 1210, 1210, 1210]      # nm for H
# exp_times  = [15,    20,    20,  60,   30,   30]      # s

# #powers    = [ 0.2,  0.5,   1]      #uW
# pos_stages = [ 390,  930, 1350] 
# centerwavs = [1210, 1210, 1210]      # nm for H with ND1
# exp_times  = [ 60,   60,   60]      # s

#powers    = [500,  200,  100,   50,   20,   10,    1,   2,    5 ]      #uW
pos_stages = [3300, 2995, 2665, 2280, 1740, 1340,   5,  395,  1890] 
centerwavs = [1200, 1200, 1200, 1210, 1210, 1210, 1210, 1210, 1210]      # nm for H 
exp_times  = [5,     5,    10,   10,   20,   20,   60,   30,  30]      # s

# #powers    = [ 1]      #uW
# pos_stages = [ 5] 
# centerwavs = [1210]      # nm for H 
# exp_times  = [60]      # s

for pos_stage, centerwav, exp_time in zip(pos_stages, centerwavs, exp_times):
    stage.move_to(pos_stage)
    time.sleep(1)
    stage.move_to(pos_stage)
    time.sleep(1)
    power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
    print(f"{power:} uW")
    # centerwav=1190
    # exp_time = 10
    num_acc = 1
    sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}$~$p2$~${centerwav}nmc$'
    fn = Andor_dual_gate_spectra_sweep(
        sample_name=sample_name,
        exp_name="$TG-BG=0$",
        cam=cam_ingaas,
        iv=iv,
        vbg_start=-8, vbg_stop=15,
        vtg_start=-8, vtg_stop=15,
        frames=231,
        repeat=1,
        centerwav=centerwav,           # nm
        exposure_time=exp_time,
        Devname=f"{DEV_NAME}",            
        save_root=r"D:\instrument_control_v3_1",
        spec=spec,
        plot=False,
        num_acc=num_acc,
        )
    print(fn)

0.21 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vtg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vbg']
start: [-8.]
end: [-7.8]
steps: 5
variable: ['Vtg']
start: [-8.]
end: [-7.8]
steps: 5
variable: ['Vbg']
start: [-7.8]
end: [-7.6]
steps: 5
variable: ['Vtg']
start: [-7.8]
end: [-7.6]
steps: 5
variable: ['Vbg']
start: [-7.6]
end: [-7.4]
steps: 4
variable: ['Vtg']
start: [-7.6]
end: [-7.4]
steps: 4
variable: ['Vbg']
start: [-7.4]
end: [-7.2]
steps: 5
variable: ['Vtg']
start: [-7.4]
end: [-7.2]
steps: 5
variable: ['Vbg']
start: [-7.2]
end: [-7.]
steps: 5
variable: ['Vtg']
start: [-7.2]
end: [-7.]
steps: 5
variable: ['Vbg']
start: [-7.]
end: [-6.8]
steps: 5
variable: ['Vtg']
start: [-7.]
end: [-6.8]
steps: 5
variable: ['Vbg']
start: [-6.8]
end: [-6.6]
steps: 5
variable: ['Vtg']
start: [-6.8]
end: [-6.6]
steps: 5
variable: ['Vbg']
start: [-6.6]
end: [-6.4]
steps: 4
variable: ['Vtg']
start: [-6.6]
end: [-6.4]
steps: 4
variable: ['Vbg']
start: [-6.4]
end

KeyboardInterrupt: 

In [ ]:
#background

centerwav=1210
exp_time = 60
num_acc = 10
sample_name = f'${DEV_NAME}$~$1.67KPL{exp_time}sx{num_acc}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG=BG=0bg$",
cam=cam_ingaas,
iv=iv,
vbg_start=0, vbg_stop=0,
vtg_start=0, vtg_stop=0,
frames=1,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)
print("saved:", fn)

repeat 1/1
variable: ['Vbg']
start: [0.]
end: [0.]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0.]
steps: 2


In [ ]:
centerwav=1190
exp_time = 4
num_acc = 1
# power_coeff = 0.36
# meas_power =  c_double()
# tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
# power = np.round(meas_power.value*1e6*power_coeff,2){power}uw

sample_name = f'${DEV_NAME}$~$1.67KPL660nm470uW{exp_time}s$~$p8$~${centerwav}nmc$'
# sample_name = f'${DEV_NAME}$~$6KREF{exp_time}x{num_acc}s$~$p1$~${centerwav}nmc$'

# fn = Andor_dual_gate_spectra_sweep(
# sample_name=sample_name,
# exp_name="$TGonly$",
# cam=cam_ingaas,
# iv=iv,
# vbg_start=0, vbg_stop=0,
# vtg_start=-9, vtg_stop=10,
# frames=96,
# repeat=1,
# centerwav=centerwav,           # nm
# exposure_time=exp_time,
# Devname=f"{DEV_NAME}",            
# save_root=r"D:\instrument_control_v3_1",
# spec=spec,
# plot=False,
# num_acc=num_acc,
# )

# fn = Andor_dual_gate_spectra_sweep(
# sample_name=sample_name,
# exp_name="$TGonly rev$",
# cam=cam_ingaas,
# iv=iv,
# vbg_start=0, vbg_stop=0,
# vtg_start=10, vtg_stop=-9,
# frames=96,
# repeat=1,
# centerwav=centerwav,           # nm
# exposure_time=exp_time,
# Devname=f"{DEV_NAME}",            
# save_root=r"D:\instrument_control_v3_1",
# spec=spec,
# plot=False,
# num_acc=num_acc,
# )

# fn = Andor_dual_gate_spectra_sweep(
# sample_name=sample_name,
# exp_name="$BGonly$",
# cam=cam_ingaas,
# iv=iv,
# vbg_start=-9, vbg_stop=10,
# vtg_start=0, vtg_stop=0,
# frames=96,
# repeat=1,
# centerwav=centerwav,           # nm
# exposure_time=exp_time,
# Devname=f"{DEV_NAME}",            
# save_root=r"D:\instrument_control_v3_1",
# spec=spec,
# plot=False,
# num_acc=num_acc,
# )

# fn = Andor_dual_gate_spectra_sweep(
# sample_name=sample_name,
# exp_name="$BGonly rev$",
# cam=cam_ingaas,
# iv=iv,
# vbg_start=10, vbg_stop=-9,
# vtg_start=0, vtg_stop=0,
# frames=96,
# repeat=1,
# centerwav=centerwav,           # nm
# exposure_time=exp_time,
# Devname=f"{DEV_NAME}",            
# save_root=r"D:\instrument_control_v3_1",
# spec=spec,
# plot=False,
# num_acc=num_acc,
# )
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-10, vbg_stop=15,
vtg_start=-10, vtg_stop=15,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

# fn = Andor_dual_gate_spectra_sweep(
# sample_name=sample_name,
# exp_name="$TG-BG=0 rev$",
# cam=cam_ingaas,
# iv=iv,
# vbg_start=10, vbg_stop=-9,
# vtg_start=10, vtg_stop=-9,
# frames=191,
# repeat=1,
# centerwav=centerwav,           # nm
# exposure_time=exp_time,
# Devname=f"{DEV_NAME}",            
# save_root=r"D:\instrument_control_v3_1",
# spec=spec,
# plot=False,
# num_acc=num_acc,
# )
print("saved:", fn)


repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-10.]
steps: 101
variable: ['Vtg']
start: [0.]
end: [-10.]
steps: 101
variable: ['Vbg']
start: [-10.]
end: [-9.9]
steps: 1
variable: ['Vtg']
start: [-10.]
end: [-9.9]
steps: 1
variable: ['Vbg']
start: [-10.]
end: [-9.8]
steps: 2
variable: ['Vtg']
start: [-10.]
end: [-9.8]
steps: 2
variable: ['Vbg']
start: [-9.8]
end: [-9.7]
steps: 2
variable: ['Vtg']
start: [-9.8]
end: [-9.7]
steps: 2
variable: ['Vbg']
start: [-9.7]
end: [-9.6]
steps: 1
variable: ['Vtg']
start: [-9.7]
end: [-9.6]
steps: 1
variable: ['Vbg']
start: [-9.7]
end: [-9.5]
steps: 2
variable: ['Vtg']
start: [-9.7]
end: [-9.5]
steps: 2
variable: ['Vbg']
start: [-9.5]
end: [-9.4]
steps: 1
variable: ['Vtg']
start: [-9.5]
end: [-9.4]
steps: 1
variable: ['Vbg']
start: [-9.5]
end: [-9.3]
steps: 2
variable: ['Vtg']
start: [-9.5]
end: [-9.3]
steps: 2
variable: ['Vbg']
start: [-9.3]
end: [-9.2]
steps: 2
variable: ['Vtg']
start: [-9.3]
end: [-9.2]
steps: 2
variable: ['Vbg']
start: [-9.2]
end

In [ ]:
### megasweep
centerwav=1100
exp_time=1

power_coeff = 0.36
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = np.round(meas_power.value*1e6*power_coeff,2)

sample_name = f'$YZ272MEGASWEEP$~$7KPL633nm{power}uw{exp_time}s$~$pbn1$~${centerwav}nmc$'


Andor_dual_gate_megasweep(sample_name, cam_ingaas, iv,
                               vbg_start=-7, vbg_stop=7, vtg_start=-7, vtg_stop=7,
                               frames=141, repeat=1, centerwav=centerwav, exposure_time=exp_time,
                               plot=False, num_acc=1)

## valley polarization with Andor

In [32]:
from backend.instruments import RS232EP300
ep300 = RS232EP300.RS232EP300(address='ASRL5::INSTR', axes=[1])
ep300.connect()
# ep300.set_speed(2, 20)
ep300.set_speed(1, 20)
# axis 2 : 4deg/s 
# axis 1 : 40deg/s
# ep300.set_position(2,11)

In [33]:
ep300.get_positions(1)

347.5

In [ ]:
stage.move_to(0)
ep300.set_position(1, 302.5+45)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 3)
print(f"{power:} uW")

1.989 uW


In [35]:
import pylablib as pll
from pylablib.devices import Thorlabs
rot_mount = Thorlabs.ElliptecMotor("COM4")

In [58]:
# POWER_EFF =1
rot_mount.move_to(250)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 3)
print(f"{power:} uW")

0.285 uW


In [37]:
# doping dependent
# at each doping, get two PL signals of K and Kp respectively
import time

def move_hwp_with_retry(rot_mount, angle, wait_after_move=3.0, max_retry=3):
    """
    Move Thorlabs rotation mount with retry.
    Prevent occasional sens_error from stopping the whole measurement.
    """

    for attempt in range(max_retry):
        try:
            print(f"  moving HWP to {angle} deg, attempt {attempt + 1}/{max_retry}")
            rot_mount.move_to(angle)
            time.sleep(wait_after_move)
            return

        except Exception as e:
            print(f"  WARNING: failed to move to {angle} deg")
            print(f"  Error: {e}")

            if attempt < max_retry - 1:
                print("  wait 5 s and retry...")
                time.sleep(5)
            else:
                raise RuntimeError(
                    f"Failed to move HWP to {angle} deg after {max_retry} attempts."
                ) from e
            
def Andor_dual_gate_spectra_sweep_HWP_two_angles(
    sample_name, exp_name,
    cam,
    iv,
    rot_mount,
    vbg_start, vbg_stop, vtg_start, vtg_stop,
    frames, repeat,
    centerwav,                 # nm
    exposure_time,
    *,
    Devname="Dev1",
    save_root=".",
    spec=None,
    plot=False,
    num_acc=1,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait=1.0,
):
    """
    Minimize rotation frequency.

    Doping point 1: 205 -> 250
    Doping point 2: 250 -> 205
    Doping point 3: 205 -> 250
    ...

    Data are still saved into separate files according to the actual HWP angle.
    """

    save_dir = os.path.join(save_root, str(Devname), "initial data")
    os.makedirs(save_dir, exist_ok=True)

    file_names = {}
    for angle in angles:
        file_path = os.path.join(
            save_dir,
            f"{sample_name}~{exp_name}~out{angle}deg.csv"
        )
        file_names[angle] = unique_filename(file_path)

    vbgs = np.linspace(vbg_start, vbg_stop, frames)
    vtgs = np.linspace(vtg_start, vtg_stop, frames)

    fig = ax = line = None

    wl_nm = None
    if spec is not None:
        spec.set_wavelength(centerwav * 1e-9)
        spec.setup_pixels_from_camera(cam)
        wl_nm = np.asarray(spec.get_calibration()) * 1e9

    if wl_nm is None:
        n_pix_guess = 512
        wl_nm = np.arange(n_pix_guess, dtype=float)

    wl_nm = wl_nm[::-1]

    for angle in angles:
        with open(file_names[angle], "w") as f:
            cols = np.concatenate(
                (
                    np.array(["Vbg", "Vtg", "Vtg+Vbg", "Vtg-Vbg"], dtype="U"),
                    wl_nm.astype("U")
                )
            ).reshape(1, -1)

            np.savetxt(f, cols, fmt="%s", delimiter=",")

    for i in range(repeat):
        print(f"repeat {i + 1}/{repeat}")

        for idx, (vbg, vtg) in enumerate(zip(vbgs, vtgs)):
            iv.x_goto("Vbg", float(vbg), 0.05, 0.05)
            iv.x_goto("Vtg", float(vtg), 0.05, 0.05)

            print(f"Vbg = {vbg:.5g}, Vtg = {vtg:.5g}")

            if idx % 2 == 0:
                angle_order = angles
            else:
                angle_order = angles[::-1]

            for angle in angle_order:
                print(f"  move HWP to {angle} deg")

                move_hwp_with_retry(rot_mount, angle, wait_after_move=rot_wait)

                x, spectra, fig, ax, line = get_one_spectra(
                    cam,
                    exposure_time=exposure_time,
                    num_acc=num_acc,
                    plot=plot,
                    fig=fig,
                    ax=ax,
                    line=line,
                    spec=spec,
                    include_wl=False,
                    timeout=timeout,
                    discard_first=discard_first,
                )

                if spectra.size != wl_nm.size:
                    raise RuntimeError(
                        f"Spectra length changed ({spectra.size}) != wavelength axis length ({wl_nm.size}). "
                        "ROI/binning changed mid-run; start a new file."
                    )

                row = np.concatenate(
                    (
                        np.array([vbg, vtg, vbg + vtg, vtg - vbg], dtype=np.float64),
                        spectra
                    )
                ).reshape(1, -1)

                with open(file_names[angle], "a") as f:
                    np.savetxt(f, row, fmt="%.5e", delimiter=",")

    iv.x_goto("Vbg", 0, 0.2, 0.1)
    iv.x_goto("Vtg", 0, 0.2, 0.1)

    return file_names



In [38]:
def Andor_dual_gate_spectra_list_HWP_two_angles(
    sample_name, exp_name,
    cam,
    iv,
    rot_mount,
    vbgs,
    vtgs,
    repeat,
    centerwav,                 # nm
    exposure_time,
    *,
    Devname="Dev1",
    save_root=".",
    spec=None,
    plot=False,
    num_acc=1,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait=3.0,
    max_rot_retry=3,
):
    """
    Measure selected non-equally-spaced doping points.

    vbgs and vtgs are arrays/lists, for example:
        vbgs = [-7, -5, -2, 0, 3, 8, 15]
        vtgs = [-7, -5, -2, 0, 3, 8, 15]

    Doping point 1: 205 -> 250
    Doping point 2: 250 -> 205
    Doping point 3: 205 -> 250
    ...

    Data are saved into separate CSV files according to actual HWP angle.
    """

    vbgs = np.asarray(vbgs, dtype=float)
    vtgs = np.asarray(vtgs, dtype=float)

    if vbgs.size != vtgs.size:
        raise ValueError(
            f"vbgs and vtgs must have the same length, but got "
            f"{vbgs.size} and {vtgs.size}."
        )

    save_dir = os.path.join(save_root, str(Devname), "initial data")
    os.makedirs(save_dir, exist_ok=True)

    file_names = {}
    for angle in angles:
        file_path = os.path.join(
            save_dir,
            f"{sample_name}~{exp_name}~out{angle}deg.csv"
        )
        file_names[angle] = unique_filename(file_path)

    fig = ax = line = None

    wl_nm = None
    if spec is not None:
        spec.set_wavelength(centerwav * 1e-9)
        spec.setup_pixels_from_camera(cam)
        wl_nm = np.asarray(spec.get_calibration()) * 1e9

    if wl_nm is None:
        n_pix_guess = 512
        wl_nm = np.arange(n_pix_guess, dtype=float)

    wl_nm = wl_nm[::-1]

    for angle in angles:
        with open(file_names[angle], "w") as f:
            cols = np.concatenate(
                (
                    np.array(["Vbg", "Vtg", "Vtg+Vbg", "Vtg-Vbg"], dtype="U"),
                    wl_nm.astype("U")
                )
            ).reshape(1, -1)

            np.savetxt(f, cols, fmt="%s", delimiter=",")

    for i in range(repeat):
        print(f"repeat {i + 1}/{repeat}")

        for idx, (vbg, vtg) in enumerate(zip(vbgs, vtgs)):
            iv.x_goto("Vbg", float(vbg), 0.2, 0.05)
            iv.x_goto("Vtg", float(vtg), 0.2, 0.05)

            print(f"point {idx + 1}/{len(vbgs)}")
            print(f"Vbg = {vbg:.5g}, Vtg = {vtg:.5g}")

            if idx % 2 == 0:
                angle_order = angles
            else:
                angle_order = angles[::-1]

            for angle in angle_order:
                move_hwp_with_retry(
                    rot_mount,
                    angle,
                    wait_after_move=rot_wait,
                    max_retry=max_rot_retry,
                )

                x, spectra, fig, ax, line = get_one_spectra(
                    cam,
                    exposure_time=exposure_time,
                    num_acc=num_acc,
                    plot=plot,
                    fig=fig,
                    ax=ax,
                    line=line,
                    spec=spec,
                    include_wl=False,
                    timeout=timeout,
                    discard_first=discard_first,
                )

                if spectra.size != wl_nm.size:
                    raise RuntimeError(
                        f"Spectra length changed ({spectra.size}) != wavelength axis length ({wl_nm.size}). "
                        "ROI/binning changed mid-run; start a new file."
                    )

                row = np.concatenate(
                    (
                        np.array([vbg, vtg, vbg + vtg, vtg - vbg], dtype=np.float64),
                        spectra
                    )
                ).reshape(1, -1)

                with open(file_names[angle], "a") as f:
                    np.savetxt(f, row, fmt="%.5e", delimiter=",")

    iv.x_goto("Vbg", 0, 0.2, 0.1)
    iv.x_goto("Vtg", 0, 0.2, 0.1)

    return file_names

In [53]:
#v3_00.test Kp-(K&Kp) 420uW
rot_wait = 2
hwin = 347.5
ep300.set_position(1, hwin)

stage.move_to(3000)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 10
num_acc = 1
sample_name = f'$vglist$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p4x5y5$~${centerwav}nmc$'
vbgs = [ 0, 1]
vtgs = [ 0, 1]
fn = Andor_dual_gate_spectra_list_HWP_two_angles(
        sample_name =sample_name, exp_name="$TG-BG=0$",
        cam=cam_ingaas,
        iv=iv,
        rot_mount = rot_mount,
        vbgs = vbgs,
        vtgs = vtgs,
        repeat = 1,
        centerwav = centerwav,                 # nm
        exposure_time =exp_time,
        Devname=f"{DEV_NAME}",
        save_root=r"D:\instrument_control_v3_1",
        spec=spec,
        plot=False,
        num_acc=num_acc,
        timeout=10.0,
        discard_first=False,
        angles=(205, 250),
        rot_wait= rot_wait,
        max_rot_retry=3,
)
print(fn)

181.87 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [0.]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0.]
steps: 2
point 1/2
Vbg = 0, Vtg = 0
  moving HWP to 205 deg, attempt 1/3
  moving HWP to 250 deg, attempt 1/3
variable: ['Vbg']
start: [0.]
end: [1.]
steps: 21
variable: ['Vtg']
start: [0.]
end: [1.]
steps: 21
point 2/2
Vbg = 1, Vtg = 1
  moving HWP to 250 deg, attempt 1/3
  moving HWP to 205 deg, attempt 1/3
variable: ['Vbg']
start: [1.]
end: [0]
steps: 6
variable: ['Vtg']
start: [1.]
end: [0]
steps: 6
{205: 'D:\\instrument_control_v3_1\\YZD300_202604_PL\\initial data\\$vglist$~$1.67KPL660nm181.87uW10sx1in347.5$~$p4x5y5$~$1190nmc$~$TG-BG=0$~out205deg.csv', 250: 'D:\\instrument_control_v3_1\\YZD300_202604_PL\\initial data\\$vglist$~$1.67KPL660nm181.87uW10sx1in347.5$~$p4x5y5$~$1190nmc$~$TG-BG=0$~out250deg.csv'}


In [139]:
#v3_01. Kp-(K&Kp) powe dependent
rot_wait = 2
hwin = 347.5
ep300.set_position(1, hwin)
num_acc = 1

# #powers    = [420,  350,   300,  200,  100,  50,  20,   10,    5,    2,    1]      #uW
# pos_stages = [3300, 3201, 3160, 3033, 2760, 2420, 1890, 1520, 1060, 560,  150] 
# centerwavs = [1200, 1200, 1200, 1200, 1200, 1210, 1210, 1220, 1220, 1230, 1230]      # nm
# exp_times  = [10,    10,   10,   15,   15,   15,   20,   20,   20,   45,   60]      # s

# #powers    = [420,  350,   300,  200,  100,  50,  20,   10,    5,    2,    1]      #uW
# pos_stages = [3300, 3201, 3160, 3033, 2760, 2420, 1890, 1520, 1060, 560,  150] 
# centerwavs = [1200, 1200, 1200, 1200, 1200, 1210, 1210, 1210, 1210, 1210, 1210]      # nm, for H
# exp_times  = [15,    15,   15,   20,   20,   20,   30,   30,   30,   60,   60]      # s

# #powers    = [420,  200,   50,   10,    5,    1]      #uW
# pos_stages = [3300, 3033, 2420, 1520, 1060,  150] 
# centerwavs = [1200, 1200, 1210, 1220, 1220, 1230]      # nm
# exp_times  = [10,    15,   15,   20,   20,   60]      # s

#powers    = [420,  200,   50,   10,    5,    1]      #uW
pos_stages = [3300, 3033, 2420, 1520, 1060,  150] 
centerwavs = [1200, 1200, 1210, 1210, 1210, 1210]      # nm for H
exp_times  = [15,    20,   20,   30,   30,   60]      # s

# #powers    = [      1]      #uW
# pos_stages = [   150] 
# centerwavs = [  1210]      # nm, for H
# exp_times  = [   60] 

# # powers    = [ 0.1]      #uW with ND1
# pos_stages = [ 140] 
# centerwavs = [ 1210]      # nm
# exp_times  = [  90]      # s

for pos_stage, centerwav, exp_time in zip(pos_stages, centerwavs, exp_times):
    stage.move_to(pos_stage)
    POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
    power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 3)
    print(f"{power:} uW")
    # centerwav=1190
    # exp_time = 10
    num_acc = num_acc
    sample_name = f'$vglist$~$1.67K3TPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p2$~${centerwav}nmc$'
    # vbgs = [-5, -2.5, -1, 0, 0.3, 2, 5, 7.5, 10, 12.5]
    # vtgs = [-5, -2.5, -1, 0, 0.3, 2, 5, 7.5, 10, 12.5]
    vbgs = [0, 0.3]
    vtgs = [0, 0.3]
    fn = Andor_dual_gate_spectra_list_HWP_two_angles(
        sample_name =sample_name, exp_name="$TG-BG=0$",
        cam=cam_ingaas,
        iv=iv,
        rot_mount = rot_mount,
        vbgs = vbgs,
        vtgs = vtgs,
        repeat = 1,
        centerwav = centerwav,                 # nm
        exposure_time =exp_time,
        Devname=f"{DEV_NAME}",
        save_root=r"D:\instrument_control_v3_1",
        spec=spec,
        plot=False,
        num_acc=num_acc,
        timeout=10.0,
        discard_first=False,
        angles=(205, 250),
        rot_wait= rot_wait,
        max_rot_retry=3,
    )
    print(fn)

423.245 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [0.]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0.]
steps: 2
point 1/2
Vbg = 0, Vtg = 0
  moving HWP to 205 deg, attempt 1/3
  moving HWP to 250 deg, attempt 1/3
variable: ['Vbg']
start: [0.]
end: [0.3]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0.3]
steps: 2
point 2/2
Vbg = 0.3, Vtg = 0.3
  moving HWP to 250 deg, attempt 1/3
  moving HWP to 205 deg, attempt 1/3
variable: ['Vbg']
start: [0.3]
end: [0]
steps: 2
variable: ['Vtg']
start: [0.3]
end: [0]
steps: 2
{205: 'D:\\instrument_control_v3_1\\YZD300_202604_PL\\initial data\\$vglist$~$1.67K3TPL660nm423.245uW15sx1in347.5$~$p2$~$1200nmc$~$TG-BG=0$~out205deg.csv', 250: 'D:\\instrument_control_v3_1\\YZD300_202604_PL\\initial data\\$vglist$~$1.67K3TPL660nm423.245uW15sx1in347.5$~$p2$~$1200nmc$~$TG-BG=0$~out250deg.csv'}
201.41 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [0.]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0.]
steps: 2
point 1/2
Vbg = 0, Vtg = 0
  moving HWP to 205

In [55]:
#v3_02. Kp-(K&Kp) power dependent 
rot_wait = 2
hwin = 347.5
ep300.set_position(1, hwin)
num_acc = 1

# #powers    = [420,  350,   300,  200,  100,  50,  20,   10,    5,    2,    1]      #uW
# pos_stages = [3300, 3201, 3160, 3033, 2760, 2420, 1890, 1520, 1060, 560,  150] 
# centerwavs = [1200, 1200, 1200, 1200, 1200, 1210, 1210, 1220, 1220, 1230, 1230]      # nm
# exp_times  = [10,    10,   10,   15,   15,   15,   20,   20,   20,   45,   60]      # s

#powers    = [420,  200,  100,    2,    5,   20]      #uW
# pos_stages = [3300, 3033, 2760,  560, 1060, 1890] 
# centerwavs = [1200, 1200, 1200, 1230, 1230, 1220]      # nm
# exp_times  = [15,    20,    20,  60,   30,   30]      # s

# #powers    = [420,  200,  100,    1,    5,   20]      #uW
# pos_stages = [3300, 3033, 2760,  140, 1060, 1890] 
# centerwavs = [1200, 1200, 1200, 1210, 1210, 1210]      # nm for H
# exp_times  = [15,    20,    20,  60,   30,   30]      # s

#powers    = [  1,   20,   2]      #uW
pos_stages = [ 160, 1890, 560] 
centerwavs = [1210, 1210, 1210]      # nm for H
exp_times  = [ 60,   30,   60]      # s

for pos_stage, centerwav, exp_time in zip(pos_stages, centerwavs, exp_times):
    stage.move_to(pos_stage)
    time.sleep(1)
    stage.move_to(pos_stage)
    POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
    power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
    print(f"{power:} uW")
    # centerwav=1190
    # exp_time = 10
    num_acc = 1
    sample_name = f'$vgmap$~$1.67K0TPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p2$~${centerwav}nmc$'
    fn = Andor_dual_gate_spectra_sweep_HWP_two_angles(
        sample_name =sample_name, exp_name="$TG-BG=0$",
        cam=cam_ingaas,
        iv=iv,
        rot_mount = rot_mount,
        vbg_start = -8, vbg_stop = 15, 
        vtg_start = -8, vtg_stop = 15,
        frames = 116, 
        repeat = 1,
        centerwav = centerwav,                 # nm
        exposure_time =exp_time,
        Devname=f"{DEV_NAME}",
        save_root=r"D:\instrument_control_v3_1",
        spec=spec,
        plot=False,
        num_acc=num_acc,
        timeout=10.0,
        discard_first=False,
        angles=(205, 250),
        rot_wait= rot_wait,              # 转完半波片后等待时间，单位 s
    )
    print(fn)

1.0 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vtg']
start: [0.]
end: [-8.]
steps: 161
Vbg = -8, Vtg = -8
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
variable: ['Vbg']
start: [-8.]
end: [-7.8]
steps: 5
variable: ['Vtg']
start: [-8.]
end: [-7.8]
steps: 5
Vbg = -7.8, Vtg = -7.8
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
variable: ['Vbg']
start: [-7.8]
end: [-7.6]
steps: 5
variable: ['Vtg']
start: [-7.8]
end: [-7.6]
steps: 5
Vbg = -7.6, Vtg = -7.6
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
variable: ['Vbg']
start: [-7.6]
end: [-7.4]
steps: 4
variable: ['Vtg']
start: [-7.6]
end: [-7.4]
steps: 4
Vbg = -7.4, Vtg = -7.4
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
va

In [ ]:
#v2_00.test Kp-(K&Kp) 420uW
stage.move_to(3300)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p4x5y5$~${centerwav}nmc$'

fn = Andor_dual_gate_spectra_sweep_HWP_two_angles(
    sample_name =sample_name, exp_name="$TG-BG=0$",
    cam=cam_ingaas,
    iv=iv,
    rot_mount = rot_mount,
    vbg_start = 0, vbg_stop = 1, 
    vtg_start = 0, vtg_stop = 1,
    frames = 6, 
    repeat = 1,
    centerwav = centerwav,                 # nm
    exposure_time =exp_time,
    Devname=f"{DEV_NAME}",
    save_root=r"D:\instrument_control_v3_1",
    spec=spec,
    plot=False,
    num_acc=num_acc,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait=rot_wait,              # 转完半波片后等待时间，单位 s
)


420.85 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [0.]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0.]
steps: 2
Vbg = 0, Vtg = 0
  move HWP to 205 deg
  move HWP to 250 deg
variable: ['Vbg']
start: [0.]
end: [0.2]
steps: 3
variable: ['Vtg']
start: [0.]
end: [0.2]
steps: 3
Vbg = 0.2, Vtg = 0.2
  move HWP to 250 deg
  move HWP to 205 deg
variable: ['Vbg']
start: [0.2]
end: [0.4]
steps: 3
variable: ['Vtg']
start: [0.2]
end: [0.4]
steps: 3
Vbg = 0.4, Vtg = 0.4
  move HWP to 205 deg
  move HWP to 250 deg
variable: ['Vbg']
start: [0.4]
end: [0.6]
steps: 3
variable: ['Vtg']
start: [0.4]
end: [0.6]
steps: 3
Vbg = 0.6, Vtg = 0.6
  move HWP to 250 deg
  move HWP to 205 deg
variable: ['Vbg']
start: [0.6]
end: [0.8]
steps: 2
variable: ['Vtg']
start: [0.6]
end: [0.8]
steps: 2
Vbg = 0.8, Vtg = 0.8
  move HWP to 205 deg
  move HWP to 250 deg
variable: ['Vbg']
start: [0.8]
end: [1.]
steps: 2
variable: ['Vtg']
start: [0.8]
end: [1.]
steps: 2
Vbg = 1, Vtg = 1
  move HWP to 250 deg
  move HWP to 2

In [51]:
rot_wait = 2.0
#v2_01. Kp-(K&Kp) 420uW
stage.move_to(3300)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1200
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67K9TPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p4x5y5$~${centerwav}nmc$'

fn = Andor_dual_gate_spectra_sweep_HWP_two_angles(
    sample_name =sample_name, exp_name="$TG-BG=0$",
    cam=cam_ingaas,
    iv=iv,
    rot_mount = rot_mount,
    vbg_start = -8, vbg_stop = 15, 
    vtg_start = -8, vtg_stop = 15,
    frames = 116, 
    repeat = 1,
    centerwav = centerwav,                 # nm
    exposure_time =exp_time,
    Devname=f"{DEV_NAME}",
    save_root=r"D:\instrument_control_v3_1",
    spec=spec,
    plot=False,
    num_acc=num_acc,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait= rot_wait,              # 转完半波片后等待时间，单位 s
)


#v2_02. Kp-(K&Kp) 200uW
stage.move_to(3033)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1200
exp_time = 15
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67K9TPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p4x5y5$~${centerwav}nmc$'

fn = Andor_dual_gate_spectra_sweep_HWP_two_angles(
    sample_name =sample_name, exp_name="$TG-BG=0$",
    cam=cam_ingaas,
    iv=iv,
    rot_mount = rot_mount,
    vbg_start = -8, vbg_stop = 15, 
    vtg_start = -8, vtg_stop = 15,
    frames = 116, 
    repeat = 1,
    centerwav = centerwav,                 # nm
    exposure_time =exp_time,
    Devname=f"{DEV_NAME}",
    save_root=r"D:\instrument_control_v3_1",
    spec=spec,
    plot=False,
    num_acc=num_acc,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait=rot_wait,              # 转完半波片后等待时间，单位 s
)


#v2_03. Kp-(K&Kp) 100uW
stage.move_to(2760)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1200
exp_time = 15
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67K9TPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p4x5y5$~${centerwav}nmc$'

fn = Andor_dual_gate_spectra_sweep_HWP_two_angles(
    sample_name =sample_name, exp_name="$TG-BG=0$",
    cam=cam_ingaas,
    iv=iv,
    rot_mount = rot_mount,
    vbg_start = -8, vbg_stop = 15, 
    vtg_start = -8, vtg_stop = 15,
    frames = 116, 
    repeat = 1,
    centerwav = centerwav,                 # nm
    exposure_time =exp_time,
    Devname=f"{DEV_NAME}",
    save_root=r"D:\instrument_control_v3_1",
    spec=spec,
    plot=False,
    num_acc=num_acc,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait=rot_wait,              # 转完半波片后等待时间，单位 s
)


#v2_04. Kp-(K&Kp) 1uW
stage.move_to(150)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1230
exp_time = 60
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67K9TPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p4x5y5$~${centerwav}nmc$'

fn = Andor_dual_gate_spectra_sweep_HWP_two_angles(
    sample_name =sample_name, exp_name="$TG-BG=0$",
    cam=cam_ingaas,
    iv=iv,
    rot_mount = rot_mount,
    vbg_start = -8, vbg_stop = 15, 
    vtg_start = -8, vtg_stop = 15,
    frames = 116, 
    repeat = 1,
    centerwav = centerwav,                 # nm
    exposure_time =exp_time,
    Devname=f"{DEV_NAME}",
    save_root=r"D:\instrument_control_v3_1",
    spec=spec,
    plot=False,
    num_acc=num_acc,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait=rot_wait,              # 转完半波片后等待时间，单位 s
)


#v2_05. Kp-(K&Kp) 5uW
stage.move_to(1060)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1220
exp_time = 20
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67K9TPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p4x5y5$~${centerwav}nmc$'

fn = Andor_dual_gate_spectra_sweep_HWP_two_angles(
    sample_name =sample_name, exp_name="$TG-BG=0$",
    cam=cam_ingaas,
    iv=iv,
    rot_mount = rot_mount,
    vbg_start = -8, vbg_stop = 15, 
    vtg_start = -8, vtg_stop = 15,
    frames = 116, 
    repeat = 1,
    centerwav = centerwav,                 # nm
    exposure_time =exp_time,
    Devname=f"{DEV_NAME}",
    save_root=r"D:\instrument_control_v3_1",
    spec=spec,
    plot=False,
    num_acc=num_acc,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait=rot_wait,              # 转完半波片后等待时间，单位 s
)

#v2_06. Kp-(K&Kp) 20uW
stage.move_to(1890)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1210
exp_time = 20
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67K9TPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p4x5y5$~${centerwav}nmc$'

fn = Andor_dual_gate_spectra_sweep_HWP_two_angles(
    sample_name =sample_name, exp_name="$TG-BG=0$",
    cam=cam_ingaas,
    iv=iv,
    rot_mount = rot_mount,
    vbg_start = -8, vbg_stop = 15, 
    vtg_start = -8, vtg_stop = 15,
    frames = 116, 
    repeat = 1,
    centerwav = centerwav,                 # nm
    exposure_time =exp_time,
    Devname=f"{DEV_NAME}",
    save_root=r"D:\instrument_control_v3_1",
    spec=spec,
    plot=False,
    num_acc=num_acc,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait=rot_wait,              # 转完半波片后等待时间，单位 s
)

422.98 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vtg']
start: [0.]
end: [-8.]
steps: 161
Vbg = -8, Vtg = -8
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
variable: ['Vbg']
start: [-8.]
end: [-7.8]
steps: 5
variable: ['Vtg']
start: [-8.]
end: [-7.8]
steps: 5
Vbg = -7.8, Vtg = -7.8
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
variable: ['Vbg']
start: [-7.8]
end: [-7.6]
steps: 5
variable: ['Vtg']
start: [-7.8]
end: [-7.6]
steps: 5
Vbg = -7.6, Vtg = -7.6
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
variable: ['Vbg']
start: [-7.6]
end: [-7.4]
steps: 4
variable: ['Vtg']
start: [-7.6]
end: [-7.4]
steps: 4
Vbg = -7.4, Vtg = -7.4
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3

In [48]:
rot_wait = 2.0
#v2_01. Kp-(K&Kp) 420uW
stage.move_to(3300)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}$~$p4x5y5$~${centerwav}nmc$'

fn = Andor_dual_gate_spectra_sweep_HWP_two_angles(
    sample_name =sample_name, exp_name="$TG-BG=0$",
    cam=cam_ingaas,
    iv=iv,
    rot_mount = rot_mount,
    vbg_start = -8, vbg_stop = 15, 
    vtg_start = -8, vtg_stop = 15,
    frames = 116, 
    repeat = 1,
    centerwav = centerwav,                 # nm
    exposure_time =exp_time,
    Devname=f"{DEV_NAME}",
    save_root=r"D:\instrument_control_v3_1",
    spec=spec,
    plot=False,
    num_acc=num_acc,
    timeout=10.0,
    discard_first=False,
    angles=(205, 250),
    rot_wait= rot_wait,              # 转完半波片后等待时间，单位 s
)

422.27 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vtg']
start: [0.]
end: [-8.]
steps: 161
Vbg = -8, Vtg = -8
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
variable: ['Vbg']
start: [-8.]
end: [-7.8]
steps: 5
variable: ['Vtg']
start: [-8.]
end: [-7.8]
steps: 5
Vbg = -7.8, Vtg = -7.8
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
variable: ['Vbg']
start: [-7.8]
end: [-7.6]
steps: 5
variable: ['Vtg']
start: [-7.8]
end: [-7.6]
steps: 5
Vbg = -7.6, Vtg = -7.6
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
variable: ['Vbg']
start: [-7.6]
end: [-7.4]
steps: 4
variable: ['Vtg']
start: [-7.6]
end: [-7.4]
steps: 4
Vbg = -7.4, Vtg = -7.4
  move HWP to 250 deg
  moving HWP to 250 deg, attempt 1/3
  move HWP to 205 deg
  moving HWP to 205 deg, attempt 1/3

In [510]:
# #1. K-K 1uW
# stage.move_to(130)
# hwin = 302.5
# ep300.set_position(1, hwin)
# POWER_EFF = 0.41   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
# hwout = 249
# rot_mount.move_to(hwout)
# power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
# print(f"{power:} uW")
# centerwav=1230
# exp_time = 30
# num_acc = 1
# sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
# fn = Andor_dual_gate_spectra_sweep(
# sample_name=sample_name,
# exp_name="$TG-BG=0$",
# cam=cam_ingaas,
# iv=iv,
# vbg_start=-9, vbg_stop=16,
# vtg_start=-9, vtg_stop=16,
# frames=251,
# repeat=1,
# centerwav=centerwav,           # nm
# exposure_time=exp_time,
# Devname=f"{DEV_NAME}",            
# save_root=r"D:\instrument_control_v3_1",
# spec=spec,
# plot=False,
# num_acc=num_acc,
# )

# #2. K-Kp 1uW
# hwout = 204
# rot_mount.move_to(hwout)
# power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
# print(f"{power:} uW")
# centerwav=1230
# exp_time = 30
# num_acc = 1
# sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
# fn = Andor_dual_gate_spectra_sweep(
# sample_name=sample_name,
# exp_name="$TG-BG=0$",
# cam=cam_ingaas,
# iv=iv,
# vbg_start=-9, vbg_stop=16,
# vtg_start=-9, vtg_stop=16,
# frames=251,
# repeat=1,
# centerwav=centerwav,           # nm
# exposure_time=exp_time,
# Devname=f"{DEV_NAME}",            
# save_root=r"D:\instrument_control_v3_1",
# spec=spec,
# plot=False,
# num_acc=num_acc,
# )



#3. Kp-K 1uW
stage.move_to(140)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 249
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1230
exp_time = 30
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

#4. Kp-Kp 1uW
hwout = 204
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1230
exp_time = 30
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)


#5. Kp-K 20uW
stage.move_to(1890)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 249
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1210
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

#6. Kp-Kp 20uW
hwout = 204
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1210
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)


#7. Kp-K 100uW
stage.move_to(2760)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 249
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 5
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

#8. Kp-Kp 100uW
hwout = 204
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 5
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)



#9. Kp-K 200uW
stage.move_to(3033)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 249
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 5
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

#10. Kp-Kp 200uW
hwout = 204
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 5
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)


#11. Kp-K 5uW
stage.move_to(1060)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 249
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1220
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

#12. Kp-Kp 5uW
hwout = 204
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1220
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)


0.97 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-9.]
steps: 91
variable: ['Vtg']
start: [0.]
end: [-9.]
steps: 91
variable: ['Vbg']
start: [-9.]
end: [-8.9]
steps: 1
variable: ['Vtg']
start: [-9.]
end: [-8.9]
steps: 1
variable: ['Vbg']
start: [-9.]
end: [-8.8]
steps: 2
variable: ['Vtg']
start: [-9.]
end: [-8.8]
steps: 2
variable: ['Vbg']
start: [-8.8]
end: [-8.7]
steps: 2
variable: ['Vtg']
start: [-8.8]
end: [-8.7]
steps: 2
variable: ['Vbg']
start: [-8.7]
end: [-8.6]
steps: 1
variable: ['Vtg']
start: [-8.7]
end: [-8.6]
steps: 1
variable: ['Vbg']
start: [-8.7]
end: [-8.5]
steps: 2
variable: ['Vtg']
start: [-8.7]
end: [-8.5]
steps: 2
variable: ['Vbg']
start: [-8.5]
end: [-8.4]
steps: 1
variable: ['Vtg']
start: [-8.5]
end: [-8.4]
steps: 1
variable: ['Vbg']
start: [-8.5]
end: [-8.3]
steps: 2
variable: ['Vtg']
start: [-8.5]
end: [-8.3]
steps: 2
variable: ['Vbg']
start: [-8.3]
end: [-8.2]
steps: 2
variable: ['Vtg']
start: [-8.3]
end: [-8.2]
steps: 2
variable: ['Vbg']
start: [-8.2]
end

In [517]:
#13. K-K 5uW
stage.move_to(1030)
hwin = 302.5
ep300.set_position(1, hwin)
POWER_EFF = 0.41   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 249
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1220
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

#14. K-Kp 5uW
hwout = 204
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1220
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-9, vbg_stop=16,
vtg_start=-9, vtg_stop=16,
frames=251,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

4.98 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-9.]
steps: 91
variable: ['Vtg']
start: [0.]
end: [-9.]
steps: 91
variable: ['Vbg']
start: [-9.]
end: [-8.9]
steps: 1
variable: ['Vtg']
start: [-9.]
end: [-8.9]
steps: 1
variable: ['Vbg']
start: [-9.]
end: [-8.8]
steps: 2
variable: ['Vtg']
start: [-9.]
end: [-8.8]
steps: 2
variable: ['Vbg']
start: [-8.8]
end: [-8.7]
steps: 2
variable: ['Vtg']
start: [-8.8]
end: [-8.7]
steps: 2
variable: ['Vbg']
start: [-8.7]
end: [-8.6]
steps: 1
variable: ['Vtg']
start: [-8.7]
end: [-8.6]
steps: 1
variable: ['Vbg']
start: [-8.7]
end: [-8.5]
steps: 2
variable: ['Vtg']
start: [-8.7]
end: [-8.5]
steps: 2
variable: ['Vbg']
start: [-8.5]
end: [-8.4]
steps: 1
variable: ['Vtg']
start: [-8.5]
end: [-8.4]
steps: 1
variable: ['Vbg']
start: [-8.5]
end: [-8.3]
steps: 2
variable: ['Vtg']
start: [-8.5]
end: [-8.3]
steps: 2
variable: ['Vbg']
start: [-8.3]
end: [-8.2]
steps: 2
variable: ['Vtg']
start: [-8.3]
end: [-8.2]
steps: 2
variable: ['Vbg']
start: [-8.2]
end

In [102]:
# #1. K-K 420uW
# stage.move_to(3250)
# hwin = 302.5
# ep300.set_position(1, hwin)
# POWER_EFF = 0.41   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
# hwout = 249
# rot_mount.move_to(hwout)
# power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
# print(f"{power:} uW")
# centerwav=1190
# exp_time = 4
# num_acc = 1
# sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
# fn = Andor_dual_gate_spectra_sweep(
# sample_name=sample_name,
# exp_name="$TG-BG=0$",
# cam=cam_ingaas,
# iv=iv,
# vbg_start=-9, vbg_stop=16,
# vtg_start=-9, vtg_stop=16,
# frames=251,
# repeat=1,
# centerwav=centerwav,           # nm
# exposure_time=exp_time,
# Devname=f"{DEV_NAME}",            
# save_root=r"D:\instrument_control_v3_1",
# spec=spec,
# plot=False,
# num_acc=num_acc,
# )

# #2. K-Kp 420uW
# hwout = 204
# rot_mount.move_to(hwout)
# power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
# print(f"{power:} uW")
# centerwav=1190
# exp_time = 4
# num_acc = 1
# sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
# fn = Andor_dual_gate_spectra_sweep(
# sample_name=sample_name,
# exp_name="$TG-BG=0$",
# cam=cam_ingaas,
# iv=iv,
# vbg_start=-9, vbg_stop=16,
# vtg_start=-9, vtg_stop=16,
# frames=251,
# repeat=1,
# centerwav=centerwav,           # nm
# exposure_time=exp_time,
# Devname=f"{DEV_NAME}",            
# save_root=r"D:\instrument_control_v3_1",
# spec=spec,
# plot=False,
# num_acc=num_acc,
# )

#3. Kp-K 420uW
stage.move_to(3300)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 250
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 5
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p2$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-8, vbg_stop=15,
vtg_start=-8, vtg_stop=15,
frames=116,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

#4. Kp-Kp 420uW
hwout = 205
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 5
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p2$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-8, vbg_stop=15,
vtg_start=-8, vtg_stop=15,
frames=116,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

420.05 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vtg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vbg']
start: [-8.]
end: [-7.8]
steps: 5
variable: ['Vtg']
start: [-8.]
end: [-7.8]
steps: 5
variable: ['Vbg']
start: [-7.8]
end: [-7.6]
steps: 5
variable: ['Vtg']
start: [-7.8]
end: [-7.6]
steps: 5
variable: ['Vbg']
start: [-7.6]
end: [-7.4]
steps: 4
variable: ['Vtg']
start: [-7.6]
end: [-7.4]
steps: 4
variable: ['Vbg']
start: [-7.4]
end: [-7.2]
steps: 5
variable: ['Vtg']
start: [-7.4]
end: [-7.2]
steps: 5
variable: ['Vbg']
start: [-7.2]
end: [-7.]
steps: 5
variable: ['Vtg']
start: [-7.2]
end: [-7.]
steps: 5
variable: ['Vbg']
start: [-7.]
end: [-6.8]
steps: 5
variable: ['Vtg']
start: [-7.]
end: [-6.8]
steps: 5
variable: ['Vbg']
start: [-6.8]
end: [-6.6]
steps: 5
variable: ['Vtg']
start: [-6.8]
end: [-6.6]
steps: 5
variable: ['Vbg']
start: [-6.6]
end: [-6.4]
steps: 4
variable: ['Vtg']
start: [-6.6]
end: [-6.4]
steps: 4
variable: ['Vbg']
start: [-6.4]
e

KeyboardInterrupt: 

In [530]:
#15. K-Kp 420uW without 2nd HWP
stage.move_to(3248)
hwin = 302.5
ep300.set_position(1, hwin)
POWER_EFF = 0.41   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 0
# rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 4
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-8, vbg_stop=16,
vtg_start=-8, vtg_stop=16,
frames=241,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)


#16. Kp-Kp 420uW  without 2nd HWP
stage.move_to(3300)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
# hwout = 249
# rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 4
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-8, vbg_stop=16,
vtg_start=-8, vtg_stop=16,
frames=241,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)


418.72 uW
repeat 1/1
variable: ['Vbg']
start: [-8.7]
end: [-8.]
steps: 7
variable: ['Vtg']
start: [-8.7]
end: [-8.]
steps: 7
variable: ['Vbg']
start: [-8.]
end: [-7.9]
steps: 1
variable: ['Vtg']
start: [-8.]
end: [-7.9]
steps: 1
variable: ['Vbg']
start: [-8.]
end: [-7.8]
steps: 3
variable: ['Vtg']
start: [-8.]
end: [-7.8]
steps: 3
variable: ['Vbg']
start: [-7.8]
end: [-7.7]
steps: 1
variable: ['Vtg']
start: [-7.8]
end: [-7.7]
steps: 1
variable: ['Vbg']
start: [-7.8]
end: [-7.6]
steps: 3
variable: ['Vtg']
start: [-7.8]
end: [-7.6]
steps: 3
variable: ['Vbg']
start: [-7.6]
end: [-7.5]
steps: 1
variable: ['Vtg']
start: [-7.6]
end: [-7.5]
steps: 1
variable: ['Vbg']
start: [-7.6]
end: [-7.4]
steps: 2
variable: ['Vtg']
start: [-7.6]
end: [-7.4]
steps: 2
variable: ['Vbg']
start: [-7.4]
end: [-7.3]
steps: 2
variable: ['Vtg']
start: [-7.4]
end: [-7.3]
steps: 2
variable: ['Vbg']
start: [-7.3]
end: [-7.2]
steps: 1
variable: ['Vtg']
start: [-7.3]
end: [-7.2]
steps: 1
variable: ['Vbg']
start: [-7.3]

In [49]:
#17. test Kp-K 420uW repeat01
stage.move_to(3300)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 250
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-8, vbg_stop=15,
vtg_start=-8, vtg_stop=15,
frames=116,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)


#18. test Kp-K 420uW repeat02
# stage.move_to(3300)
# hwin = 347.5
# ep300.set_position(1, hwin)
# POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
# hwout = 249
# rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1190
exp_time = 10
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-8, vbg_stop=15,
vtg_start=-8, vtg_stop=15,
frames=115,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

419.07 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vtg']
start: [0.]
end: [-8.]
steps: 161
variable: ['Vbg']
start: [-8.]
end: [-7.8]
steps: 5
variable: ['Vtg']
start: [-8.]
end: [-7.8]
steps: 5
variable: ['Vbg']
start: [-7.8]
end: [-7.6]
steps: 5
variable: ['Vtg']
start: [-7.8]
end: [-7.6]
steps: 5
variable: ['Vbg']
start: [-7.6]
end: [-7.4]
steps: 4
variable: ['Vtg']
start: [-7.6]
end: [-7.4]
steps: 4
variable: ['Vbg']
start: [-7.4]
end: [-7.2]
steps: 5
variable: ['Vtg']
start: [-7.4]
end: [-7.2]
steps: 5
variable: ['Vbg']
start: [-7.2]
end: [-7.]
steps: 5
variable: ['Vtg']
start: [-7.2]
end: [-7.]
steps: 5
variable: ['Vbg']
start: [-7.]
end: [-6.8]
steps: 5
variable: ['Vtg']
start: [-7.]
end: [-6.8]
steps: 5
variable: ['Vbg']
start: [-6.8]
end: [-6.6]
steps: 5
variable: ['Vtg']
start: [-6.8]
end: [-6.6]
steps: 5
variable: ['Vbg']
start: [-6.6]
end: [-6.4]
steps: 4
variable: ['Vtg']
start: [-6.6]
end: [-6.4]
steps: 4
variable: ['Vbg']
start: [-6.4]
e

In [594]:
#3. Kp-K 1uW
stage.move_to(140)
hwin = 347.5
ep300.set_position(1, hwin)
POWER_EFF = 0.38   #0.41 for 302.5deg of hwin; 0.38 for 347.5deg
hwout = 250
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1230
exp_time = 30
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-2, vbg_stop=4,
vtg_start=-2, vtg_stop=4,
frames=61,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

#4. Kp-Kp 1uW
hwout = 205
rot_mount.move_to(hwout)
power = np.round(float(inst.query("MEAS:POW?").strip()) * 1_000_000 *POWER_EFF, 2)
print(f"{power:} uW")
centerwav=1230
exp_time = 30
num_acc = 1
sample_name = f'${DEV_NAME}$~$1.67KPL660nm{power}uW{exp_time}sx{num_acc}in{hwin}out{hwout}$~$p4x5y5$~${centerwav}nmc$'
fn = Andor_dual_gate_spectra_sweep(
sample_name=sample_name,
exp_name="$TG-BG=0$",
cam=cam_ingaas,
iv=iv,
vbg_start=-2, vbg_stop=4,
vtg_start=-2, vtg_stop=4,
frames=61,
repeat=1,
centerwav=centerwav,           # nm
exposure_time=exp_time,
Devname=f"{DEV_NAME}",            
save_root=r"D:\instrument_control_v3_1",
spec=spec,
plot=False,
num_acc=num_acc,
)

0.99 uW
repeat 1/1
variable: ['Vbg']
start: [0.]
end: [-2.]
steps: 21
variable: ['Vtg']
start: [0.]
end: [-2.]
steps: 21
variable: ['Vbg']
start: [-2.]
end: [-1.9]
steps: 2
variable: ['Vtg']
start: [-2.]
end: [-1.9]
steps: 2
variable: ['Vbg']
start: [-1.9]
end: [-1.8]
steps: 1
variable: ['Vtg']
start: [-1.9]
end: [-1.8]
steps: 1
variable: ['Vbg']
start: [-1.9]
end: [-1.7]
steps: 2
variable: ['Vtg']
start: [-1.9]
end: [-1.7]
steps: 2
variable: ['Vbg']
start: [-1.7]
end: [-1.6]
steps: 1
variable: ['Vtg']
start: [-1.7]
end: [-1.6]
steps: 1
variable: ['Vbg']
start: [-1.7]
end: [-1.5]
steps: 2
variable: ['Vtg']
start: [-1.7]
end: [-1.5]
steps: 2
variable: ['Vbg']
start: [-1.5]
end: [-1.4]
steps: 2
variable: ['Vtg']
start: [-1.5]
end: [-1.4]
steps: 2
variable: ['Vbg']
start: [-1.4]
end: [-1.3]
steps: 2
variable: ['Vtg']
start: [-1.4]
end: [-1.3]
steps: 2
variable: ['Vbg']
start: [-1.3]
end: [-1.2]
steps: 1
variable: ['Vtg']
start: [-1.3]
end: [-1.2]
steps: 1
variable: ['Vbg']
start: [-1.3]
e

## general PL/REF

In [ ]:
center = '0'
exp_time = '300000'

lf6.change_spectra_center(center)
lf6.change_expose_time(exp_time)
lf6.change_roi_FullSensor()

In [ ]:
#x_goto(self, x_name, target, delta, delay)
vtgs = [-10 , 10  ]
vbgs = [10  , -10 ]
centers = ['0','880']
exp_time = '300000'
# vtgs     = [0]
# vbgs     = [0]
# centers  = ['880']
# exp_time = '1000'  # ms

for vtg, vbg in zip(vtgs, vbgs):
    for center in centers:
        lf6.change_spectra_center(center)
        lf6.change_expose_time(exp_time)
        lf6.change_roi_FullSensor()

        iv.x_goto('Vtg', vtg, 0.2, 0.05)
        iv.x_goto('Vbg', vbg, 0.2, 0.05)
        iv.report_status()

        # Power only in filename (not a CSV column)
        meas_power = c_double()
        tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
        power = round(meas_power.value * 1e6 * POWER_EFF, 3)  # µW

        base_name = f"$YZD222$~$PL730nm{power}uW$~$6Kp2n8$~${center}nm{int(exp_time)/1000}s$~$TG1={vtg}TG2={vbg}$"
        file_name = unique_filename(base_name + ".csv")

        # --- Header: Vbg, Vtg, Y, then 1024 energies (eV) ---
        wls = np.asarray(lf6.get_wavelength_calibration(), dtype=np.float64)
        if wls.size != 1024:
            raise ValueError(f"wavelength calibration length={wls.size}, expected 1024")
        energies = 1240.0 / wls  # eV

        with open(file_name, "w") as f:
            header = np.concatenate((
                np.array(['Vbg', 'Vtg', 'Y'], ndmin=1, dtype='U'),
                wls.astype('U')
            )).reshape(1, -1)
            np.savetxt(f, header, fmt='%s', delimiter=',')

        # --- Acquire one spectra and write 256 data rows ---
        spectra = lf6.acquire()  # flat 1-D, 1024*256 numbers
        a = np.asarray(spectra, dtype=np.float64).ravel()
        if a.size != 256 * 1024:
            raise ValueError(f"acquire() returned {a.size} values, expected 262144")

        # If image appears transposed later, swap with a.reshape(1024, 256).T
        M = a.reshape(256, 1024)

        # Y axis 1..256
        y_col = np.arange(1, M.shape[0] + 1, dtype=np.float64).reshape(-1, 1)

        # Two meta columns repeated per row + Y column
        meta_cols = np.column_stack((
            np.full((M.shape[0], 1), vbg, dtype=np.float64),
            np.full((M.shape[0], 1), vtg, dtype=np.float64),
            y_col
        ))  # shape (256, 3)

        # Final output: (256 x (3 + 1024)) = (256 x 1027)
        out = np.hstack((meta_cols, M))

        with open(file_name, "a") as f:
            np.savetxt(f, out, fmt="%.6e", delimiter=",")

        print("Saved:", file_name)

In [226]:
probe_stage.move_to(0)
time.sleep(2)
wavelength = c_double(730)
tlPM.setWavelength(wavelength,TLPM_DEFAULT_CHANNEL)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = meas_power.value*1e6*POWER_EFF
pump_power = power-0
print(power)
print(pump_power)

0.052518049600000005
0.052518049600000005


In [ ]:
# 500 probe off 2000 probe on

stage.move_to(3600)
time.sleep(2)
wavelength = c_double(730)
tlPM.setWavelength(wavelength,TLPM_DEFAULT_CHANNEL)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = meas_power.value*1e6*POWER_EFF
pump_power = power-0.041
print(power)
print(pump_power)

In [ ]:
lf6.change_expose_time(15000)


In [ ]:
#x_goto(self, x_name, target, delta, delay)
iv.x_goto('Vtg', 0, 0.2, 0.05)
iv.x_goto('Vbg', 0, 0.2, 0.05)
iv.x_goto('Vtg2', 0, 0.2,0.05)
iv.report_status()


In [ ]:
# probe_stage.move_to(1000)
# time.sleep(2)
wavelength = c_double(730)
tlPM.setWavelength(wavelength,TLPM_DEFAULT_CHANNEL)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = meas_power.value*1e6*POWER_EFF
power_dig = round(power, 3)
print(power_dig)

In [ ]:
# stage_poss = [3600,  3400,      3350,   3300,   3200,   3100,   3000,   2900,   2800,   2700, 2600,     2450,   2200,   2000,   1500,   1000,   200]
# exp_times = ['10000', '10000','15000', '15000','15000', '15000','20000', '20000','20000','25000', '25000','25000','25000','30000','30000','30000','60000']

stage_poss = [1000]
exp_times = ['2000']
# vtg2 = -10

for stage_pos, exp_time in zip(stage_poss, exp_times):
    probe_stage.move_to(stage_pos)
    time.sleep(2)
    # wavelength = c_double(730)
    # tlPM.setWavelength(wavelength,TLPM_DEFAULT_CHANNEL)
    meas_power =  c_double()
    tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
    power = meas_power.value*1e6*POWER_EFF
    rounded_power = round(power, 4)
    print(power)
    lf6.change_expose_time(exp_time)
    # iv.x_goto('Vtg2', vtg2, 0.2,0.05)

    sample_name = f'$YZ336$~$p3n2$~$6KPL730nm{rounded_power}uw{exp_time}ms$~$880nmc$'
    exps = spectral_experiments.SpectralExperimentsCollections()
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start= -3, vbg_stop= 3,
    #                                 vtg_start= -3, vtg_stop=3, frames= 121, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG1-TG2=0$', iv, lf6, vbg_start= -14, vbg_stop= 14,
    #                                 vtg_start= -14, vtg_stop=14, frames= 281, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, f'$TG1={vtg2}_TG2={vtg2}_BGonly$', iv, lf6, vbg_start= -10, vbg_stop= 10,
    #                                 vtg_start= vtg2, vtg_stop=vtg2, frames= 201, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG1+TG2=0$', iv, lf6, vbg_start= -14, vbg_stop= 14,
    #                                 vtg_start= 14, vtg_stop=-14, frames= 281, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start= -10, vbg_stop= 6,
    #                                    vtg_start=0, vtg_stop=0, frames= 161, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG1only$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                                    vtg_start= -14, vtg_stop= 14, frames= 281, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG2only$', iv, lf6, vbg_start=-14, vbg_stop=14,
    #                                    vtg_start= 0, vtg_stop= 0, frames= 281, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$0.75TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=4.5,
                                       vtg_start= -6, vtg_stop= 6, frames= 121, repeat=1, plot=False)

    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0rev$', iv, lf6, vbg_start=4, vbg_stop=-4,
    #                                 vtg_start=4, vtg_stop=-4, frames=161, repeat=1, plot=False)
    exps.execute(repeat=1)


In [ ]:
lf6.change_expose_time('10')


In [ ]:

Doping=5
vbg = 0.5*Doping
vtg = 0.5*Doping
iv.x_goto('Vtg', vtg, 0.2, 0.05)
iv.x_goto('Vbg', vbg, 0.2, 0.05)
iv.report_status()

In [358]:
vbg = 8
vtg = 8
iv.x_goto('Vtg', vtg, 0.2, 0.05)
iv.x_goto('Vbg', vbg, 0.2, 0.05)

variable: ['Vtg']
start: [0.]
end: [8]
steps: 41
variable: ['Vbg']
start: [0.]
end: [8]
steps: 41


In [359]:
move_stage_checked(probe_stage, 700)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = meas_power.value*1e6*POWER_EFF
power_dig = round(power, 3)
print(power_dig)


0.294


In [319]:
exp_times = ["500"]

stage_poss=[1500]

for stage_pos, exp_time in zip(stage_poss, exp_times):
    if not move_stage_checked(probe_stage, stage_pos): 
        print(f"CRITICAL: Could not reach {round(stage_pos,1)} after 3 attempts. Skipping.")
        continue
    time.sleep(1)
    lf6.change_expose_time(exp_time)

    meas_power = c_double()    
    tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
    power = meas_power.value * 1e6 * POWER_EFF
    power = round(power,3)


    sample_name = f'$N72$~$6KPL730nm{power}uW{exp_time}ms$~$pa(4.7 4.7)$~$C880nm$'
    exps = spectral_experiments.SpectralExperimentsCollections()


    exps.add_dual_gate_spectra_sweep(sample_name, '$bg1only-TG$', iv, lf6, vbg_start= 0, vbg_stop= 0,
                                       vtg_start= -10, vtg_stop= 10, frames= 101, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$bg2only-BG$', iv, lf6, vbg_start= -10, vbg_stop= 10,
                                       vtg_start= 0, vtg_stop= 0, frames= 101, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$bg1=bg2_TG-BG=0$', iv, lf6, vbg_start= -10, vbg_stop= 13,
    #                               vtg_start= -10, vtg_stop= 13, frames= 231, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$bg1=bg2=8_TG-BG=0$', iv, lf6, vbg_start= 8, vbg_stop= 8.2,
    #                                    vtg_start= 8, vtg_stop= 8.2, frames= 3, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$bg1+bg2=0$', iv, lf6, vbg_start= -10, vbg_stop= 10,
    #                                    vtg_start= 10, vtg_stop= -10, frames= 101, repeat=1, plot=False)

    exps.execute(repeat=1)


Exposetime(ms): 500.0
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0]
steps: 2
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
variable: ['Vtg']
start: [0.]
end: [-10]
steps: 101
variable: ['Vbg', 'Vtg']
start: [  0 -10]
end: [ 0 10]
steps: 101
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
variable: ['Vtg']
start: [10.]
end: [0]
steps: 101
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0]
steps: 2
variable: ['Vbg']
start: [0.]
end: [-10]
steps: 101
variable: ['Vtg']
start: [0.]
end: [0]
steps: 2
variable: ['Vbg', 'Vtg']
start: [-10   0]
end: [10  0]
steps: 101
variable: ['Vbg']
start: [10.]
end: [0]
steps: 101
variable: ['Vtg']
start: [0.]
end: [0]
steps: 2


In [360]:
# exp_times = ["60000"]
exp_times = ["60000", "60000", "30000","15000"]
# stage_poss=[1500]
stage_poss=[700, 1500,  2900, 3480]
for stage_pos, exp_time in zip(stage_poss, exp_times):
    if not move_stage_checked(probe_stage, stage_pos): 
        print(f"CRITICAL: Could not reach {round(stage_pos,1)} after 3 attempts. Skipping.")
        continue
    time.sleep(1)
    lf6.change_expose_time(exp_time)

    meas_power = c_double()    
    tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
    power = meas_power.value * 1e6 * POWER_EFF
    power = round(power,3)


    sample_name = f'$N72$~$6KPL730nm{power}uW{exp_time}ms$~$pbx6.1y5.8$~$C860nm$'
    exps = spectral_experiments.SpectralExperimentsCollections()


    # exps.add_dual_gate_spectra_sweep(sample_name, '$bg1only-TG-BG=0$', iv, lf6, vbg_start= 0, vbg_stop= 0,
    #                                    vtg_start= 8, vtg_stop= 8.2, frames= 3, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$bg2only-TG-BG=0$', iv, lf6, vbg_start= 8, vbg_stop= 8.2,
    #                                    vtg_start= 0, vtg_stop= 0, frames= 3, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$bg1=bg2_TG-BG=0$', iv, lf6, vbg_start= -3, vbg_stop= 13,
                                  vtg_start= -3, vtg_stop= 13, frames= 161, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$bg1=bg2=8_TG-BG=0$', iv, lf6, vbg_start= 8, vbg_stop= 8.2,
    #                                    vtg_start= 8, vtg_stop= 8.2, frames= 3, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$bg1+bg2=0$', iv, lf6, vbg_start= -10, vbg_stop= 10,
    #                                    vtg_start= 10, vtg_stop= -10, frames= 101, repeat=1, plot=False)

    exps.execute(repeat=1)


Exposetime(ms): 60000.0
variable: ['Vbg']
start: [8.]
end: [0]
steps: 81
variable: ['Vtg']
start: [8.]
end: [0]
steps: 81
variable: ['Vbg']
start: [0.]
end: [-3]
steps: 31
variable: ['Vtg']
start: [0.]
end: [-3]
steps: 31
variable: ['Vbg', 'Vtg']
start: [-3 -3]
end: [13 13]
steps: 161
variable: ['Vbg']
start: [13.]
end: [0]
steps: 131
variable: ['Vtg']
start: [13.]
end: [0]
steps: 131
  -> Position Mismatch! Target: 1500, Actual: 0.00 (Diff: -1500.00)
  -> Retry move to 1500 (Attempt 2)...
Exposetime(ms): 60000.0
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0]
steps: 2
variable: ['Vbg']
start: [0.]
end: [-3]
steps: 31
variable: ['Vtg']
start: [0.]
end: [-3]
steps: 31
variable: ['Vbg', 'Vtg']
start: [-3 -3]
end: [13 13]
steps: 161
variable: ['Vbg']
start: [13.]
end: [0]
steps: 131
variable: ['Vtg']
start: [13.]
end: [0]
steps: 131
Exposetime(ms): 30000.0
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0]
steps: 2

In [ ]:
lf6.change_expose_time(2000)

In [ ]:
center_wavs = [720, 630]
# center_wavs = [720]
for centerwav in center_wavs:
    lf6.change_spectra_center(centerwav)
    sample_name = f'$7KYZ297$~$REF{centerwav}nm$~$p7$~$60msx20x1$~'
    # sample_name = '$7KYZ297$~$REF70msx20x1p6$~$720nm$'

    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start= -7, vbg_stop= 7,
                                       vtg_start= -7, vtg_stop= 7, frames= 141, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start= -7, vbg_stop= 7,
    #                                vtg_start=0, vtg_stop=0, frames= 71, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
#                                    vtg_start= -10, vtg_stop= 10, frames= 101, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start= -10, vbg_stop= 10,
#                                   vtg_start= -10, vtg_stop= 10, frames= 201, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$', iv, lf6, vbg_start= -9, vbg_stop= 9,
#                                    vtg_start= 9, vtg_stop= -9, frames= 181, repeat=1, plot=False)
    exps.execute(repeat=1)


### Reflection

In [ ]:
lf6.change_spectra_center(710)
sample_name = '$YZD77$~$REF$~$7KpEl$~$710nmC300g$~$2s$~'
exps = spectral_experiments.SpectralExperimentsCollections()
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-4.5$', iv, lf6, vbg_start=-4.5, vbg_stop=0.0,
                                         vtg_start=0.0, vtg_stop=-4.5, frames=91, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.5$', iv, lf6, vbg_start=-4.5, vbg_stop=1.0,
                                         vtg_start=1.0, vtg_stop=-4.5, frames=111, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2.5$', iv, lf6, vbg_start=-4.5, vbg_stop=2.0,
                                         vtg_start=2.0, vtg_stop=-4.5, frames=131, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-1.5$', iv, lf6, vbg_start=-4.5, vbg_stop=3.0,
                                         vtg_start=3.0, vtg_stop=-4.5, frames=151, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-0.5$', iv, lf6, vbg_start=-4.5, vbg_stop=4.0,
                                         vtg_start=4.0, vtg_stop=-4.5, frames=171, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.5$', iv, lf6, vbg_start=4.5, vbg_stop=-4.0,
                                         vtg_start=-4.0, vtg_stop=4.5, frames=171, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$', iv, lf6, vbg_start=4.5, vbg_stop=-4.5,
                                         vtg_start=-4.5, vtg_stop=4.5, frames=181, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.5$', iv, lf6, vbg_start=4.5, vbg_stop=-3.0,
                                         vtg_start=-3.0, vtg_stop=4.5, frames=151, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.5$', iv, lf6, vbg_start=4.5, vbg_stop=-2.0,
                                         vtg_start=-2.0, vtg_stop=4.5, frames=131, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=3.5$', iv, lf6, vbg_start=4.5, vbg_stop=-1.0,
                                         vtg_start=-1.0, vtg_stop=4.5, frames=111, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.5$', iv, lf6, vbg_start=4.5, vbg_stop=0.0,
                                         vtg_start=0.0, vtg_stop=4.5, frames=91, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.execute(repeat=1)

lf6.change_spectra_center(620)
sample_name = '$YZD77$~$REF$~$7KpEl$~$620nmC300g$~$2s$~'
exps = spectral_experiments.SpectralExperimentsCollections()
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-4.5$', iv, lf6, vbg_start=-4.5, vbg_stop=0.0,
                                         vtg_start=0.0, vtg_stop=-4.5, frames=91, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.5$', iv, lf6, vbg_start=-4.5, vbg_stop=1.0,
                                         vtg_start=1.0, vtg_stop=-4.5, frames=111, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2.5$', iv, lf6, vbg_start=-4.5, vbg_stop=2.0,
                                         vtg_start=2.0, vtg_stop=-4.5, frames=131, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-1.5$', iv, lf6, vbg_start=-4.5, vbg_stop=3.0,
                                         vtg_start=3.0, vtg_stop=-4.5, frames=151, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-0.5$', iv, lf6, vbg_start=-4.5, vbg_stop=4.0,
                                         vtg_start=4.0, vtg_stop=-4.5, frames=171, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.5$', iv, lf6, vbg_start=4.5, vbg_stop=-4.0,
                                         vtg_start=-4.0, vtg_stop=4.5, frames=171, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$', iv, lf6, vbg_start=4.5, vbg_stop=-4.5,
                                         vtg_start=-4.5, vtg_stop=4.5, frames=181, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.5$', iv, lf6, vbg_start=4.5, vbg_stop=-3.0,
                                         vtg_start=-3.0, vtg_stop=4.5, frames=151, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.5$', iv, lf6, vbg_start=4.5, vbg_stop=-2.0,
                                         vtg_start=-2.0, vtg_stop=4.5, frames=131, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=3.5$', iv, lf6, vbg_start=4.5, vbg_stop=-1.0,
                                         vtg_start=-1.0, vtg_stop=4.5, frames=111, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.5$', iv, lf6, vbg_start=4.5, vbg_stop=0.0,
                                         vtg_start=0.0, vtg_stop=4.5, frames=91, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.execute(repeat=1)


lf6.change_spectra_center(710)
sample_name = '$YZD77$~$REF$~$7KpEl$~$710nmC300g$~$2s$~'
exps = spectral_experiments.SpectralExperimentsCollections()
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-4.5$', iv, lf6, vbg_start=0.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=0.0, frames=91, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-3.5$', iv, lf6, vbg_start=-1.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=1.0, frames=111, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-2.5$', iv, lf6, vbg_start=-2.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=2.0, frames=131, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-1.5$', iv, lf6, vbg_start=-3.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=3.0, frames=151, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-0.5$', iv, lf6, vbg_start=-4.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=4.0, frames=171, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0.5$', iv, lf6, vbg_start=-4.5, vbg_stop=4.0,
                                         vtg_start=-4.0, vtg_stop=4.5, frames=171, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=4.5, frames=181, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1.5$', iv, lf6, vbg_start=-4.5, vbg_stop=3.0,
                                         vtg_start=-3.0, vtg_stop=4.5, frames=151, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=2.5$', iv, lf6, vbg_start=-4.5, vbg_stop=2.0,
                                         vtg_start=-2.0, vtg_stop=4.5, frames=131, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=3.5$', iv, lf6, vbg_start=-4.5, vbg_stop=1.0,
                                         vtg_start=-1.0, vtg_stop=4.5, frames=111, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=4.5$', iv, lf6, vbg_start=-4.5, vbg_stop=0.0,
                                         vtg_start=0.0, vtg_stop=4.5, frames=91, repeat=4, plot=False)
exps.execute(repeat=1)

lf6.change_spectra_center(620)
sample_name = '$YZD77$~$REF$~$7KpEl$~$620nmC300g$~$2s$~'
exps = spectral_experiments.SpectralExperimentsCollections()
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-4.5$', iv, lf6, vbg_start=0.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=0.0, frames=91, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-3.5$', iv, lf6, vbg_start=-1.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=1.0, frames=111, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-2.5$', iv, lf6, vbg_start=-2.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=2.0, frames=131, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-1.5$', iv, lf6, vbg_start=-3.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=3.0, frames=151, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-0.5$', iv, lf6, vbg_start=-4.0, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=4.0, frames=171, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0.5$', iv, lf6, vbg_start=-4.5, vbg_stop=4.0,
                                         vtg_start=-4.0, vtg_stop=4.5, frames=171, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=4.5,
                                         vtg_start=-4.5, vtg_stop=4.5, frames=181, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1.5$', iv, lf6, vbg_start=-4.5, vbg_stop=3.0,
                                         vtg_start=-3.0, vtg_stop=4.5, frames=151, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=2.5$', iv, lf6, vbg_start=-4.5, vbg_stop=2.0,
                                         vtg_start=-2.0, vtg_stop=4.5, frames=131, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=3.5$', iv, lf6, vbg_start=-4.5, vbg_stop=1.0,
                                         vtg_start=-1.0, vtg_stop=4.5, frames=111, repeat=4, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=4.5$', iv, lf6, vbg_start=-4.5, vbg_stop=0.0,
                                         vtg_start=0.0, vtg_stop=4.5, frames=91, repeat=4, plot=False)
exps.execute(repeat=1)


## VP Check 

In [ ]:
ep300.set_position(2,69)
#     max113 338uW 79.8 50uW 68.5 1.5uW
#max113 275uw, 81 50uw,69.4 1.5uw
#max69 210uw 92   100uw 99.5  50uw 107.7 10uw 109.5 5uw 110.9 2.5uw
#max69 230uw 92   100uw 99.5  50uw 107.7 10uw 109.5 5uw 110.9 2.5uw 
#max69 308uw 96.7 100uw 102.1 50uw 108.8 10uw 110.4 5uw 111.5 2.5uw 


In [ ]:
ep300.set_position(1,18)
# 48 93
#56KK` 101KK 
#91KK` 46KK 
#18KK 63KKp

In [ ]:
valleys = [18,63]
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$noquarter210uW$~$1s$~'
for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=5,
                                   vtg_start=5, vtg_stop=-5, frames=201, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=5,
#                                    vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=2.0$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=3,
#                                    vtg_start=-3, vtg_stop=5, frames=161, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=3.0$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=2,
#                                    vtg_start=-2, vtg_stop=5, frames=141, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=4.0$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=1,
#                                    vtg_start=-1, vtg_stop=5, frames=121, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=5.0$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=0,
#                                    vtg_start=0, vtg_stop=5, frames=101, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6$~${}degree$'.format(valley), iv, lf6, vbg_start=1, vbg_stop=5,
#                                    vtg_start=5, vtg_stop=1, frames=81, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.6$~${}degree$'.format(valley), iv, lf6, vbg_start=-3.4, vbg_stop=5,
#                                    vtg_start=5, vtg_stop=-3.4, frames=169, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1.6$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=3.4,
#                                    vtg_start=-3.4, vtg_stop=5, frames=169, repeat=1, plot=False)
            exps.execute(repeat=1)

## PL Efield and Doping with valley


In [ ]:
import numpy as np

number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$50uW$~$5s$~'
vbg = -5
vtg =5
valleys = [46,91]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
            
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)

In [ ]:
import numpy as np
ep300.set_position(2,99.5)
lf6.change_expose_time(2000)
number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$50uW$~$2s$~'
vbg = -5
vtg =5
valleys = [18,63]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
for valley in valleys:
    ep300.set_position(1,valley)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6$~${}degree$'.format(valley), iv, lf6, vbg_start=1, vbg_stop=5,
                                        vtg_start=5, vtg_stop=1, frames=81, repeat=1, plot=False) 
    exps.execute(repeat=1)
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
for valley in valleys:
    ep300.set_position(1,valley)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1.6$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=3.4,
                                        vtg_start=-3.4, vtg_stop=5, frames=169, repeat=1, plot=False)
    exps.execute(repeat=1)
# 
ep300.set_position(2,110.9)
lf6.change_expose_time(6000)
number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$2.5uW$~$6s$~'
vbg = -5
vtg =5
valleys = [18,63]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
for valley in valleys:
    ep300.set_position(1,valley)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6$~${}degree$'.format(valley), iv, lf6, vbg_start=1, vbg_stop=5,
                                        vtg_start=5, vtg_stop=1, frames=81, repeat=1, plot=False) 
    exps.execute(repeat=1)
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
for valley in valleys:
    ep300.set_position(1,valley)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1.6$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=3.4,
                                        vtg_start=-3.4, vtg_stop=5, frames=169, repeat=1, plot=False)
    exps.execute(repeat=1)


In [ ]:
#max69 210uw 92 100uw 99.5 50uw 107.7 10uw 109.5 5uw 110.9 2.5uw
import numpy as np
ep300.set_position(2,69)
lf6.change_expose_time(1000)
number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$210uW$~$1s$~'
vbg = -5
vtg =5
valleys = [18,63]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
            
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
#
ep300.set_position(2,110.9)
lf6.change_expose_time(5000)
number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$2.5uW$~$5s$~'
vbg = -5
vtg =5
valleys = [18,63]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
            
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
#
ep300.set_position(2,107.7)
lf6.change_expose_time(5000)
number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$10uW$~$5s$~'
vbg = -5
vtg =5
valleys = [18,63]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
            
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
#
ep300.set_position(2,99.5)
lf6.change_expose_time(2000)
number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$50uW$~$2s$~'
vbg = -5
vtg =5
valleys = [18,63]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
            
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
#
ep300.set_position(2,92)
lf6.change_expose_time(1000)
number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$100uW$~$1s$~'
vbg = -5
vtg =5
valleys = [18,63]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
            
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)

## PL Efield and Doping without valley


In [ ]:
import numpy as np
number = 9
sample_name = '$YZ165$~$REF$~$7KpP$~$710nmC300g$~$2s$~'
vbg = -4
vtg =4
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG+BG={}$'.format(dv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                        vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
        print(vbgs, vbge, vtgs, vtge, exptitle, frs)
        exps.execute(repeat=3)
            
vgs = -5
vge =5
number1 = 11
for dv in np.linspace(vgs,vge,number1):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG-BG={}$'.format(dv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                        vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
        print(vbgs, vbge, vtgs, vtge, exptitle, frs)
        exps.execute(repeat=3)

In [ ]:
import numpy as np
number = 11
sample_name = '$DX156$~$REF$~$7Kp4up$~'
vbg = -5
vtg =5
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        lf6.change_spectra_center(710)
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$710nmC300g$~$2s$~$TG+BG={}$'.format(dv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                        vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
        print(vbgs, vbge, vtgs, vtge, exptitle, frs)
        exps.execute(repeat=4)
        lf6.change_spectra_center(620)
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$620nmC300g$~$2s$~$TG+BG={}$'.format(dv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                        vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
        print(vbgs, vbge, vtgs, vtge, exptitle, frs)
        exps.execute(repeat=4)
            
vgs = -5
vge =5
number1 = 11
for dv in np.linspace(vgs,vge,number1):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        lf6.change_spectra_center(710)
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$710nmC300g$~$2s$~$TG-BG={}$'.format(dv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                        vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
        print(vbgs, vbge, vtgs, vtge, exptitle, frs)
        exps.execute(repeat=4)
        lf6.change_spectra_center(620)
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$620nmC300g$~$2s$~$TG-BG={}$'.format(dv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                        vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
        print(vbgs, vbge, vtgs, vtge, exptitle, frs)
        exps.execute(repeat=4)
        

In [ ]:
import numpy as np
number = 11
sample_name = '$DX156_20231224$~$PL$~$7Kp4up$~$730nm5uW$~$2s$~'
vbg = -5
vtg =5
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG+BG={}$'.format(dv)
#         exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
#                                         vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
        print(vbgs, vbge, vtgs, vtge, exptitle, frs)
        #         exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
#                                         vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
#         exps.execute(repeat=1)
            
# vgs = -5
# vge =5
# number1 = 5
# for dv in np.linspace(vgs,vge,number1):
#     if dv <= 0:
#         vbge = vge
#         vtgs = vgs
#         vbgs = vtgs - dv
#         vtge = vbge + dv
#     else:
#         vtge = vge
#         vbgs = vgs
#         vbge = vtge - dv
#         vtgs = vbgs + dv
#     frs = int((vtge - vtgs)/0.05 +1)
#     if frs == 1:
#         pass;
#     else:
#         exps = spectral_experiments.SpectralExperimentsCollections()
#         exptitle = '$TG-BG={}$'.format(dv)
# #         exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
# #                                         vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
#         print(vbgs, vbge, vtgs, vtge, exptitle, frs)
# #         exps.execute(repeat=3)

In [ ]:
bgvalue = -5
tgvalue = -5

bgend = np.around(np.linspace(0, 5, 51),decimals=2)
tgstart = bgend
bgstart = np.full_like(bgend,bgvalue)
tgend = np.full_like(bgend,tgvalue)
frs = np.linspace(101,201,51)
dopingvalue = np.around(np.linspace(-5, 0, 51),decimals=2)
sample_name = '$DX156_20231224-MEGA$~$PL$~$7Kp4up$~$730nm5uW$~$3s$~'
for bgsv,bgev,tgsv,tgev,frsv,dopingv in zip(bgstart,bgend,tgstart,tgend,frs,dopingvalue):
    if frsv < 0:
        pass
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG+BG={}$'.format(dopingv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=bgsv, vbg_stop=bgev,
                                        vtg_start=tgsv, vtg_stop=tgev, frames=int(frsv), repeat=1, plot=False)
            
        exps.execute(repeat=1)
        print(bgsv,bgev,tgsv,tgev,exptitle,int(frsv))
print('-------------------')
bgvalue = 5
tgvalue = 5
bgend = np.around(np.linspace(0, -5, 51),decimals=2)
tgstart = bgend
bgstart = np.full_like(bgend,bgvalue)
tgend = np.full_like(bgend,tgvalue)
frs = np.linspace(101,201,51)
dopingvalue = np.around(np.linspace(5, 0, 51),decimals=2)
for bgsv,bgev,tgsv,tgev,frsv,dopingv in zip(bgstart,bgend,tgstart,tgend,frs,dopingvalue):
    if frsv < 0:
        pass
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG+BG={}$'.format(dopingv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=bgsv, vbg_stop=bgev,
                                        vtg_start=tgsv, vtg_stop=tgev, frames=int(frsv), repeat=1, plot=False)
            
        exps.execute(repeat=1)
        print(bgsv,bgev,tgsv,tgev,exptitle,int(frsv))

## Step and Glue

In [ ]:
centers_leftedge = np.array([810,837,864,891,918,945,972])#wav 815nm to 995nm

centers = centers_leftedge+15
sample_name = '$YZD154$~$PL$~$10Kp51+1+1$~$730nm50uW1200g$~$4s$~'
for center in centers:
            lf6.change_spectra_center(int(center))
            exps = spectral_experiments.SpectralExperimentsCollections()
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$~${}center$'.format(center), iv, lf6, vbg_start=-5, vbg_stop=5,
#                                    vtg_start=5, vtg_stop=-5, frames=201, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$~${}center$'.format(center), iv, lf6, vbg_start=-5, vbg_stop=5,
#                                    vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=False)
            exps.execute(repeat=1)
#             print(center)

## Power dep


In [ ]:
##load linear stage
import pylablib as pll
from pylablib.devices import Thorlabs
stage = Thorlabs.ElliptecMotor("COM5")

In [ ]:
POWER_EFF = 0.4

In [ ]:
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = meas_power.value*1e6*POWER_EFF
print(power)

In [ ]:
stage.close()

In [268]:
probe_stage.move_to(1500)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),TLPM_DEFAULT_CHANNEL)
power = meas_power.value*1e6*POWER_EFF
print(power)

1.0247837039999999


In [312]:
# --- 1. Setup Parameters ---
stage_poss = np.linspace(0, 3600, 37)
vbg = 0
vtg = 0
iv.x_goto('Vtg', vtg, 0, 0.05)
iv.x_goto('Vbg', vbg, 0, 0.05)
iv.report_status()

center = '860'
exp_time = '2000'
base_name = f'$N72$~$PL730nm$~$6Kpb(5.0 5.1)$~$Power{center}nm{int(exp_time)/1000}s$~$TG={vtg}BG={vbg}$'

# Get unique filename
final_filename = unique_filename(base_name + '.csv')
temp_filename = base_name + '_temp.csv'

lf6.change_spectra_center(center)
lf6.change_expose_time(exp_time)

# --- 2. Create Header & Initialize File ---
# Get wavelengths and convert to energy
wls = lf6.get_wavelength_calibration()
energies = 1240 / wls

# Create column names: Fixed metadata + dynamic energy columns
columns = ['Vbg', 'Vtg', 'power'] + list(energies.astype(str))

# Create the file and write the header immediately
# We create an empty DataFrame just to write the header
pd.DataFrame(columns=columns).to_csv(temp_filename, index=False)

print(f"Scanning... Data saving safely to {temp_filename}")

# --- 3. Loop and Save (Safe Method) ---
for stage_pos in stage_poss:
    probe_stage.move_to(stage_pos)
    time.sleep(1)
    
    meas_power = c_double()    
    tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
    power = meas_power.value * 1e6 * POWER_EFF
    spectra = lf6.acquire()
    
    # Create a single-row DataFrame
    # Note: We combine [metadata] + [spectra data]
    row_data = np.concatenate(([vbg, vtg, power], spectra))
    df_row = pd.DataFrame([row_data], columns=columns)
    
    # Append to CSV immediately (mode='a', header=False)
    df_row.to_csv(temp_filename, mode='a', header=False, index=False)
    
    print(f"Saved: Pos {stage_pos}, Power {power:.2f}")

# --- 4. Load, Sort, and Finalize ---
print("Sorting data...")

try:
    # 1. Read the full dataset
    df = pd.read_csv(temp_filename)
    
    # 2. Sort by 'power' (Ascending: True, Descending: False)
    df_sorted = df.sort_values(by='power', ascending=True)
    
    # 3. Save to final file
    df_sorted.to_csv(final_filename, index=False)
    
    print(f"Done! Sorted data saved to {final_filename}")
    
    # Clean up temp file
    os.remove(temp_filename)

except Exception as e:
    print(f"Error sorting: {e}")
    print(f"Raw data is safe in {temp_filename}")

variable: ['Vtg']
start: [0.]
end: [0]
steps: 2
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
x_channel Vtg: value: 0.0
x_channel Vbg: value: 0.0
x_channel Vbias: value: 0.0
y_channel measured_Vtg: value: 0.0
y_channel Vtg_leakage: value: -1.965039e-11
y_channel measured_Vbg: value: 0.0
y_channel Vbg_leakage: value: -8.766919e-12
y_channel measured_Vbias: value: 0.0
y_channel Vbias_leakage: value: -1.812556e-11
Center Wave Length: 860.0
Grating: [1um,600][0][0]
Exposetime(ms): 2000.0
Scanning... Data saving safely to $N72$~$PL730nm$~$6Kpb(5.0 5.1)$~$Power860nm2.0s$~$TG=0BG=0$_temp.csv
Saved: Pos 0.0, Power 0.11
Saved: Pos 100.0, Power 0.12
Saved: Pos 200.0, Power 0.14
Saved: Pos 300.0, Power 0.16
Saved: Pos 400.0, Power 0.19
Saved: Pos 500.0, Power 0.21
Saved: Pos 600.0, Power 0.24
Saved: Pos 700.0, Power 0.29
Saved: Pos 800.0, Power 0.34
Saved: Pos 900.0, Power 0.39
Saved: Pos 1000.0, Power 0.45
Saved: Pos 1100.0, Power 0.53
Saved: Pos 1200.0, Power 0.61
Saved: Pos 1300.0, Power 0.7

In [ ]:
print(np.linspace(-3,3,121))

In [ ]:
    # --- 1. Setup Parameters ---
stage_poss = np.linspace(0, 3600, 73)
vtgs = np.linspace(-3,3,121)
vbgs = np.linspace(-3,3,121)
for vtg,vbg in zip(vtgs,vbgs):
    # vbg = 0
    # vtg = 0
    iv.x_goto('Vtg', vtg, 0, 0.05)
    iv.x_goto('Vbg', vbg, 0, 0.05)
    iv.report_status()

    center = '860'
    exp_time = '5000'
    base_name = f'$YZD112$~$PL730nm$~$6KpD$~$Power{center}nm{int(exp_time)/1000}s$~$TG={vtg}BG={vbg}$'

    # Get unique filename
    final_filename = unique_filename(base_name + '.csv')
    temp_filename = base_name + '_temp.csv'

    lf6.change_spectra_center(center)
    lf6.change_expose_time(exp_time)

    # --- 2. Create Header & Initialize File ---
    # Get wavelengths and convert to energy
    wls = lf6.get_wavelength_calibration()
    energies = 1240 / wls

    # Create column names: Fixed metadata + dynamic energy columns
    columns = ['Vbg', 'Vtg', 'power'] + list(energies.astype(str))

    # Create the file and write the header immediately
    # We create an empty DataFrame just to write the header
    pd.DataFrame(columns=columns).to_csv(temp_filename, index=False)

    print(f"Scanning... Data saving safely to {temp_filename}")

    # --- 3. Loop and Save (Safe Method) ---
    for stage_pos in stage_poss:
        probe_stage.move_to(stage_pos)
        time.sleep(1)
        
        meas_power = c_double()    
        tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
        power = meas_power.value * 1e6 * POWER_EFF
        spectra = lf6.acquire()
        
        # Create a single-row DataFrame
        # Note: We combine [metadata] + [spectra data]
        row_data = np.concatenate(([vbg, vtg, power], spectra))
        df_row = pd.DataFrame([row_data], columns=columns)
        
        # Append to CSV immediately (mode='a', header=False)
        df_row.to_csv(temp_filename, mode='a', header=False, index=False)
        
        print(f"Saved: Pos {stage_pos}, Power {power:.2f}")

    # --- 4. Load, Sort, and Finalize ---
    print("Sorting data...")

    try:
        # 1. Read the full dataset
        df = pd.read_csv(temp_filename)
        
        # 2. Sort by 'power' (Ascending: True, Descending: False)
        df_sorted = df.sort_values(by='power', ascending=True)
        
        # 3. Save to final file
        df_sorted.to_csv(final_filename, index=False)
        
        print(f"Done! Sorted data saved to {final_filename}")
        
        # Clean up temp file
        os.remove(temp_filename)

    except Exception as e:
        print(f"Error sorting: {e}")
        print(f"Raw data is safe in {temp_filename}")

### EPS300 power dep

In [ ]:
ep300.set_position(2,115.05)

#max 78 344uW;88.5 300uw;98.15 200uw;106.7 100uw;111.8 50uW;115.2 25uw;118.15 10uw;119.6 5uw; 120.6 2.5uW;121.5 1uw;123 0.02UW 


In [ ]:
power_angles = [123, 115,  104,  99.5,   95,  89.7, 86.2, 84.4, 83.2, 81.7, 80.4, 80.1][::-1]
print(power_angles)
powers =       [325, 300, 200,  150,    100,  50,    25,   15,    10,   5,    2.5,1.5][::-1]
print(powers)

In [ ]:
import numpy as np
#max 56 358uW; 67.6 300uw;76.7 200uw;85 100uw;90 50uW;93.3 25uw;96.25 10uw;97.75 5uw;98.75 2.5uW;100.1 1uw;100.9 0.66UW 
power_angles = [100.1,90]
powers = [1,10,25]
diff_times = [120000, 30000,15000]
gating_times = [120000, 30000,15000]
center = 860
for power_angle, power, diff_time, gating_time in zip(power_angles,powers,diff_times,gating_times):
    print(power_angle, power, diff_time, gating_time)
#     start = 2
#     frame = 121
    if power > 40:
        start = -4
        frame = 201
    else:
        start = -6
        frame =241
    ep300.set_position(2,power_angle)
    lf6.change_spectra_center(center)
    lf6.change_expose_time(gating_time)
    lf6.change_roi_FullSensor()
    sample_name = '$YZD139_20231229$~$Diff860nmc$~$7Kp3$~${}uw$~${}ms$~'.format(power,gating_time)
    exps = spectral_experiments.SpectralExperimentsCollections()
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=start, vbg_stop=3,
#                                   vtg_start=start, vtg_stop=3, frames=frame, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=start, vbg_stop=6,
                                  vtg_start=0, vtg_stop=0, frames=frame, repeat=1, plot=False)
    exps.execute(repeat=1)
    
    if power > 40:
        pass
    else:
        lf6.change_spectra_center(0)
        lf6.change_expose_time(diff_time)
        lf6.change_roi_FullSensor()
        sample_name = '$YZD139_20231229$~$Diff0nm$~$7Kp3$~${}uw$~${}ms$~'.format(power,diff_time)
        exps = spectral_experiments.SpectralExperimentsCollections()
        exps.add_dual_gate_spectra_sweep(sample_name, '$check1$', iv, lf6, vbg_start=0, vbg_stop=0,
                                      vtg_start=0, vtg_stop=0, frames=1, repeat=1, plot=False)
    #     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=start, vbg_stop=3,
    #                                   vtg_start=start, vtg_stop=3, frames=frame, repeat=1, plot=False)
        exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=start, vbg_stop=6,
                                      vtg_start=0, vtg_stop=0, frames=frame, repeat=1, plot=False)
        exps.add_dual_gate_spectra_sweep(sample_name, '$check2$', iv, lf6, vbg_start=0, vbg_stop=0,
                                      vtg_start=0, vtg_stop=0, frames=1, repeat=1, plot=False)
        exps.execute(repeat=1)


In [ ]:
import numpy as np
#max 78 331uW;87 300uw;97.5 200uw;106.35 100uw;111.5 50uW;115.05 25uw;118.02 10uw;119.5 5uw; 120.55 2.5uW;121.48 1uw;123 0.03UW 
power_angles = [121.48,119.5,118.02,111.5]
powers = [1,5,10,50]
diff_times = [120000,60000,30000,10000]
gating_times = [120000,60000,30000,10000]
center = 860
for power_angle, power, diff_time, gating_time in zip(power_angles,powers,diff_times,gating_times):
    print(power_angle, power, diff_time, gating_time)
#     start = 2
#     frame = 121
    ep300.set_position(2,power_angle)
    lf6.change_spectra_center(center)
    lf6.change_expose_time(gating_time)
    lf6.change_roi_FullSensor()
    sample_name = '$YZD95_20240120$~$Diff860nmc$~$7KpH2$~${}uw$~${}ms$~'.format(power,gating_time)
    exps = spectral_experiments.SpectralExperimentsCollections()
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$', iv, lf6, vbg_start=-3.5, vbg_stop=3.5,
#                                    vtg_start=3.5, vtg_stop=-3.5, frames=141, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-3.5, vbg_stop=3,
                                  vtg_start=-3.5, vtg_stop=3, frames=261, repeat=1, plot=False)
    exps.execute(repeat=1)

In [ ]:
#max 78 331uW;87 300uw;97.5 200uw;106.35 100uw;111.5 50uW;115.05 25uw;118.02 10uw;119.5 5uw; 120.55 2.5uW;121.48 1uw;123 0.03UW 
x0 = np.concatenate((np.arange(0,0.5,0.05),np.arange(0.5,0.85,0.015),np.arange(0.85,1,0.01)))
power_angles = x0*45+78
x = x0*np.pi/4
powers = np.cos(2*x)*np.cos(2*x)*331
plt.scatter(power_angles,powers)
for p,p_a in zip(powers, power_angles):
    print(p,' ',p_a,'\n')

In [ ]:
# setup diffusion time:
diff_expose_times = np.zeros_like(x0)
for i,power in enumerate(powers):
    if power < 2:
        diff_expose_times[i] = 90000
    elif  power >= 2 and power < 10:
        diff_expose_times[i] = 60000
    elif  power >= 10 and power < 50:
        diff_expose_times[i] = 40000   
    elif  power>= 50:
        diff_expose_times[i] = 15000
plt.scatter(powers,diff_expose_times)

In [ ]:
# setup diffusion time:
spectra_expose_times = np.zeros_like(x0)
for i,power in enumerate(powers):
    if power < 5:
        spectra_expose_times[i] = 3000
    elif  power >= 5 and power < 50:
        spectra_expose_times[i] = 1000
    elif  power>= 50:
        spectra_expose_times[i] = 1000
plt.scatter(powers,spectra_expose_times)

In [ ]:
print(int(diff_expose_time))
lf6.change_expose_time(int(diff_expose_time))


In [ ]:
import numpy as np
file_name = '$YZD163$~$PL730nm850SP$~$7Kp6$'
# power_angles = []
# powers = []
bg_voltages = [-3.5, -3 ,-2.4,-2.2, -1.925,-1, -0.1, 0.5,0.925, 1.2,1.4, 1.6,1.85,2.2] #low energy
# bg_voltages = [2.85, 3.3, 3.75,4, 4.2, 4.45, 4.7]
# diff_expose_times = []
# spectra_expose_times = []
center = 850
for bg_voltage in bg_voltages:
#     Diffusion image
    lf6.change_roi_FullSensor()
    lf6.change_spectra_center(0)
    iv.x_goto('Vbg', bg_voltage, 0.02, 0.1)
    with open('{}~$PowerDiff$~${}$.csv'.format(file_name,bg_voltage), 'a') as f:
        wls = lf6.get_wavelength_calibration()
        cols = np.concatenate((np.array(['Vbg','power_angle','power','expose_time'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
        np.savetxt(f, cols, fmt='%s', delimiter=',')
        for power_angle, power, diff_expose_time in zip(power_angles, powers, diff_expose_times):
            ep300.set_position(2,power_angle)
            lf6.change_expose_time(int(diff_expose_time))
            spectra = lf6.acquire()
            data = np.concatenate((np.array([bg_voltage, power_angle, power, diff_expose_time], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
            np.savetxt(f, data, fmt='%.5e', delimiter=',')
#   Linespectra   
    lf6.change_roi_LineSensor()
    lf6.change_spectra_center(center)
    with open('{}~$PowerSpectra$~${}$.csv'.format(file_name,bg_voltage), 'a') as f:
        wls = lf6.get_wavelength_calibration()
        cols = np.concatenate((np.array(['Vbg','power_angle','power','expose_time'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
        np.savetxt(f, cols, fmt='%s', delimiter=',')
        for power_angle, power, spectra_expose_time in zip(power_angles, powers, spectra_expose_times):
            ep300.set_position(2,power_angle)
            lf6.change_expose_time(int(spectra_expose_time))
            spectra = lf6.acquire()
            data = np.concatenate((np.array([bg_voltage, power_angle, power, spectra_expose_time], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
            np.savetxt(f, data, fmt='%.5e', delimiter=',')
iv.x_goto('Vbg', 0, 0.02, 0.1)
       

In [ ]:
ep300.set_position(2, 114.0)

In [ ]:
#max 78 331uW;87 300uw;97.5 200uw;106.35 100uw;111.5 50uW;115.05 25uw;118.02 10uw;119.5 5uw; 120.55 2.5uW;121.48 1uw;123 0.03UW 

# x0 = np.concatenate((np.arange(0,0.5,0.075),np.arange(0.5,0.85,0.015),np.arange(0.85,0.99,0.01)))
x0 = np.concatenate((np.arange(0.6,0.86,0.02),np.arange(0.86,0.99,0.01)))

power_angles = x0*45+78
rounded_power_angles = np.around(power_angles,decimals=3)[::-1]
x = x0*np.pi/4
powers = np.cos(2*x)*np.cos(2*x)*331+0.03
rounded_powers = np.around(powers,decimals=3)[::-1]
# plt.scatter(power_angles,powers)

# for p,p_a in zip(powers, power_angles):
#     print(p,' ',p_a,'\n')
# print(len(powers))

diff_expose_times = np.zeros_like(x0)
times = 0
for i,power in enumerate(rounded_powers):
    if power < 1:
        diff_expose_times[i] = 120000
    elif  power >= 1 and power < 3:
        diff_expose_times[i] = 90000
    elif  power >= 3 and power < 5:
        diff_expose_times[i] = 60000
    elif  power >= 5 and power < 10:
        diff_expose_times[i] = 45000
    elif  power >= 10 and power < 20:
        diff_expose_times[i] = 25000 
    elif  power >= 20 and power < 30:
        diff_expose_times[i] = 15000  
    elif  power >= 30 and power < 50:
        diff_expose_times[i] = 10000 
    elif  power >= 50 and power < 200:
        diff_expose_times[i] = 10000 
    elif  power>= 200:
        diff_expose_times[i] = 5000
# plt.scatter(powers,diff_expose_times)
for p,p_a,dt in zip(rounded_powers, rounded_power_angles,diff_expose_times):
    print(p,' ',p_a,' ',int(dt),'\n')
    times = times + dt
print(times)
    


In [ ]:
# power dep 
#max 56 358uW; 67.6 300uw;76.7 200uw;85 100uw;90 50uW;93.3 25uw;96.25 10uw;97.75 5uw;100.1 1uw;
import numpy as np
# power_angles = [  67.6, 85,   90, 93.3,96.25,97.75 ,98.75, 100.1][::-1]
# powers =       [   300, 100,  50, 25,  10   ,5   ,2.5  ,1][::-1] 

lf6.change_roi_FullSensor()
lf6.change_spectra_center(860)
for power_angle, power,diff_expose_time in zip(rounded_power_angles, rounded_powers,diff_expose_times):
    lf6.change_expose_time(int(diff_expose_time))
    ep300.set_position(2,power_angle)
    sample_name = '$YZ139_20231229$~$Diff860nmc~In$~$7Kp3$~${}uw$~${}ms$~'.format(power,int(diff_expose_time))
    exps = spectral_experiments.SpectralExperimentsCollections()

    exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-3.5, vbg_stop=3.5,
                                   vtg_start=-3.5, vtg_stop=3.5, frames=141, repeat=1, plot=False)


    exps.execute(repeat=1)

In [ ]:
print(diff_expose_time)

In [ ]:
# power dep 
#max 56 358uW; 67.6 300uw;76.7 200uw;85 100uw;90 50uW;93.3 25uw;96.25 10uw;97.75 5uw;100.1 1uw;
import numpy as np
# [94, 86.3, 81.1, 76.5, 71.7, 69.2, 66.2, 62.6, 60.6, 59.5, 57.7, 56.4, 55]
# [300, 250, 200, 150, 100, 75, 50, 25, 15, 10, 5, 2.5, 1.3] 
# power_angles = [110, 100, 92, 88, 84, 79, 75.5, 73.7, 72.6, 71, 69.7,67.7][::-1]
# powers = [370, 300, 200, 150, 100,  50, 25, 15, 10, 5, 2.5,1.5][::-1] 
# power_angles = [56,  67.6, 76.7, 85,   90, 93.3,96.25,97.75 ,100.1][::-1]
# powers =       [358, 300,  200,  100,  50, 25,  10   ,5     ,1][::-1] 
power_angles = [  67.6, 85,   90, 93.3,96.25,97.75 ,98.75, 100.1][::-1]
powers =       [   300, 100,  50, 25,  10   ,5   ,2.5  ,1][::-1] 
expose_time = 4
lf6.change_expose_time(4000)
# -5 0.0 0.0 -5 $TG+BG=-5.0$ 101
# -5 1.0 1.0 -5 $TG+BG=-4.0$ 121
# -5 2.0 2.0 -5 $TG+BG=-3.0$ 141
# -5 3.0 3.0 -5 $TG+BG=-2.0$ 161
# -5 4.0 4.0 -5 $TG+BG=-1.0$ 181
# -5 5.0 5.0 -5 $TG+BG=0.0$ 201
# 5 -4.0 -4.0 5 $TG+BG=1.0$ 181
# 5 -3.0 -3.0 5 $TG+BG=2.0$ 161
# 5 -2.0 -2.0 5 $TG+BG=3.0$ 141
# 5 -1.0 -1.0 5 $TG+BG=4.0$ 121
# 5 0.0 0.0 5 $TG+BG=5.0$ 101

# 0.0 5 -5 0.0 $TG-BG=-5.0$ 101
# -2.5 5 -5 2.5 $TG-BG=-2.5$ 151
# -5.0 5 -5 5.0 $TG-BG=0.0$ 201
# -5 2.5 -2.5 5 $TG-BG=2.5$ 151
# -5 0.0 0.0 5 $TG-BG=5.0$ 101
for power_angle, power in zip(power_angles, powers):
    if power > 20:
        lf6.change_expose_time(2000)
        expose_time = 2
    ep300.set_position(2,power_angle)
    sample_name = '$DX156_20231224$~$PL730nm$~$7Kp4up$~${}uw$~${}s$~'.format(power,expose_time)
    exps = spectral_experiments.SpectralExperimentsCollections()
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-5.0$', iv, lf6, vbg_start=-5, vbg_stop=0,
#                                        vtg_start=0, vtg_stop=-5, frames=101, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-4.0$', iv, lf6, vbg_start=-5, vbg_stop=1,
#                                        vtg_start=1, vtg_stop=-5, frames=121, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.0$', iv, lf6, vbg_start=-5, vbg_stop=2,
#                                        vtg_start=2, vtg_stop=-5, frames=141, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2.0$', iv, lf6, vbg_start=-5, vbg_stop=3,
#                                        vtg_start=3, vtg_stop=-5, frames=161, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-1.0$', iv, lf6, vbg_start=-5, vbg_stop=4,
#                                        vtg_start=4, vtg_stop=-5, frames=181, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                        vtg_start=5, vtg_stop=-5, frames=201, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.0$', iv, lf6, vbg_start=5, vbg_stop=-4,
#                                        vtg_start=-4, vtg_stop=5, frames=181, repeat=1, plot=True)   
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.0$', iv, lf6, vbg_start=5, vbg_stop=-3,
#                                        vtg_start=-3, vtg_stop=5, frames=161, repeat=1, plot=True)     
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=3.0$', iv, lf6, vbg_start=5, vbg_stop=-2,
#                                        vtg_start=-2, vtg_stop=5, frames=141, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.0$', iv, lf6, vbg_start=5, vbg_stop=-1,
#                                        vtg_start=-1, vtg_stop=5, frames=121, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.0$', iv, lf6, vbg_start=5, vbg_stop=0,
#                                        vtg_start=0, vtg_stop=5, frames=101, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.5$', iv, lf6, vbg_start=5, vbg_stop=0.5,
#                                        vtg_start=0.5, vtg_stop=5, frames=91, repeat=1, plot=True)    
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.0$', iv, lf6, vbg_start=5, vbg_stop=1,
#                                        vtg_start=1, vtg_stop=5, frames=81, repeat=1, plot=True)

#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-5.0$', iv, lf6, vbg_start=0, vbg_stop=5,
#                                   vtg_start=-5, vtg_stop=0, frames=101, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-2.5$', iv, lf6, vbg_start=-2.5, vbg_stop=5,
#                                   vtg_start=-5, vtg_stop=2.5, frames=151, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                   vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=2.5$', iv, lf6, vbg_start=-5, vbg_stop=2.5,
#                                   vtg_start=-2.5, vtg_stop=5, frames=151, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=5.0$', iv, lf6, vbg_start=-5, vbg_stop=0,
#                                   vtg_start=0, vtg_stop=5, frames=101, repeat=1, plot=True)

#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                   vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2.5$', iv, lf6, vbg_start=-5, vbg_stop=2.5,
#                                    vtg_start=2.5, vtg_stop=-5, frames=151, repeat=1, plot=False)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.5$', iv, lf6, vbg_start=-2.5, vbg_stop=5,
#                                    vtg_start=5, vtg_stop=-2.5, frames=151, repeat=1, plot=False)    
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-5.0$', iv, lf6, vbg_start=-5, vbg_stop=0,
#                                    vtg_start=0, vtg_stop=-5, frames=101, repeat=1, plot=False)
    exps.execute(repeat=1)

In [ ]:
lf6.change_expose_time(2000)


In [ ]:
# power dep 
# min 55 1.3uw max 100 300uw  82 200uw  72 100uw 66.5 50uw 60 11uw
import numpy as np
# [91, 80.5, 76, 71.5, 69, 66, 62.5, 60.7,]
# [300, 200, 150, 100, 75, 50, 25, 15,] 
power_angles = [59.5,57.7,56.4,55]
powers = [10,5,2.5,1.3] 
# doping = np.linspace()
for power_angle, power in zip(power_angles, powers):
    ep300.set_position(2,power_angle)
    sample_name = '$YZ129_2ndCD_5cmlens_5050BS_VP_roi$~$PL$~$P2new$~$730nm{}uw$~$8s$~'.format(power)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.0$', iv, lf6, vbg_start=0, vbg_stop=5,
                                       vtg_start=5, vtg_stop=0, frames=101, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0$', iv, lf6, vbg_start=-5, vbg_stop=5,
                                       vtg_start=5, vtg_stop=-5, frames=201, repeat=1, plot=False)
    exps.execute(repeat=1)

In [ ]:
ep300.set_position(2,69)


In [ ]:
file_name = '$YZD73$~$PL730nm$~$7Kpb$~$Power5s$~$TG+BG=0$'
power_angles = [69,  79.5, 87,  93.3, 96.5,  100,  101.6, 103.4,  104.35,  106.6, 108, 108.3,  108.6,  109, 109.35, 109.75, 110.2, 110.7, 111,  112.1]
power =        [229, 200,  150, 100,  75,    50,   40,    30,     20,      15,    10,  9,      8,      7,   6,      5,      4,     3,     2.5,  1]
vbg = 0
vtg = 0
with open('{}.csv'.format(file_name), 'a') as f:
    wls = lf6.get_wavelength_calibration()
    cols = np.concatenate((np.array(['Vbg','Vtg','power'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
    np.savetxt(f, cols, fmt='%s', delimiter=',')
    for i in range(len(power_angles)):
        ep300.set_position(2,power_angles[i])
        spectra = lf6.acquire()
        data = np.concatenate((np.array([vbg, vtg, power[i]], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
        np.savetxt(f, data, fmt='%.5e', delimiter=',')
        
        

## Vtg-Vbg Mega Sweep

In [ ]:
# import numpy as np
# f_name = '$YZD72$~$PL730nm$~$7Kp11$~$2.5uW$~$1s$~$Megasweep$'
# vbgs = np.linspace(-1, 0, 11)
# vtgs = np.linspace(-1, 0, 11)
# flag = True
# for vbg in vbgs:
#     if flag == True:
#         vtgseq = vtgs
#     else:
#         vtgseq = np.flip(vtgs)
#     for vtg in vtgseq:
#         print(vbg,vtg)
#     flag = not flag


In [ ]:
import numpy as np
#  real 19h - 14H
f_name = '$YZ237$~$PL633nm2uw1s$~$6Kp2$~$Megasweep1$'
# vbgs=np.concatenate((np.arange(0,0.75,0.03),np.arange(0.75,1,0.01)))
# vbgs = np.linspace(-4.5, 6, 211)
vbgs = np.linspace(-7, 7, 71)
vtgs = np.linspace(-7, 7, 71)
flag = True
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vtg in vtgs:
                iv.x_goto('Vtg', vtg, 0.1, 0.05)
                if flag == True:
                    vbgseq = vbgs
                else:
                    vbgseq = np.flip(vbgs)
                for vbg in vbgseq:
                    iv.x_goto('Vbg', vbg, 0.05, 0.05)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
            iv.x_goto('Vbg', 0, 0.05, 0.05)
            iv.x_goto('Vtg', 0, 0.05, 0.05)
            print('vg:', 0)

In [ ]:
import numpy as np
#  real 19h - 14H
f_name = '$YZD218$~$PL633nm10uw$~$6KPL$~$P3$~$vtg1=-vtg2_vbgMegasweep$'
# vbgs=np.concatenate((np.arange(0,0.75,0.03),np.arange(0.75,1,0.01)))
# vbgs = np.linspace(-4.5, 6, 211)
# vbgs = np.linspace(-10, 10, 101)
# vtgs = np.linspace(10, -10, 101)
# vyzs = np.linspace(-5, 5, 51)
vbgs = np.linspace(-8, 10, 91)
vtgs = np.linspace(8, -10, 91)
vyzs = np.linspace(-3, 4, 36)
flag = True
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vbg, vtg in zip(vbgs,vtgs):
                iv.x_goto('Vbg', vbg, 0.03, 0.1)
                iv.x_goto('Vtg', vtg, 0.03, 0.1)
                if flag == True:
                    vyzseq = vyzs
                else:
                    vyzseq = np.flip(vyzs)
                for vyz in vyzseq:
                    iv.x_goto('Vyuze', vyz, 0.03, 0.1)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1)
            iv.x_goto('Vyuze', 0, 0.03, 0.1)
            print('vg:', 0)

In [ ]:
# # reflection megasweep with background 
# sample_name = '$YZD154$~$REF$~$10Kp51+1+1$~$720nmC600g$~$2s$~$REF_forBackground$'
# flag = True
# exps = spectral_experiments.SpectralExperimentsCollections()
# exps.add_dual_gate_spectra_sweep(sample_name, '$start$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                     vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=False)
# exps.execute(repeat=1)
# with open('{}.csv'.format(f_name), 'a') as f:
#             wls = lf6.get_wavelength_calibration()
#             cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#             np.savetxt(f, cols, fmt='%s', delimiter=',')
#             for vbg in vbgs:
#                 iv.x_goto('Vbg', vbg, 0.03, 0.1)
#                 if flag == True:
#                     vtgseq = vtgs
#                 else:
#                     vtgseq = np.flip(vtgs)
#                 for vtg in vtgseq:
#                     iv.x_goto('Vtg', vtg, 0.03, 0.1)
#                     spectra = lf6.acquire()
#                     data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#                     np.savetxt(f, data, fmt='%.5e', delimiter=',') 
#                 flag = not flag
#             iv.x_goto('Vbg', 0, 0.03, 0.1)
#             iv.x_goto('Vtg', 0, 0.03, 0.1)
#             print('vg:', 0)
# exps = spectral_experiments.SpectralExperimentsCollections()
# exps.add_dual_gate_spectra_sweep(sample_name, '$end$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                     vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=False)
# exps.execute(repeat=1)

In [ ]:
import numpy as np
f_name = '$YZD73$~$PL730nm$~$7Kpb$~$355uW$~$1s$~$VPMegasweep$'
# f_name = 'test20231001'
vbgs = np.linspace(-5, 5, 101)
vtgs = np.linspace(-5, 5, 101)

In [ ]:
# VP megasweep 
# KK KKp
halfangles = [46,91]
flag = True
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg','halfangle'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vbg in vbgs:
                iv.x_goto('Vbg', vbg, 0.03, 0.1)
                if flag == True:
                    vtgseq = vtgs
                else:
                    vtgseq = np.flip(vtgs)
                for vtg in vtgseq:
                    iv.x_goto('Vtg', vtg, 0.03, 0.1)
                    for halfangle in halfangles:
                        ep300.set_position(1,halfangle)
                        spectra = lf6.acquire()
                        data = np.concatenate((np.array([vbg, vtg, halfangle], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                        np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1)
            print('vg:', 0)

## Check gate


In [ ]:
iv.x_goto('Vbg', 0, 0.1, 0.1)
iv.x_goto('Vtg', 0, 0.1, 0.1)
iv.report_status()


## Pump Probe

In [10]:
import os, csv
def check_file_name(sample_name: str):
    file_number = 0
    while True:
        file_number += 1
        new_file_name = '{}_{:0>3d}'.format(sample_name, file_number)
        csv_name = new_file_name + '.csv'
        if not os.path.exists(csv_name):
            return new_file_name
        
def transpose_csv_wls_to_energy(in_path: str, out_path: str):
    """
    Transpose a CSV produced by dual_gate_multi_spectra_sweep_saveasrow
    and replace the wavelength labels by energy (eV) in the transposed file.

    Assumes original header is:
    Vbg, Vtg, Vtg+Vbg, Vtg-Vbg, wl1, wl2, ...
    """
    # ---- First pass: read header, get wavelengths & energies ----
    with open(in_path, 'r', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)             # first row
        n_cols = len(header)

        # header[4:] are wavelength values in nm (as strings)
        wl_strings = header[4:]
        wavelengths = np.array([float(x) for x in wl_strings], dtype=float)
        energies = 1240.0 / wavelengths   # eV, if λ is in nm

    # ---- Second step: transpose, but swap wl -> energy on the spectral rows ----
    with open(out_path, 'w', newline='') as fout:
        writer = csv.writer(fout)

        # each col_idx of the original becomes one row in the transposed file
        for col_idx in range(n_cols):
            row_out = []
            with open(in_path, 'r', newline='') as fin:
                reader = csv.reader(fin)
                for row in reader:
                    if col_idx < len(row):
                        row_out.append(row[col_idx])
                    else:
                        row_out.append('')

            # For spectral columns (original columns 4+),
            # row_out[0] currently = wavelength; override with energy
            if col_idx >= 4:
                energy_idx = col_idx - 4
                row_out[0] = f"{energies[energy_idx]:.6f}"  # e.g. 1.234567

            writer.writerow(row_out)    

def dual_gate_multi_spectra_sweep_saveasrow(sample_name: str, exp_name: str, frames_per_exposure: int, image_mode, 
                                  vbg_start, vbg_stop, vtg_start, vtg_stop, steps, 
                                  save_raw_spectras: bool = False, chunk_size: int = 100):
    
    new_file_name = check_file_name(f"{sample_name}_{exp_name}")
    output_file_name = new_file_name + '.csv'

    Vtgs = np.linspace(vtg_start, vtg_stop, steps)
    Vbgs = np.linspace(vbg_start, vbg_stop, steps)
    lf6.set_frames_to_save(frames=str(frames_per_exposure))

    wls = lf6.get_wavelength_calibration()

    with open(output_file_name, 'a') as f:
        cols = np.concatenate((np.array(['Vbg','Vtg','Vtg+Vbg','Vtg-Vbg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
        np.savetxt(f, cols, fmt='%s', delimiter=',')

        for bg_voltage, tg_voltage in zip(Vbgs, Vtgs):
    #     Diffusion image
            
            iv.x_goto('Vbg', bg_voltage, 0.1, 0.1)
            iv.x_goto('Vtg', tg_voltage, 0.1, 0.1)

            spectra = lf6.multi_frame_acquire(image_mode=image_mode, x_pixels_num=1024, y_pixels_num=208)
            spectra_omit = spectra[4:,:]
            differentials = spectra_omit[::2] - spectra_omit[1::2]

            # differential = np.zeros((spectra.shape[1],))  # Initialize as zero array for 1600 wavelengths
            differential = np.sum(differentials, axis=0)

            # on_spectra = np.zeros((spectra.shape[1],))
            # off_spectra = np.zeros((spectra.shape[1],))
            # for i in range(frames_per_exposure):
            #     differentials += abs(spectra[i + 1] - spectra[i])
            #     on_spectra += spectra[i + 1]
            #     off_spectra += spectra[i]
            # # Adjust loop to handle cases where frames_per_exposure < chunk_size
            # total_frames = spectra.shape[0]
            # num_chunks = total_frames // chunk_size
            # remaining_frames = total_frames % chunk_size

            # # If there are fewer frames than the chunk size, process the remaining frames
            # if remaining_frames > 0:
            #     num_chunks += 1  # Ensure the last chunk is processed as well

            # # Process in chunks
            # for chunk in range(num_chunks):
            #     # Determine the range of frames in this chunk
            #     start_idx = chunk * chunk_size
            #     end_idx = min(start_idx + chunk_size, total_frames)  # Handle remaining frames
                
            #     # Process the frames for this chunk
            #     for i in range(start_idx, end_idx, 2):
            #         differentials += abs(spectra[i + 1] - spectra[i])
            #         on_spectra += spectra[i + 1]
            #         off_spectra += spectra[i]
            abs_differential = np.abs(differential)
            abs_max_idx  = np.argmax(abs_differential)
            max = differential[abs_max_idx]
            if max == abs_differential[abs_max_idx]:
                data_differential = differential
            else:
                data_differential = -differential

            data = np.concatenate((np.array([bg_voltage, tg_voltage, tg_voltage+bg_voltage, tg_voltage-bg_voltage], ndmin=1, dtype=np.float64), 
                                   data_differential)).reshape([1, -1])
            with open(output_file_name, 'a') as f:
                np.savetxt(f, data, fmt='%.5e', delimiter=',')

    print(f"Data saved to {output_file_name}")

    # Make transposed CSV with energy instead of wavelength
    transposed_output_file_name = new_file_name + '_T.csv'
    transpose_csv_wls_to_energy(output_file_name, transposed_output_file_name)
    print(f"Transposed (energy-axis) data saved to {transposed_output_file_name}")

    iv.x_goto('Vbg', 0, 0.1, 0.1)
    iv.x_goto('Vtg', 0, 0.1, 0.1)


In [65]:
probe_pwr = 0.5
meas_power = c_double()
tlPM.setWavelength(c_double(730),TLPM_DEFAULT_CHANNEL)

tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
print('Measured Power (uW): ', power)

Measured Power (uW):  -0.489


In [ ]:
#x_goto(self, x_name, target, delta, delay)
iv.x_goto('Vtg',2.1, 0.1, 0.05)
iv.x_goto('Vbg', 1.575, 0.1, 0.05)
iv.report_status()


In [ ]:
stage.move_to(2700)
probe_pwr = 0.5
meas_power = c_double()
tlPM.setWavelength(c_double(660),TLPM_DEFAULT_CHANNEL)

tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
print('Measured Power (uW): ', power)

In [ ]:
probe_stage.get_position()
# 800 0.25uw
# 1400 0.58uw

In [91]:
probe_stage.move_to(1500)
probe_pwr = 0.0
meas_power = c_double()
tlPM.setWavelength(c_double(730),TLPM_DEFAULT_CHANNEL)

tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
print('Measured Power (uW): ', power)

Measured Power (uW):  1.009


In [ ]:
# stage_poss = np.concatenate((np.linspace(100, 1500, 31),np.linspace(1500, 3000, 35)))
# stage_pos = [0.0, 164.12, 286.72, 387.71, 475.92, 555.77, 629.69, 699.23, 765.37, 828.79, 
#              890.01, 949.43, 1007.35, 1064.04, 1119.7, 1174.5, 1493.58, 1548.8, 1604.53, 
#              1660.67, 1717.06, 1773.57, 1830.08, 1886.46, 1942.61, 1998.42, 2053.84, 2108.83, 
#              2163.36, 2217.43, 2271.04, 2324.23, 2377.06, 2429.56, 2481.8, 2533.88, 2585.89, 
#              2637.96, 2690.22, 2742.84, 2796.01, 2849.97, 2904.99, 2961.42, 3019.67, 3080.24, 
#              3143.76, 3211.02, 3283.13, 3361.64, 3600.0]
stage_poss = np.concatenate((0,np.linspace(200, 1000, 21),np.linspace(1025, 2700, 71), np.linspace(2725, 3600, 31)[1:]))
for stage_pos in stage_poss:
    stage.move_to(stage_pos)
    time.sleep(1)
    probe_pwr = 0.5
    meas_power = c_double()
    tlPM.setWavelength(c_double(660),TLPM_DEFAULT_CHANNEL)

    tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
    power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
    print(f'Stage: {stage_pos}, Measured Power (uW): {power}')

In [ ]:
# Gate-dep at fixed power
stage_poss = [1000]
# stage_poss = np.linspace(0,3600,37)
#1uw 10uw 30uw 
center = '860'
lf6.change_spectra_center(center)
frames_per_exposure = 100
probe_pwr = 0.25
for stage_pos in stage_poss:
    stage.move_to(stage_pos)
    time.sleep(1)

    # Measure power
    meas_power = c_double()
    tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
    power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
    print(f"Stage @ {stage_pos}, Measured power: {power} µW")
    
    sample_name = (f'$YZD112$~$PL{center}nmcSpectra$~$660nmPump{power}uw$~$730nmProbe{probe_pwr}uw$~'
                    f'$CCDRpT90msfrs={frames_per_exposure}$~$6Kpdn3$~$4_8Hz$~')
    exp_name = 'TG-BG=0'

    dual_gate_multi_spectra_sweep_saveasrow(sample_name=sample_name, exp_name=exp_name, 
                                            frames_per_exposure=frames_per_exposure, image_mode=False, 
                                    vbg_start = -3, vbg_stop= 3, 
                                    vtg_start= -3 , vtg_stop= 3, steps=121, 
                                    save_raw_spectras = False, chunk_size = 100)

In [ ]:
import numpy as np

def solve_vtg_vbg(doping):
    doping = np.asarray(doping, dtype=float)
    vtg = (2/3) * doping
    vbg = 0.5 * doping
    return vtg, vbg

D = [-5.6, -3.77, -1.87, 0, 1.59, 3.17, 4.76]
vtg, vbg = solve_vtg_vbg(D)
print("vtg:", vtg)
print("vbg:", vbg)


In [ ]:
stage_poss = np.linspace(0, 3600, 17)
print(stage_poss)

### Power dep at Fixed gate and Fixed Center


In [35]:
# Helper function to acquire and save
def acquire_and_save(center_key, power, bg_voltage, tg_voltage, centers, frames_per_exposure, stage_pos):
    info = centers[center_key]
    lf6.change_spectra_center(info['center_value'])
    lf6.set_frames_to_save(frames=str(frames_per_exposure))

    spectra = lf6.multi_frame_acquire(image_mode=False, x_pixels_num=1024, y_pixels_num=200)
    spectra_omit = spectra[4:, :]
    differentials = spectra_omit[::2] - spectra_omit[1::2]
    differential = np.sum(differentials, axis=0)

    abs_differential = np.abs(differential)
    abs_max_idx = np.argmax(abs_differential)
    max_val = differential[abs_max_idx]
    data_differential = differential if max_val == abs_differential[abs_max_idx] else -differential

    # ---- reduce digits here ----
    # gates & power: 3 decimal places
    gates = np.array(
        [bg_voltage, tg_voltage, tg_voltage + bg_voltage, stage_pos, power],
        ndmin=1, dtype=np.float64
    )
    gates = np.round(gates, 3)

    # spectrum: 4 decimal places
    spec = np.round(data_differential/frames_per_exposure, 4)

    data = np.concatenate((gates, spec)).reshape([1, -1])
    fmt = ['%.3f'] * 5 + ['%.4f'] * spec.size

    with open(f"{info['newfilename']}.csv", 'a') as f:
        np.savetxt(f, data, fmt=fmt, delimiter=',')

import csv      
def transpose_and_sort_by_power(in_path, out_path):
    """
    Transpose a CSV file and sort the transposed columns by 'Power' ascending.
    Does NOT load the full spectra matrix into memory:
    - keeps only O(n_rows) info (powers + index mapping).
    """
    # ---------- PASS 1: read header, find power column, collect powers ----------
    with open(in_path, 'r', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)  # original header row
        n_cols = len(header)

        # Find the 'Power' column index robustly
        try:
            power_col_idx = header.index('Power')
        except ValueError:
            # Fallback to 3 if header name not found
            power_col_idx = 3

        powers = []  # list of (row_index, power_value)
        for i, row in enumerate(reader, start=1):
            if len(row) <= power_col_idx:
                continue
            try:
                p = float(row[power_col_idx])
            except ValueError:
                # skip rows that don't have numeric power (shouldn't happen here)
                continue
            powers.append((i, p))

    if not powers:
        raise RuntimeError("No numeric power values found; cannot sort by power.")

    # Sort data rows by power ascending
    sorted_row_indices = [row_idx for (row_idx, p) in sorted(powers, key=lambda x: x[1])]

    # Column 0 in the transposed file will correspond to original row 0 (header),
    # subsequent columns correspond to these sorted data rows.
    row_order = [0] + sorted_row_indices
    n_rows_out = len(row_order)

    # Map original row index -> position in transposed columns
    row_pos = {row_idx: pos for pos, row_idx in enumerate(row_order)}

    # ---------- PASS 2: transpose in this column order ----------
    with open(out_path, 'w', newline='') as fout:
        writer = csv.writer(fout)

        # For each original column j, build a transposed row:
        for col_idx in range(n_cols):
            # One output row = this column across all rows, ordered by row_order
            row_out = [''] * n_rows_out

            with open(in_path, 'r', newline='') as fin:
                reader = csv.reader(fin)
                for i, row in enumerate(reader):
                    # we only care about header row (0) and data rows that have power
                    if i not in row_pos:
                        continue

                    pos = row_pos[i]
                    val = row[col_idx] if col_idx < len(row) else ''
                    row_out[pos] = val

            writer.writerow(row_out)

In [ ]:
Vtgs=np.linspace(1.5,-1.6,17)
print(2*Vtgs)


In [ ]:
stage_poss = np.concatenate([np.linspace(0, 3000, 65), np.linspace(3000, 3600, 13)])
print(stage_poss)

In [ ]:
current_pos = stage.get_position()
print(current_pos)

In [105]:
stage.move_to(2315)
probe_pwr = 0.5
meas_power = c_double()
tlPM.setWavelength(c_double(660),TLPM_DEFAULT_CHANNEL)

tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
print('Measured Power (uW): ', power)

Measured Power (uW):  200.772


In [106]:
# Power dep at Fixed gate
# ---- User settings ----
probe_pwr = 0.5 # in µW
mode = 'diffusion'  # 'spectra' or 'diffusion'
# centers_num = [860]  # <-- Add/remove centers here
tlPM.setWavelength(c_double(660),TLPM_DEFAULT_CHANNEL)

stage_poss = np.concatenate(([0,np.linspace(200, 1000, 15),np.linspace(1025, 2700, 71), np.linspace(2725, 3600, 21)][1:]))

stage_poss = [2315]
centers_num = [877.76]  # <-- Add/remove centers here


# define your gate pairs: e.g. (vtg=0,vbg=2), (vtg=1,vbg=3), ...
# Vtgs = np.array([ 0, 0.5,  0.8,    1.3,  1.7,    2,  2.25 ][::-1])
# Vbgs = np.array([ 0, 0.375,  0.6 ,0.975,  1.275,1.5,1.6875 ][::-1])
Vtgs = np.array([0])
Vbgs = np.array([0 ])

# Stage @ 2365.0, Power: 212.605 µW
#   -> LE Acquisition @ 879.4 nm
# Center Wave Length: 879.4

frames_per_exposure = 1000

exp_time = '85'  # in ms
lf6.change_expose_time(exp_time)


# ========== MAIN: loop over gate pairs, then power (stage) ==========
for tg_voltage, bg_voltage in zip(Vtgs, Vbgs):
    print(f"\n=== New gate pair: Vtg={tg_voltage} V, Vbg={bg_voltage} V ===")

    # set gates for this whole power sweep
    iv.x_goto('Vbg', bg_voltage, 0.1, 0.1)
    iv.x_goto('Vtg', tg_voltage, 0.1, 0.1)

    # build centers dict *for this gate pair* (so filenames include Vtg/Vbg)
    centers = {}
    for i, center in enumerate(centers_num, start=1):
        center_key = f'center_{i}'

        # filename includes Vtg= and Vbg=
        filename = (
            f'$YZD219_powerdep$~${center}nm{mode}$~'
            f'$Vtg={tg_voltage:.3f}V$~$Vbg={bg_voltage:.3f}V$~'
            f'$660nmPump$~$730nm{probe_pwr}uwProbe$~'
            f'$RpT{exp_time}ms_f={frames_per_exposure}$~$6Kp3n2$~$4_8Hz300g$~'
        )

        newfilename = check_file_name(filename)

        lf6.change_spectra_center(str(center))
        wls = lf6.get_wavelength_calibration()
        energy = np.round(1240 / wls, 5)

        centers[center_key] = {
            "center_value": str(center),
            "filename": filename,
            "newfilename": newfilename,
            "wavelengths": wls,
            "energy": energy,
            "Vbg": bg_voltage,
            "Vtg": tg_voltage,
            "frames_per_exposure": frames_per_exposure,
        }

        # write header row (energy axis)
        with open(f'{newfilename}.csv', 'w', newline='') as f:
            cols = np.concatenate(
                (np.array(['Vbg', 'Vtg', 'Vtg+Vbg', 'StagePos','Power'], ndmin=1, dtype='U'),
                 energy.astype('U'))
            ).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')

    # ---- sweep power for this fixed gate pair ----
    for stage_pos in stage_poss:
        stage_pos = round(stage_pos,1)
        # ---- CONFIGURATION ----
        max_retries = 3
        tolerance = 1.0  # Acceptable error margin (units depend on your stage)
        move_success = False

        for attempt in range(max_retries):
            try:
                # Only print retry attempts to keep log clean
                if attempt > 0:
                    print(f"  -> Retry move to {stage_pos} (Attempt {attempt + 1})...")
                
                stage.move_to(stage_pos)
                time.sleep(1.0) 
                # --- VERIFICATION ---
                current_pos = stage.get_position() 
                # Check if we are close enough (within +/- 1.0)
                if abs(current_pos - stage_pos) <= tolerance:
                    move_success = True
                    break 
                else:
                    diff = current_pos - stage_pos
                    print(f"  -> Position Mismatch! Target: {stage_pos}, Actual: {current_pos:.2f} (Diff: {diff:.2f})")
            except Exception as e:
                print(f"  -> Error on move attempt {attempt + 1}: {e}")
                time.sleep(0.5) # Wait before retrying
        if not move_success:
            print(f"CRITICAL: Could not reach {stage_pos} after {max_retries} attempts. Skipping.")
            continue

        # Measure power
        meas_power = c_double()
        tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
        power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
        print(f"Stage @ {stage_pos}, Measured power: {power} µW")
        # if power <70:
        #     frames_per_exposure=50
        # elif 70<=power <150:
        #     frames_per_exposure =100
        # elif 150<=power<500:
        #     frames_per_exposure = 200
        # else:
        #     frames_per_exposure = 500
        # acquire at all centers for current power & gate pair
        for center_key in centers.keys():
            acquire_and_save(center_key, power, bg_voltage, tg_voltage, centers, frames_per_exposure, stage_pos)

    # after finishing power sweep at this gate pair, transpose + sort by power
    for center_key, info in centers.items():
        in_file = f"{info['newfilename']}.csv"
        out_file = f"{info['newfilename']}_Tsorted.csv"
        transpose_and_sort_by_power(in_file, out_file)
        print("Transposed & power-sorted file:", out_file)

# return stage and gates to 0 at the very end
# iv.x_goto('Vbg', 0, 0.1, 0.1)
# iv.x_goto('Vtg', 0, 0.1, 0.1)
# stage.move_to(0)

Exposetime(ms): 85.0

=== New gate pair: Vtg=0 V, Vbg=0 V ===
variable: ['Vbg']
start: [0.]
end: [0]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0]
steps: 2
Center Wave Length: 877.76
Grating: [750nm,300][2][0]
Stage @ 2315, Measured power: 199.846 µW
Center Wave Length: 877.76
Grating: [750nm,300][2][0]
Frame: 1000
Frame: 1000
Transposed & power-sorted file: $YZD219_powerdep$~$877.76nmdiffusion$~$Vtg=0.000V$~$Vbg=0.000V$~$660nmPump$~$730nm0.5uwProbe$~$RpT85ms_f=1000$~$6Kp3n2$~$4_8Hz300g$~_001_Tsorted.csv


### Pump Probe Diffusion at Fix gate with different centers

In [60]:
# Helper function to acquire and save
def acquire_and_append(info, stage_pos, power, bg_voltage, tg_voltage, frames_per_exposure):
    """
    Acquires data and appends a row to the open CSV file.
    New Row format: [Vbg, Vtg, Sum, Stage_Pos, Power, Center_Used, Data_0...]
    """
    center_val = float(info['center_value'])
    filename = info['filename_actual']
    
    # Move Spectrometer
    lf6.change_spectra_center(str(center_val))
    lf6.set_frames_to_save(frames=str(frames_per_exposure))

    # Acquire
    spectra = lf6.multi_frame_acquire(image_mode=False, x_pixels_num=1024, y_pixels_num=200)
    spectra_omit = spectra[4:, :]
    differentials = spectra_omit[::2] - spectra_omit[1::2]
    differential = np.sum(differentials, axis=0)

    # Calculate Signal
    abs_differential = np.abs(differential)
    abs_max_idx = np.argmax(abs_differential)
    max_val = differential[abs_max_idx]
    data_differential = differential if max_val == abs_differential[abs_max_idx] else -differential

    # Prepare Data Row
    # Metadata Order: Vbg, Vtg, Sum, Stage_Pos, Power, Center_nm
    meta = np.array(
        [bg_voltage, tg_voltage, tg_voltage + bg_voltage, stage_pos, power, center_val],
        dtype=np.float64
    )
    meta = np.round(meta, 3)

    # Spectrum Data
    spec = np.round(data_differential/frames_per_exposure, 4)

    # Combine
    data = np.concatenate((meta, spec)).reshape([1, -1])
    
    # Format: 6 metadata columns + 1024 data columns
    fmt = ['%.3f'] * 6 + ['%.4f'] * spec.size

    # Append to file
    with open(f"{filename}.csv", 'a') as f:
        np.savetxt(f, data, fmt=fmt, delimiter=',')

import csv

def transpose_and_sort_by_power(in_path, out_path):
    """
    Transposes CSV, sorts by 'Power', and REMOVES 'Stage_Pos' from the output.
    """
    # ---------- PASS 1: Read header, find columns ----------
    with open(in_path, 'r', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        n_cols = len(header)

        # Find column indices
        try:
            power_col_idx = header.index('Power')
        except ValueError:
            print("Error: 'Power' column not found.")
            return

        try:
            stage_col_idx = header.index('Stage_Pos')
        except ValueError:
            stage_col_idx = -1  # If not found, ignore

        # Collect rows (index, power)
        powers = []
        for i, row in enumerate(reader, start=1):
            if len(row) <= power_col_idx:
                continue
            try:
                p = float(row[power_col_idx])
                powers.append((i, p))
            except ValueError:
                continue

    if not powers:
        print("No numeric power values found; skipping sort.")
        return

    # Sort rows by power
    sorted_row_indices = [row_idx for (row_idx, p) in sorted(powers, key=lambda x: x[1])]
    row_order = [0] + sorted_row_indices  # 0 is the header row
    
    # Map original row index -> position in transposed file
    row_pos = {row_idx: pos for pos, row_idx in enumerate(row_order)}
    n_rows_out = len(row_order)

    # ---------- PASS 2: Transpose (Skipping Stage_Pos) ----------
    with open(out_path, 'w', newline='') as fout:
        writer = csv.writer(fout)

        # Loop over columns
        for col_idx in range(n_cols):
            
            # IF this column is the Stage_Pos column, SKIP IT entirely
            if col_idx == stage_col_idx:
                continue

            row_out = [''] * n_rows_out

            with open(in_path, 'r', newline='') as fin:
                reader = csv.reader(fin)
                for i, row in enumerate(reader):
                    if i not in row_pos:
                        continue

                    pos = row_pos[i]
                    val = row[col_idx] if col_idx < len(row) else ''
                    row_out[pos] = val

            writer.writerow(row_out)

In [42]:
def str_to_array_rounded(raw_str, target_length=36, decimals=1):
    """
    1. Converts string to float array.
    2. Pads with NaN if copy-paste missed the last empty lines.
    3. Rounds values to specified decimals.
    """
    # Split by newline
    lines = raw_str.strip().split('\n')
    data = []
    
    for line in lines:
        line = line.strip()
        # Handle empty strings or explicit 'nan' text
        if not line or line.lower() == 'nan':
            data.append(np.nan)
        else:
            try:
                val = float(line)
                data.append(val)
            except ValueError:
                data.append(np.nan)
    
    # --- AUTO-PAD LOGIC ---
    # If we are short (e.g., got 35 but need 36), fill the rest with NaN
    current_len = len(data)
    if current_len < target_length:
        missing_count = target_length - current_len
        print(f"⚠️ Found {current_len} items. Auto-padding {missing_count} NaN(s) to match target {target_length}.")
        data.extend([np.nan] * missing_count)
    elif current_len > target_length:
        print(f"⚠️ Warning: Found {current_len} items, expected {target_length}. Check your paste!")
        
    # Convert to numpy array and round
    # We use a mask to round only non-NaN values to avoid warnings
    arr = np.array(data)
    mask = ~np.isnan(arr)
    arr[mask] = np.round(arr[mask], decimals)
    
    return arr

In [ ]:
# --- PASTE COLUMN 1 (HE) ---
raw_HE = """
865.2984784
865.198831
864.4463188
864.4557429
864.2029517
862.3652478

860.1565644
859.7219129
859.3378181
859.1162559
858.9085693
858.7027972
858.6121489
858.3693956
858.0833271
857.8706321
857.6445617
857.6423031
857.4749782
857.5873622
857.3852683
859.6517989
857.8911315
856.2195644
858.1464563
856.7647689
857.7664932
856.3860282
855.7594749
855.7594749
855.6295504
854.7506446
854.7506446
853.4914778
853.4914778

""" 
# (Paste the full column above)

# --- PASTE COLUMN 2 (LE) ---
raw_LE = """
883.2361555
883.2290874
883.2191152
883.1866155
883.0009367
882.6266045
882.2415618
881.9412499
881.621641
881.2600321
880.8648966
880.6752322
880.5616309
880.3549863
879.9699187
879.5961532
879.100077
878.9438853
878.5847755
878.2107303
877.8587682
877.3349456
877.1737439
875.1738207
874.9902797
874.6737213
873.5521236
873.4290464
873.1768467
871.1767227
872.9309386
874.6737213
872.6790264
870.8034836
871.8015427


"""
# (Paste the full column above)

# ---- User settings ----
probe_pwr = 0.4  # in µW



stage_poss = np.linspace(0, 3600, 36)
# stage_poss = [3600]

# --- CONVERT WITH AUTO-PADDING ---
# This will force them to be exactly len(stage_poss) items long
centers_HE = str_to_array_rounded(raw_HE, target_length=len(stage_poss), decimals=1)
centers_LE = str_to_array_rounded(raw_LE, target_length=len(stage_poss), decimals=1)

# --- VERIFY ---
print(f"\nFinal HE Length: {len(centers_HE)}")
print(f"Final LE Length: {len(centers_LE)}")

# Check if they look correct (print the last 5 items)
print("HE:", centers_HE)
print("LE:", centers_LE)

In [ ]:
# define your gate pairs: e.g. (vtg=0,vbg=2), (vtg=1,vbg=3), ...
Vtgs = np.array([0])
Vbgs = np.array([0])

frames_per_exposure = 500
exp_time = '90'  # in ms
lf6.change_expose_time(exp_time)

# ==========================================
#  MAIN LOOP
# ==========================================
for tg_voltage, bg_voltage in zip(Vtgs, Vbgs):
    print(f"\n=== New gate pair: Vtg={tg_voltage} V, Vbg={bg_voltage} V ===")

    # 1. Set Gates
    iv.x_goto('Vbg', bg_voltage, 0.1, 0.1)
    iv.x_goto('Vtg', tg_voltage, 0.1, 0.1)

    # 2. Initialize the TWO Master Files
    base_name = (
        f'$YZD219_Diffusion$'
        f'$Vtg={tg_voltage:.3f}V$~$Vbg={bg_voltage:.3f}V$~'
        f'$660nmPump$~$730nm{probe_pwr}uwProbe$~'
        f'$CCDRpT{exp_time}ms_800LPfrs={frames_per_exposure}$'
    )

    filename_LE = check_file_name(f"{base_name}~$LE_series$")
    filename_HE = check_file_name(f"{base_name}~$HE_series$")

    # Update Headers: Stage_Pos is now BEFORE Power
    pixel_indices = np.arange(1024).astype(str)
    header_cols = np.concatenate(
        (np.array(['Vbg', 'Vtg', 'Vtg+Vbg', 'Stage_Pos', 'Power', 'Center_nm'], dtype='U'), pixel_indices)
    )

    for fname in [filename_LE, filename_HE]:
        with open(f'{fname}.csv', 'w', newline='') as f:
            np.savetxt(f, header_cols.reshape([1, -1]), fmt='%s', delimiter=',')

    # 3. Sweep Power (Stage)
    for i, (stage_pos, le_center, he_center) in enumerate(zip(stage_poss, centers_LE, centers_HE)):
        
        # -- Move Stage --
        try:
            stage.move_to(stage_pos)
        except Exception as e:
            print(f"Error moving stage to {stage_pos}: {e}")
            continue
        time.sleep(2)

        # -- Measure Power --
        meas_power = c_double()
        tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
        power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
        print(f"Stage @ {stage_pos}, Power: {power} µW")

        # -- Measure LE (if not NaN) --
        if not np.isnan(le_center):
            print(f"  -> LE Acquisition @ {le_center} nm")
            info_LE = {
                'filename_actual': filename_LE,
                'center_value': str(le_center)
            }
            # Passing stage_pos BEFORE power
            acquire_and_append(info_LE, stage_pos, power, bg_voltage, tg_voltage, frames_per_exposure)
        else:
            print("  -> Skipping LE (NaN)")

        # -- Measure HE (if not NaN) --
        if not np.isnan(he_center):
            print(f"  -> HE Acquisition @ {he_center} nm")
            info_HE = {
                'filename_actual': filename_HE,
                'center_value': str(he_center)
            }
            # Passing stage_pos BEFORE power
            acquire_and_append(info_HE, stage_pos, power, bg_voltage, tg_voltage, frames_per_exposure)
        else:
            print("  -> Skipping HE (NaN)")

    # 4. Sort Both Files (and remove Stage_Pos column)
    print("Sorting LE file...")
    try:
        transpose_and_sort_by_power(f"{filename_LE}.csv", f"{filename_LE}_Tsorted.csv")
    except Exception as e:
        print(f"LE Sort failed: {e}")

    print("Sorting HE file...")
    try:
        transpose_and_sort_by_power(f"{filename_HE}.csv", f"{filename_HE}_Tsorted.csv")
    except Exception as e:
        print(f"HE Sort failed: {e}")

# Reset
iv.x_goto('Vbg', 0, 0.1, 0.1)
iv.x_goto('Vtg', 0, 0.1, 0.1)
stage.move_to(0)

### Pump probe Diffusion at fixed gate with read peak wavelength csv


In [40]:
stage.move_to(2500)
probe_pwr = 0.3
meas_power = c_double()
tlPM.setWavelength(c_double(660),TLPM_DEFAULT_CHANNEL)

tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
print('Measured Power (uW): ', power)

Measured Power (uW):  255.272


In [36]:
import pandas as pd
import glob
from ctypes import c_double, byref

# ==================== HELPERS ====================
def get_series_safe(df, col_name, label):
    """Safely extracts a column, handling missing data with NaNs."""
    if col_name in df.columns:
        return pd.to_numeric(df[col_name], errors='coerce').values
    print(f"Notice: '{col_name}' ({label}) missing. Filling with NaNs.")
    return np.full(len(df), np.nan)

def measure_and_save(center, filename, label, stage_pos, pwr, bg, tg, frames):
    """Checks if center is valid, then acquires and saves."""
    if not np.isnan(center):
        print(f"  -> {label} Acquisition @ {center} nm")
        info = {'filename_actual': filename, 'center_value': str(center)}
        acquire_and_append(info, stage_pos, pwr, bg, tg, frames)
    else:
        print(f"  -> Skipping {label} (NaN center)")

def safe_sort(fname, label):
    """Wraps the sorting function with error handling."""
    print(f"Sorting {label} file...")
    try:
        transpose_and_sort_by_power(f"{fname}.csv", f"{fname}_Tsorted.csv")
    except Exception as e:
        print(f"{label} Sort failed: {e}")

In [38]:
probe_stage.get_position()
# 800 0.25uw
# 1400 0.58uw
#1900 1.2uw

999.9609375

In [104]:
probe_stage.move_to(1000
                    )
probe_pwr = 0
meas_power = c_double()
tlPM.setWavelength(c_double(730),TLPM_DEFAULT_CHANNEL)

tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
print('Measured Power (uW): ', power)

Measured Power (uW):  8.75


In [ ]:
# ==================== SETUP ====================
DEV_NAME = 'YZD219'
Pump_Probe_center_folder = rf"D:\instrument_control_v3_1\{DEV_NAME}\pump_probe\centers"
# Vtgs = np.array([1.2, 1.05, 0.9, 0.8,0.7, 0.55, 0.25])[::-1]
# Vbgs = np.array([1.2, 1.05, 0.9, 0.8, 0.7, 0.55, 0.25])[::-1]
Vtgs = np.array([0.0])
Vbgs = np.array([0.0])
# Vtgs = np.array([1.2, 1.125,1.05, 0.975,0.9, ])
# Vbgs = np.array([1.2,1.125, 1.05,0.975, 0.9, ])
probe_pwr = 0.5  # in µW
frames_per_exposure, exp_time = 2000, '85'
lf6.change_expose_time(exp_time)

# ==================== MAIN LOOP ====================
for tg_voltage, bg_voltage in zip(Vtgs, Vbgs):
    print(f"\n=== New gate pair: Vtg={tg_voltage} V, Vbg={bg_voltage} V ===")
    iv.x_goto('Vbg', bg_voltage, 0.1, 0.1)
    iv.x_goto('Vtg', tg_voltage, 0.1, 0.1)

    # 1. Read Centers CSV
    tg_str, bg_str = f"Vtg={tg_voltage:.3f}V", f"Vbg={bg_voltage:.3f}V"
    search_pattern = os.path.join(Pump_Probe_center_folder, "*.csv")
    
    # Find file containing both voltage strings
    found_csv = next((f for f in glob.glob(search_pattern) if tg_str in f and bg_str in f), None)

    if not found_csv:
        print(f"CRITICAL ERROR: No CSV found for {tg_str}, {bg_str}. Skipping.")
        continue

    try:
        print(f"Reading centers from: {os.path.basename(found_csv)}")
        df = pd.read_csv(found_csv)
        
        stage_poss = get_series_safe(df, 'StagePos', 'Stage')
        if np.all(np.isnan(stage_poss)): 
            print("CRITICAL: StagePos missing/invalid. Skipping."); continue

        # Mapping: Peak 1 -> LE, Peak 2 -> HE
        centers_LE = np.round(get_series_safe(df, 'peak1_nm', 'LE'), 1)
        centers_HE = np.round(get_series_safe(df, 'peak2_nm', 'HE'), 1)

    except Exception as e:
        print(f"Error reading file: {e}"); continue

    # 2. Initialize Files
    base_name = (f'${DEV_NAME}_p3n2Diffusion$$Vtg={tg_voltage:.3f}V$~$Vbg={bg_voltage:.3f}V$~'
                 f'$660nmPump$~$730nm{probe_pwr}uwProbe$~$CCDRpT{exp_time}ms_800LPfrs={frames_per_exposure}$')
    
    f_LE = check_file_name(f"{base_name}~$LE_series$")
    f_HE = check_file_name(f"{base_name}~$HE_series$")

    header = np.concatenate((['Vbg', 'Vtg', 'Vtg+Vbg', 'Stage_Pos', 'Power', 'Center_nm'], 
                             np.arange(1024).astype(str))).reshape([1, -1])
    
    for f in [f_LE, f_HE]:
        with open(f'{f}.csv', 'w', newline='') as csvfile:
            np.savetxt(csvfile, header, fmt='%s', delimiter=',')

    # 3. Sweep Power
    for stage_pos, le_c, he_c in zip(stage_poss, centers_LE, centers_HE):
        # ---- CONFIGURATION ----
        max_retries = 3
        tolerance = 1.0  # Acceptable error margin (units depend on your stage)
        move_success = False

        for attempt in range(max_retries):
            try:
                # Only print retry attempts to keep log clean
                if attempt > 0:
                    print(f"  -> Retry move to {stage_pos} (Attempt {attempt + 1})...")
                
                stage.move_to(stage_pos)
                time.sleep(1.0) 
                # --- VERIFICATION ---
                current_pos = stage.get_position() 
                # Check if we are close enough (within +/- 1.0)
                if abs(current_pos - stage_pos) <= tolerance:
                    move_success = True
                    break 
                else:
                    diff = current_pos - stage_pos
                    print(f"  -> Position Mismatch! Target: {stage_pos}, Actual: {current_pos:.2f} (Diff: {diff:.2f})")
            except Exception as e:
                print(f"  -> Error on move attempt {attempt + 1}: {e}")
                time.sleep(0.5) # Wait before retrying
        if not move_success:
            print(f"CRITICAL: Could not reach {stage_pos} after {max_retries} attempts. Skipping.")
            continue

        meas_power = c_double()
        tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
        power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
        print(f"Stage @ {stage_pos}, Power: {power} µW")
        if power <25:
            frames_per_exposure=500
        elif 25 <=power <400:
            frames_per_exposure =1000
        else:
            frames_per_exposure = 1500
        # Run Acquisitions using helper
        measure_and_save(le_c, f_LE, "LE", stage_pos, power, bg_voltage, tg_voltage, frames_per_exposure)
        measure_and_save(he_c, f_HE, "HE", stage_pos, power, bg_voltage, tg_voltage, frames_per_exposure)

    # 4. Sort Files
    safe_sort(f_LE, "LE")
    safe_sort(f_HE, "HE")

# Reset
iv.x_goto('Vbg', 0, 0.1, 0.1); iv.x_goto('Vtg', 0, 0.1, 0.1); stage.move_to(0)

Exposetime(ms): 88.0

=== New gate pair: Vtg=0.0 V, Vbg=0.0 V ===
variable: ['Vbg']
start: [0.]
end: [0.]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0.]
steps: 2
Reading centers from: $YZD219_powerdep$~$880nmspectra$~$Vtg=0.000V$~$Vbg=0.000V$~$660nmPump$~$730nm2.0uwProbe$~$RpT85ms_f=750$~$6Kp3n2$~$4_8Hz300g$~_002_peaks_nm.csv
Notice: 'peak2_nm' (HE) missing. Filling with NaNs.
Stage @ 200.0, Power: 9.144 µW
  -> LE Acquisition @ 883.1 nm
Center Wave Length: 883.1
Grating: [750nm,300][2][0]
Frame: 150
Frame: 150
  -> Skipping HE (NaN center)
Stage @ 257.1, Power: 9.872 µW
  -> LE Acquisition @ 882.8 nm
Center Wave Length: 882.8
Grating: [750nm,300][2][0]
Frame: 150
Frame: 150
  -> Skipping HE (NaN center)
Stage @ 314.3, Power: 12.988 µW
  -> LE Acquisition @ 882.6 nm
Center Wave Length: 882.6
Grating: [750nm,300][2][0]
Frame: 150
Frame: 150
  -> Skipping HE (NaN center)
Stage @ 371.4, Power: 13.893 µW
  -> LE Acquisition @ 882.3 nm
Center Wave Length: 882.3
Grating: [750nm,300][2][0]


True

In [ ]:
for i in range(5):
    # ==================== SETUP ====================
    DEV_NAME = 'YZD219'
    Pump_Probe_center_folder = rf"D:\instrument_control_v3_1\{DEV_NAME}\pump_probe\centers"
    # Vtgs = np.array([1.2, 1.05, 0.9, 0.8,0.7, 0.55, 0.25])[::-1]
    # Vbgs = np.array([1.2, 1.05, 0.9, 0.8, 0.7, 0.55, 0.25])[::-1]
    Vtgs = np.array([0.0])
    Vbgs = np.array([0.0])
    # Vtgs = np.array([1.2, 1.125,1.05, 0.975,0.9, ])
    # Vbgs = np.array([1.2,1.125, 1.05,0.975, 0.9, ])
    probe_pwr = 0.5  # in µW
    frames_per_exposure, exp_time = 1000, '85'
    lf6.change_expose_time(exp_time)

    # ==================== MAIN LOOP ====================
    for tg_voltage, bg_voltage in zip(Vtgs, Vbgs):
        print(f"\n=== New gate pair: Vtg={tg_voltage} V, Vbg={bg_voltage} V ===")
        iv.x_goto('Vbg', bg_voltage, 0.1, 0.1)
        iv.x_goto('Vtg', tg_voltage, 0.1, 0.1)

        # 1. Read Centers CSV
        tg_str, bg_str = f"Vtg={tg_voltage:.3f}V", f"Vbg={bg_voltage:.3f}V"
        search_pattern = os.path.join(Pump_Probe_center_folder, "*.csv")
        
        # Find file containing both voltage strings
        found_csv = next((f for f in glob.glob(search_pattern) if tg_str in f and bg_str in f), None)

        if not found_csv:
            print(f"CRITICAL ERROR: No CSV found for {tg_str}, {bg_str}. Skipping.")
            continue

        try:
            print(f"Reading centers from: {os.path.basename(found_csv)}")
            df = pd.read_csv(found_csv)
            
            stage_poss = get_series_safe(df, 'StagePos', 'Stage')
            if np.all(np.isnan(stage_poss)): 
                print("CRITICAL: StagePos missing/invalid. Skipping."); continue

            # Mapping: Peak 1 -> LE, Peak 2 -> HE
            centers_LE = np.round(get_series_safe(df, 'peak1_nm', 'LE'), 1)
            centers_HE = np.round(get_series_safe(df, 'peak2_nm', 'HE'), 1)

        except Exception as e:
            print(f"Error reading file: {e}"); continue

        # 2. Initialize Files
        base_name = (f'${DEV_NAME}_p3n2Diffusion$$Vtg={tg_voltage:.3f}V$~$Vbg={bg_voltage:.3f}V$~'
                    f'$660nmPump$~$730nm{probe_pwr}uwProbe$~$CCDRpT{exp_time}ms_800LPfrs={frames_per_exposure}$')
        
        f_LE = check_file_name(f"{base_name}~$LE_series$")
        f_HE = check_file_name(f"{base_name}~$HE_series$")

        header = np.concatenate((['Vbg', 'Vtg', 'Vtg+Vbg', 'Stage_Pos', 'Power', 'Center_nm'], 
                                np.arange(1024).astype(str))).reshape([1, -1])
        
        for f in [f_LE, f_HE]:
            with open(f'{f}.csv', 'w', newline='') as csvfile:
                np.savetxt(csvfile, header, fmt='%s', delimiter=',')

        # 3. Sweep Power
        for stage_pos, le_c, he_c in zip(stage_poss, centers_LE, centers_HE):
            # ---- CONFIGURATION ----
            max_retries = 3
            tolerance = 1.0  # Acceptable error margin (units depend on your stage)
            move_success = False

            for attempt in range(max_retries):
                try:
                    # Only print retry attempts to keep log clean
                    if attempt > 0:
                        print(f"  -> Retry move to {stage_pos} (Attempt {attempt + 1})...")
                    
                    stage.move_to(stage_pos)
                    time.sleep(1.0) 
                    # --- VERIFICATION ---
                    current_pos = stage.get_position() 
                    # Check if we are close enough (within +/- 1.0)
                    if abs(current_pos - stage_pos) <= tolerance:
                        move_success = True
                        break 
                    else:
                        diff = current_pos - stage_pos
                        print(f"  -> Position Mismatch! Target: {stage_pos}, Actual: {current_pos:.2f} (Diff: {diff:.2f})")
                except Exception as e:
                    print(f"  -> Error on move attempt {attempt + 1}: {e}")
                    time.sleep(0.5) # Wait before retrying
            if not move_success:
                print(f"CRITICAL: Could not reach {stage_pos} after {max_retries} attempts. Skipping.")
                continue

            meas_power = c_double()
            tlPM.measPower(byref(meas_power), TLPM_DEFAULT_CHANNEL)
            power = round(meas_power.value * 1e6 * POWER_EFF - probe_pwr, 3)
            print(f"Stage @ {stage_pos}, Power: {power} µW")
            if power <25:
                frames_per_exposure=500
            elif 25 <=power <400:
                frames_per_exposure =1000
            else:
                frames_per_exposure = 1500
            # Run Acquisitions using helper
            measure_and_save(le_c, f_LE, "LE", stage_pos, power, bg_voltage, tg_voltage, frames_per_exposure)
            measure_and_save(he_c, f_HE, "HE", stage_pos, power, bg_voltage, tg_voltage, frames_per_exposure)

        # 4. Sort Files
        safe_sort(f_LE, "LE")
        safe_sort(f_HE, "HE")

    # Reset
    iv.x_goto('Vbg', 0, 0.1, 0.1); iv.x_goto('Vtg', 0, 0.1, 0.1); stage.move_to(0)

Exposetime(ms): 88.0

=== New gate pair: Vtg=0.0 V, Vbg=0.0 V ===
variable: ['Vbg']
start: [0.]
end: [0.]
steps: 2
variable: ['Vtg']
start: [0.]
end: [0.]
steps: 2
Reading centers from: $YZD219_powerdep$~$880nmspectra$~$Vtg=0.000V$~$Vbg=0.000V$~$660nmPump$~$730nm0.5uwProbe$~$RpT88ms_f=750$~$6Kp3n2$~$4_8Hz300g$~_007_peaks_nm.csv
Notice: 'peak2_nm' (HE) missing. Filling with NaNs.
Stage @ 200.0, Power: 11.187 µW
  -> LE Acquisition @ 883.9 nm
Center Wave Length: 883.9
Grating: [750nm,300][2][0]
Frame: 300
Frame: 300
  -> Skipping HE (NaN center)
Stage @ 257.1, Power: 11.233 µW
  -> LE Acquisition @ 883.1 nm
Center Wave Length: 883.1
Grating: [750nm,300][2][0]
Frame: 300
Frame: 300
  -> Skipping HE (NaN center)
Stage @ 314.3, Power: 12.131 µW
  -> LE Acquisition @ 883.6 nm
Center Wave Length: 883.6
Grating: [750nm,300][2][0]
Frame: 300
Frame: 300
  -> Skipping HE (NaN center)
Stage @ 371.4, Power: 13.49 µW
  -> LE Acquisition @ 883.6 nm
Center Wave Length: 883.6
Grating: [750nm,300][2][0]

KeyboardInterrupt: 

## New file writing

In [ ]:
#  f_name = input('Input a filename:')
#         with open('{}.csv'.format(f_name), 'a') as f:
#             wls = lf6.get_wavelength_calibration()
#             cols = np.concatenate((np.array(['X', 'Y'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#             np.savetxt(f, cols, fmt='%s', delimiter=',')

#             for y in np.arange(y_min, y_max, y_step):
#                 for x in np.arange(x_min, x_max, x_step):
#                     daq.set_voltages([x, y])
#                     spectra = lf6.acquire()
#                     data = np.concatenate((np.array([x, y], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#                     np.savetxt(f, data, fmt='%.5e', delimiter=',')

In [ ]:
# wls = lf6.get_wavelength_calibration()
# print(wls)
# lf6.change_center_wavelength(880)

In [ ]:
# vbgs = [1,1.5,2]
# vtgs = [0.5,1,1.5]
# # for x,y in zip(vbgs,vtgs):
# #     print(x,y)
# iv.x_goto('Vbg', 0.25, 0.05, 0.1)
# ep300.set_position(2,51)
# iv.x_goto('Vbg', 0, 0.03, 0.1)
# spectra = lf6.acquire()
# print(spectra)

In [ ]:
import numpy as np
angles = np.linspace(7, 51, 45)
halfangles = [48,93]
print(angles)
# f_name = '$YZ95_VP$~$PL$~$PH$~$730nm90uw$~$1s$~$266degree$'
f_name = '$YZ95_VP$~$PL$~$PH$~$730nm$~$2s$~'
vgs = np.linspace(-4.5,5,191)
# vgs = np.linspace(-0.2,0.2,9)
print(vgs)

In [ ]:
def gate_sweep_vp(filename, vbgs, vbge, vtgs, vtge, step, halfangles):
    vbg = np.linspace(vbgs,vbge,step)
    vtg = np.linspace(vtgs,vtgs,step)
    with open('{}.csv'.format(filename), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg','halfangles'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for step_vbg,step_vtg in zip(vbg,vtg):
                iv.x_goto('Vbg', step_vbg, 0.03, 0.1)
                iv.x_goto('Vtg', step_vtg, 0.03, 0.1)
                for halfangle in halfangles:
                    ep300.set_position(1,halfangle)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([step_vbg, step_vtg, halfangle], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',')               
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1)
    

In [ ]:
# with open('{}.csv'.format(f_name), 'a') as f:
#             wls = lf6.get_wavelength_calibration()
#             cols = np.concatenate((np.array(['Vbg','Vtg','halfangle','powerangle'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#             np.savetxt(f, cols, fmt='%s', delimiter=',')
#             for vg in vgs:
#                 iv.x_goto('Vbg', vg, 0.03, 0.1)
#                 iv.x_goto('Vtg', vg, 0.03, 0.1)
#                 print(vg)
#                 for angle in angles:
#                         ep300.set_position(2,angle)
# #                         print('powerangle:',angle)
#                         for halfangle in halfangles:
#                             ep300.set_position(1,halfangle)
# #                             print('halfangle:', halfangle)
#                             spectra = lf6.acquire()
#                             data = np.concatenate((np.array([vg, vg,halfangle,angle], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#                             np.savetxt(f, data, fmt='%.5e', delimiter=',')
                
#             iv.x_goto('Vbg', 0, 0.03, 0.1)
#             iv.x_goto('Vtg', 0, 0.03, 0.1)
#             print('vg:', 0)
#             ep300.set_position(2,7)
#             print('power', 0)

In [ ]:
# unit ms
lf6.change_expose_time(1000)

In [ ]:
# unit nm
lf6.change_spectra_center(870)

In [ ]:
lf6.change_roi_FullSensor()
lf6.change_roi_LineSensor()

In [ ]:
lf6.change_roi_LineSensor()

In [ ]:
# A = lf6.acquire()
# print(lf6.get_wavelength_calibration())
# B = lf6.acquire()
# print(lf6.get_wavelength_calibration())
# C = lf6.acquire()
# print(lf6.get_wavelength_calibration())
# D = lf6.acquire()
# print(lf6.get_wavelength_calibration())

In [ ]:
print(A)
print(np.shape(A)[0])

In [ ]:
vbg = 0
vtg = 0
iv.x_goto('Vbg', vbg, 0.03, 0.1)
iv.x_goto('Vtg', vtg, 0.03, 0.1)
iv.report_status()

In [ ]:
iv.x_goto('Vbg',1.4, 0.03, 0.1)
iv.x_goto('Vtg', 1.4, 0.03, 0.1)
iv.report_status()

In [ ]:
x = np.array([1,2,3,4,5])
for i in [1,2,3,4,5]:
    y = np.array([1,2,3,4,5])*i
    plt.cla()
    plt.plot(x,y)
    
    plt.pause(2)
    

### SHG

In [ ]:
ep300.set_position(1,0)

In [ ]:
import numpy as np
f_name = '$DX156_20240117$~$SHG$~$RTSpot7$~$900nm65mW$~$2s$'
angles = np.linspace(0,360,361)
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['SHG_angle','Vbg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for angle in angles:
                ep300.set_position(1,angle)                    
                spectra = lf6.acquire()
                data = np.concatenate((np.array([angle,0], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                np.savetxt(f, data, fmt='%.5e', delimiter=',') 

### Print experiment

In [ ]:
import numpy as np
number = 10
sample_name = '$DX156_20231224$~$PL$~$7Kp4up$~$730nm5uW$~$2s$~'
vbg = -4.5
vtg =4.5
total_frs=0
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    total_frs=total_frs+frs
    if frs == 1:
        pass;
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG+BG={}$'.format(dv)
#         exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
#                                         vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
#         print(vbgs, vbge, vtgs, vtge, exptitle, frs)
        print(f'''exps.add_dual_gate_spectra_sweep(sample_name, '{exptitle}', iv, lf6, vbg_start={vbgs}, vbg_stop={vbge},
                                         vtg_start={vtgs}, vtg_stop={vtge}, frames={frs}, repeat=4, plot=False)''')
        print(f'''exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)''')
            
vgs = -4.5
vge =4.5
number1 = 10
for dv in np.linspace(vgs,vge,number1):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    total_frs=total_frs+frs

    if frs == 1:
        pass;
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG-BG={}$'.format(dv)
        print(f'''exps.add_dual_gate_spectra_sweep(sample_name, '{exptitle}', iv, lf6, vbg_start={vbgs}, vbg_stop={vbge},
                                         vtg_start={vtgs}, vtg_stop={vtge}, frames={frs}, repeat=2, plot=False)''')
#         print(vbgs, vbge, vtgs, vtge, exptitle, frs)
print(total_frs)


In [ ]:
vbg = 0
vtg = 0
iv.x_goto('Vbg', vbg, 0.03, 0.1)
iv.x_goto('Vtg', vtg, 0.03, 0.1)
iv.report_status()